# Quantum Repeater Digital Twin -- Refactor v3.2

**New in this version (on top of v3.1's baseline comparison + structural
metrics):**

1. **Full evaluation of the EdgeLSTM's predictive capability.**
   `metrics.prediction` now reports, beyond MAE/RMSE/R^2, a TEMPORAL
   analysis of threshold-crossing timing: does F_hat(t) cross the
   admission threshold at the same TIME the true F(t) does, or does it
   lag / anticipate? `compute_temporal_prediction_metrics` and
   `compute_controller_decision_timing` report the signed timing error
   (positive = late/lagging, negative = early/anticipatory), missed
   degradation events, and false alarms -- validating not just whether
   the final decision is good, but whether it rests on a temporally
   reliable prediction.
2. **2x2 factorial ablation study.** `ablation.run_ablation_study` trains
   and evaluates the full `{EdgeLSTM, StandardLSTM} x {MSE, CS-MSE}` grid
   under the same multi-seed protocol as the rest of the pipeline
   (`StandardLSTM`, in `models.py`, is a larger, non-edge-optimized LSTM
   counterpart to `EdgeLSTM`), and decomposes the results into
   Architecture Effect / Loss Effect / Interaction Effect -- directly
   answering "what is the impact of the EdgeLSTM architecture?", "what is
   the impact of CS-MSE?", and "does the gain come from their
   combination?".
3. **Automatic experiment-tracking pipeline.** `experiment_tracking.ExperimentRun`
   (plus `track_pareto_sweep_experiment` / `track_model_comparison_experiment`
   / `track_ablation_experiment`) writes every run's configuration
   (model/hyperparameters/epochs/seeds/simulator/channel parameters),
   result tables (CSV), figures (PNG), and a `manifest.json` to a
   timestamped directory -- every result is reproducible from disk alone.
4. **Metrics module restructured.** `metrics.py` is now the `metrics/`
   package, split into `prediction.py`, `quantum.py`, `performance.py`,
   and `decision.py` -- one file per responsibility (prediction quality,
   quantum-domain metrics, performance metrics, decision/classification
   metrics), with `from qrepeater_twin.metrics import ...` unchanged.
5. **Plotting helpers.** `plotting.py` adds a matplotlib chart generator
   for every result table in this project, shared between notebook cells
   and the experiment-tracking pipeline.

Everything from v3.1 (baseline comparison against LSTM+MSE/Random
Forest/XGBoost/Transformer, admission confusion matrix, C_latencia,
throughput, QPU economy, energy accounting, decision matrix,
+/-10% weight-sensitivity analysis) and v2 (multi-seed statistical
robustness of the Pareto sweep, CUDA-Event micro-profiling, package
decomposition) remains unchanged.


## 0. Dependency installation

In [ ]:
!pip install -q "qiskit>=1.0" "qiskit-aer>=0.14" "torch" "numpy" "pandas" "scikit-learn" "matplotlib" --upgrade


## 1. Materializing the `qrepeater_twin/` package

The cells below write the decomposed package to disk (one module per
responsibility). In a real GitHub repository, these files would already
exist as versioned `.py` files and these `%%writefile` cells would be
unnecessary -- they exist here only so the notebook remains self-contained
and runnable end-to-end on a clean Colab runtime, without depending on
`git clone`.


In [ ]:
import os
os.makedirs("qrepeater_twin/metrics", exist_ok=True)
os.makedirs("tests", exist_ok=True)
os.makedirs("experiments", exist_ok=True)


### 1.1 `qrepeater_twin/__init__.py` -- public package API

In [ ]:
%%writefile qrepeater_twin/__init__.py
"""
qrepeater_twin
==============

Digital Twin of a Quantum Repeater with a predictive admission controller
(EdgeLSTM) and a Pareto Frontier sweep over the False Positive penalty of
the CS_MSELoss -- plus a cross-architecture comparison against LSTM+MSE,
Random Forest, XGBoost, and Transformer baselines, a full evaluation of
the EdgeLSTM's own predictive capability (MAE/RMSE/R^2 AND temporal
threshold-crossing timing analysis), a 2x2 factorial ablation study
({EdgeLSTM, StandardLSTM} x {MSE, CS_MSELoss}), an admission confusion
matrix (FP/FN/TP/TN) treated as a binary classification problem, the
dimensionless latency ratio C_latencia = tau_inf/T2, throughput, QPU
economy, energy accounting, a ranked multi-criteria decision matrix, an
automatic +/-10% decision-weight sensitivity analysis, plotting helpers,
and an automatic experiment-tracking pipeline that records every run's
configuration/tables/figures to disk.

This package decomposes the original prototype (a single monolithic
notebook) into independent, testable modules, suitable for a reproducible
GitHub repository:

    channel_simulator.py   -> WDMChannelSimulator (synthetic data generation)
    models.py                -> EdgeLSTM, StandardLSTM, CS_MSELoss, train_edge_lstm
    timing.py                  -> InferenceTimer (hardware-accurate profiling, CUDA Events)
    quantum_node.py              -> QuantumRepeaterNode (virtual quantum dataplane via Qiskit Aer)
    orchestrator.py                -> DigitalTwinOrchestrator (intelligent/blind loops, confusion matrix)
    pareto_sweep.py                  -> run_pareto_sweep (statistically robust, multi-seed)
    baselines.py                       -> LSTM+MSE, Random Forest, XGBoost, Transformer predictors
    metrics/                             -> split into prediction.py (MAE/RMSE/R^2 + temporal
                                             crossing-timing analysis), quantum.py (QPU economy,
                                             fidelity statistics), performance.py (throughput,
                                             C_latencia, energy), decision.py (confusion-matrix
                                             classification metrics, decision matrix)
    model_comparison.py                    -> run_model_comparison (cross-architecture, multi-seed)
    ablation.py                              -> run_ablation_study (2x2 factorial: architecture x loss)
    sensitivity.py                             -> run_weight_sensitivity_analysis (+/-10% weight robustness)
    plotting.py                                  -> matplotlib chart generators for every result table
    experiment_tracking.py                         -> ExperimentRun + per-experiment-type trackers
    config.py                                        -> configuration dataclasses
    cli.py                                             -> command-line entry point (main)
"""

from .channel_simulator import WDMChannelSimulator
from .models import EdgeLSTM, StandardLSTM, CS_MSELoss, train_edge_lstm
from .timing import InferenceTimer
from .quantum_node import QuantumRepeaterNode
from .orchestrator import DigitalTwinOrchestrator
from .pareto_sweep import run_pareto_sweep
from .baselines import (
    TinyTransformer,
    RandomForestFidelityModel,
    XGBoostFidelityModel,
    train_lstm_mse,
    train_random_forest,
    train_xgboost,
    train_transformer,
)
from .metrics import (
    compute_regression_metrics,
    evaluate_predictor_regression,
    predict_sequence,
    find_threshold_crossings,
    match_crossings,
    compute_temporal_prediction_metrics,
    extract_fidelity_arrays_from_log,
    compute_controller_decision_timing,
    compute_qpu_economy,
    compute_fidelity_statistics,
    compute_confusion_metrics,
    classify_controller_bias,
    compute_latency_ratio,
    compute_throughput,
    compute_energy_report,
    build_decision_matrix,
)
from .model_comparison import run_model_comparison
from .ablation import run_ablation_study
from .sensitivity import run_weight_sensitivity_analysis, summarize_robustness
from .experiment_tracking import (
    ExperimentRun,
    track_pareto_sweep_experiment,
    track_model_comparison_experiment,
    track_ablation_experiment,
)
from .config import (
    SimConfig,
    TrainConfig,
    QuantumConfig,
    SweepConfig,
    BaselineConfig,
    EnergyConfig,
    ComparisonConfig,
    AblationConfig,
)

__all__ = [
    "WDMChannelSimulator",
    "EdgeLSTM",
    "StandardLSTM",
    "CS_MSELoss",
    "train_edge_lstm",
    "InferenceTimer",
    "QuantumRepeaterNode",
    "DigitalTwinOrchestrator",
    "run_pareto_sweep",
    "TinyTransformer",
    "RandomForestFidelityModel",
    "XGBoostFidelityModel",
    "train_lstm_mse",
    "train_random_forest",
    "train_xgboost",
    "train_transformer",
    "compute_regression_metrics",
    "evaluate_predictor_regression",
    "predict_sequence",
    "find_threshold_crossings",
    "match_crossings",
    "compute_temporal_prediction_metrics",
    "extract_fidelity_arrays_from_log",
    "compute_controller_decision_timing",
    "compute_qpu_economy",
    "compute_fidelity_statistics",
    "compute_confusion_metrics",
    "classify_controller_bias",
    "compute_latency_ratio",
    "compute_throughput",
    "compute_energy_report",
    "build_decision_matrix",
    "run_model_comparison",
    "run_ablation_study",
    "run_weight_sensitivity_analysis",
    "summarize_robustness",
    "ExperimentRun",
    "track_pareto_sweep_experiment",
    "track_model_comparison_experiment",
    "track_ablation_experiment",
    "SimConfig",
    "TrainConfig",
    "QuantumConfig",
    "SweepConfig",
    "BaselineConfig",
    "EnergyConfig",
    "ComparisonConfig",
    "AblationConfig",
]

__version__ = "3.2.0"


### 1.2 `qrepeater_twin/config.py` -- configuration dataclasses (Sim/Train/Quantum/Sweep/Baseline/Energy/Comparison/Ablation)

In [ ]:
%%writefile qrepeater_twin/config.py
"""
Configuration dataclasses.

Centralizing hyperparameters here (instead of scattering them as
positional/keyword arguments across several functions, as in the original
notebook) improves reproducibility: an entire `SweepConfig` can be
serialized (JSON/YAML), version-controlled, and cited in a README or an
experiment report.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import List


@dataclass
class SimConfig:
    """Configuration for the synthetic data generator (WDMChannelSimulator)."""

    n_steps: int = 4000
    dt: float = 0.01
    seed: int = 42
    window_size: int = 20
    test_size: float = 0.2


@dataclass
class TrainConfig:
    """Training configuration for EdgeLSTM + CS_MSELoss."""

    hidden_size: int = 16
    epochs: int = 150
    lr: float = 0.012
    threshold: float = 0.65
    lambda_fn: float = 4.0
    discard_penalty_weight: float = 10.0
    max_discard_rate: float = 0.60


@dataclass
class QuantumConfig:
    """Configuration for the virtual quantum dataplane (QuantumRepeaterNode)."""

    T1: float = 50e-6
    T2: float = 30e-6
    depol_prob: float = 0.01
    shots: int = 512
    seed: int = 7
    success_rate_cutoff: float = 0.5


@dataclass
class SweepConfig:
    """
    Configuration for the Pareto Frontier sweep over `lambda_penalty`.

    `seeds` fixes the statistical fragility of the original prototype:
    instead of a single training run (single batch + single seed) per
    lambda value, each point on the Pareto Frontier is the mean (± standard
    deviation) of `len(seeds)` independent training runs, reducing the risk
    of any given point getting stuck in an unrepresentative local optimum.
    """

    lambda_values: List[float] = field(default_factory=lambda: [1.0, 2.0, 5.0, 10.0, 20.0, 50.0])
    seeds: List[int] = field(default_factory=lambda: [42, 43, 44, 45, 46])


@dataclass
class BaselineConfig:
    """
    Hyperparameters for the *predictor* baselines compared against the
    intelligent admission controller (EdgeLSTM + CS_MSELoss):

        - LSTM+MSE       : same EdgeLSTM architecture, trained with a plain
                            (cost-insensitive) nn.MSELoss -- isolates the
                            contribution of CS_MSELoss itself, holding the
                            architecture fixed.
        - Random Forest   : classical, non-recurrent regressor over the
                            flattened window (n_estimators/max_depth below).
        - XGBoost         : gradient-boosted trees over the flattened
                            window (n_estimators/max_depth/learning_rate
                            below). Optional dependency -- skipped with a
                            warning (not a hard failure) if `xgboost` is
                            not installed.
        - Transformer     : small Transformer encoder over the same input
                            window, trained with plain nn.MSELoss, as a
                            higher-capacity architectural baseline.
    """

    # LSTM + plain MSE (same architecture as EdgeLSTM, no CS_MSELoss)
    lstm_mse_hidden_size: int = 16
    lstm_mse_epochs: int = 150
    lstm_mse_lr: float = 0.012

    # Random Forest
    rf_n_estimators: int = 200
    rf_max_depth: int = 8

    # XGBoost
    xgb_n_estimators: int = 200
    xgb_max_depth: int = 5
    xgb_learning_rate: float = 0.1

    # Transformer encoder
    transformer_d_model: int = 32
    transformer_nhead: int = 4
    transformer_num_layers: int = 2
    transformer_dim_feedforward: int = 64
    transformer_epochs: int = 150
    transformer_lr: float = 0.005


@dataclass
class EnergyConfig:
    """
    Coefficients of the (illustrative) energy-accounting model used by
    `qrepeater_twin.metrics.compute_energy_report`.

    These are order-of-magnitude, documented estimates -- not vendor
    datasheet values -- meant to make the *relative* energy trade-off
    between "ask the network first" (predictive admission, pays a small,
    constant classical-inference energy on every cycle) and "always
    purify" (blind baseline, pays the full quantum-operation energy on
    every cycle) visible and comparable across predictor models.

        - joules_per_1q_gate / joules_per_2q_gate : energy per logical
          gate operation executed by the QPU (or its control electronics)
          during one BBPSSW purification attempt.
        - joules_per_shot_overhead                : fixed per-shot
          overhead (state prep + measurement + reset) independent of gate
          count.
        - classical_inference_power_w             : average power draw of
          the edge accelerator while a predictor model runs one forward
          pass / one prediction (Watts). Combined with the measured
          per-cycle latency to obtain per-cycle classical energy.
        - classical_idle_power_w                  : power draw of that
          same edge device when it is *not* running a predictor at all
          (the blind baseline never invokes one), included so the blind
          baseline's classical energy isn't silently zero.
    """

    joules_per_1q_gate: float = 5e-9
    joules_per_2q_gate: float = 2e-8
    joules_per_shot_overhead: float = 1e-9
    classical_inference_power_w: float = 0.5
    classical_idle_power_w: float = 0.05

    # BBPSSW circuit gate counts (see quantum_node.build_bbpssw_circuit):
    # 2x H, 2x CX (Bell-pair prep) + 4x id + 2x CX (bilateral CNOTs).
    gates_1q_per_attempt: int = 6
    gates_2q_per_attempt: int = 4


@dataclass
class AblationConfig:
    """
    Configuration for `qrepeater_twin.ablation.run_ablation_study`, which
    trains and evaluates the 2x2 factorial grid
    `{StandardLSTM, EdgeLSTM} x {MSE, CS_MSELoss}` under the same
    multi-seed protocol as `run_pareto_sweep` / `run_model_comparison`, to
    isolate the individual and combined contribution of the architecture
    (compact/edge vs. larger/conventional LSTM) and the loss function
    (cost-insensitive MSE vs. cost-sensitive CS-MSE).

    `EdgeLSTM`'s own hyperparameters are taken from `TrainConfig`
    (`hidden_size`, `threshold`, `lambda_fn`, etc.); this dataclass only
    adds what's specific to the ablation grid: `StandardLSTM`'s capacity,
    and the shared training budget (`epochs`/`lr`) applied to all four
    grid cells so no cell is unfairly under/over-trained relative to the
    others.
    """

    standard_lstm_hidden_size: int = 64
    standard_lstm_num_layers: int = 2
    standard_lstm_dropout: float = 0.1

    epochs: int = 150
    lr: float = 0.012
    # lambda_penalty used for both CS-MSE grid cells (EdgeLSTM+CS-MSE and
    # StandardLSTM+CS-MSE); threshold/lambda_fn/discard-rate stabilizers
    # come from TrainConfig, shared with the rest of the pipeline.
    representative_lambda: float = 10.0

    seeds: List[int] = field(default_factory=lambda: [42, 43, 44, 45, 46])
    cycle_time_s: float = 1e-3

    # Headline metrics on which the 2x2 factorial decomposition
    # (architecture effect / loss effect / interaction effect) is
    # reported. Each must be a numeric column produced by
    # `ablation.run_ablation_study`'s per-seed rows.
    headline_metrics: List[str] = field(default_factory=lambda: ["qpu_yield_pct", "mae", "fp"])


@dataclass
class ComparisonConfig:
    """
    Configuration for `qrepeater_twin.model_comparison.run_model_comparison`,
    which trains/evaluates every predictor baseline (EdgeLSTM+CS_MSELoss at
    a representative lambda, LSTM+MSE, Random Forest, XGBoost, Transformer)
    under the same multi-seed protocol as `run_pareto_sweep`, and reports
    regression accuracy (MAE/RMSE/R^2), the admission confusion matrix
    (FP/FN/TP/TN), throughput, QPU economy, energy, the dimensionless
    latency ratio C_latencia = tau_inf / T2, and a multi-criteria decision
    matrix.
    """

    representative_lambda: float = 10.0
    seeds: List[int] = field(default_factory=lambda: [42, 43, 44, 45, 46])
    include_xgboost: bool = True
    cycle_time_s: float = 1e-3
    # +/- fractional perturbation applied to each decision-matrix weight in
    # `qrepeater_twin.sensitivity.run_weight_sensitivity_analysis` (10% by
    # default, per the requested robustness check).
    sensitivity_perturbation_pct: float = 0.10
    # Weights for the decision matrix (must be non-negative; renormalized
    # internally). Each criterion is oriented so that a HIGHER weighted
    # score is always better (cost-type criteria -- energy and the
    # dimensionless latency ratio C_latencia = tau_inf / T2 -- are inverted
    # before weighting; see metrics._COST_CRITERIA). C_latencia replaces a
    # raw millisecond comparison with a dimensionless ratio against the
    # qubit coherence time T2, turning "inference latency" from an IT
    # metric into a physical quantum constraint: C_latencia << 1 means the
    # forward pass is negligible next to decoherence; C_latencia >~ 1 means
    # the memory has already substantially decohered before a decision is
    # even made.
    decision_weights: dict = field(default_factory=lambda: {
        "qpu_yield_pct": 0.25,
        "throughput_pairs_per_s": 0.20,
        "qpu_cycles_saved_pct": 0.20,
        "energy_saved_pct": 0.20,
        "latency_ratio_c": 0.15,
    })


### 1.3 `qrepeater_twin/channel_simulator.py` -- `WDMChannelSimulator`

Generates synthetic time series via an Ornstein-Uhlenbeck process and
derives the latent quantum fidelity F(t). Unchanged from v2.

In [ ]:
%%writefile qrepeater_twin/channel_simulator.py
"""
Component 1 -- `WDMChannelSimulator`.

Generates synthetic time series via an Ornstein-Uhlenbeck process and
derives the latent quantum fidelity F(t). Logic identical to the original
prototype; only extracted into an importable, testable module.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import MinMaxScaler


class WDMChannelSimulator:
    """
    Synthetic simulator of a WDM (Wavelength Division Multiplexing) optical
    channel operating at the network edge.

    Generates two continuous physical variables via Ornstein-Uhlenbeck (OU)
    processes:

        - phase_deviation : classical optical signal phase deviation (rad, >= 0)
        - temp_gradient   : local temperature gradient (K/m, >= 0)

    The latent quantum fidelity F(t) is derived from the phase deviation:
    inversely proportional to it, plus Gaussian noise, clipped to [0, 1].
    """

    def __init__(self, n_steps: int = 4000, dt: float = 0.01, seed: int = 42):
        self.n_steps = n_steps
        self.dt = dt
        self.rng = np.random.default_rng(seed)

    def _ornstein_uhlenbeck(self, theta: float, mu: float, sigma: float, x0: float) -> np.ndarray:
        """
        Numerically integrates (Euler-Maruyama) an Ornstein-Uhlenbeck process:
            dX_t = theta * (mu - X_t) * dt + sigma * dW_t
        """
        x = np.zeros(self.n_steps, dtype=np.float64)
        x[0] = x0
        sqrt_dt = np.sqrt(self.dt)
        for t in range(1, self.n_steps):
            dW = self.rng.normal(0.0, sqrt_dt)
            x[t] = x[t - 1] + theta * (mu - x[t - 1]) * self.dt + sigma * dW
        return x

    def generate_dataset(self) -> pd.DataFrame:
        """Generates the full synthetic dataset (features + latent fidelity)."""
        phase_deviation = self._ornstein_uhlenbeck(theta=0.70, mu=0.30, sigma=0.15, x0=0.30)
        phase_deviation = np.abs(phase_deviation)

        temp_gradient = self._ornstein_uhlenbeck(theta=0.50, mu=0.50, sigma=0.10, x0=0.50)
        temp_gradient = np.abs(temp_gradient)

        alpha = 1.4
        eps = self.rng.normal(0.0, 0.03, self.n_steps)
        fidelity = 1.0 - alpha * phase_deviation + eps
        fidelity = np.clip(fidelity, 0.0, 1.0)

        return pd.DataFrame({
            "phase_deviation": phase_deviation,
            "temp_gradient": temp_gradient,
            "fidelity": fidelity,
        })

    def preprocess(self, df: pd.DataFrame, window_size: int = 20, test_size: float = 0.2):
        """
        Normalizes features with MinMaxScaler, builds sliding windows
        (batch, seq_len, n_features), and splits train/test without
        shuffling (preserves chronological order).
        """
        features = df[["phase_deviation", "temp_gradient"]].values
        target = df[["fidelity"]].values

        feat_scaler = MinMaxScaler(feature_range=(0.0, 1.0))
        features_scaled = feat_scaler.fit_transform(features)

        X, y = [], []
        for i in range(len(features_scaled) - window_size):
            X.append(features_scaled[i:i + window_size])
            y.append(target[i + window_size])
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)

        split_idx = int(len(X) * (1.0 - test_size))
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]

        X_train_t = torch.tensor(X_train, dtype=torch.float32)
        y_train_t = torch.tensor(y_train, dtype=torch.float32)
        X_test_t = torch.tensor(X_test, dtype=torch.float32)
        y_test_t = torch.tensor(y_test, dtype=torch.float32)

        return X_train_t, y_train_t, X_test_t, y_test_t, feat_scaler


### 1.4 `qrepeater_twin/timing.py` -- `InferenceTimer` (micro-profiling fix)

Context manager that times the forward pass using `torch.cuda.Event`
(hardware measurement, on the CUDA stream) on GPU, `time.perf_counter()`
only as fallback on CPU. Unchanged from v2.

In [ ]:
%%writefile qrepeater_twin/timing.py
"""
High-precision profiling utility for PyTorch inference.

Fixes the "Micro-Profiling Inaccuracies" flaw from the original prototype:
`time.perf_counter()` measures host-side (CPU) wall-clock time. Even with
`torch.cuda.synchronize()` called before/after the forward pass -- which
guarantees *correctness* (the measurement won't stop before the CUDA kernel
has actually finished) -- the value itself still includes OS scheduler
jitter, Python/CUDA context-switch overhead, and the limited resolution of
the host clock. For inference in the microsecond/millisecond range on a
compact network like EdgeLSTM, that jitter can be on the same order of
magnitude as the signal being measured.

`torch.cuda.Event(enable_timing=True)` inserts markers directly into the
CUDA stream and measures elapsed time in hardware (on the GPU), via
`elapsed_time()`, isolating the measurement from host-side jitter. It is
the mechanism recommended by the PyTorch documentation for benchmarking
GPU inference latency.

CPU has no equivalent "hardware event" concept (there is no asynchronous
stream to synchronize against), so `time.perf_counter()` remains the best
available option in that case -- and is used only as a fallback.
"""

from __future__ import annotations

import time

import torch


class InferenceTimer:
    """
    Context manager that times a block of code (typically `model(x)`),
    automatically choosing the most accurate profiling mechanism available
    for the given `device`:

        - device.type == "cuda" -> torch.cuda.Event (hardware-level
          measurement, immune to host-side OS jitter).
        - otherwise (CPU)        -> time.perf_counter() (best-effort
          fallback; CPU exposes no asynchronous stream events).

    Usage:
        with InferenceTimer(device) as timer:
            pred = model(x)
        tau_inf = timer.elapsed_s  # seconds, always in this unit
    """

    def __init__(self, device: torch.device):
        self.device = device
        self.use_cuda_events = device.type == "cuda"
        self.elapsed_s: float = 0.0

        if self.use_cuda_events:
            self._start_evt = torch.cuda.Event(enable_timing=True)
            self._end_evt = torch.cuda.Event(enable_timing=True)
        else:
            self._t0 = 0.0

    def __enter__(self) -> "InferenceTimer":
        if self.use_cuda_events:
            # Drain the stream before marking the start: ensures no
            # previously pending work is included in the measured window.
            torch.cuda.synchronize()
            self._start_evt.record()
        else:
            self._t0 = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb) -> None:
        if self.use_cuda_events:
            self._end_evt.record()
            # Required: elapsed_time() requires both events to have
            # already completed on the device.
            torch.cuda.synchronize()
            elapsed_ms = self._start_evt.elapsed_time(self._end_evt)
            self.elapsed_s = elapsed_ms / 1000.0
        else:
            self.elapsed_s = time.perf_counter() - self._t0


### 1.5 `qrepeater_twin/models.py` -- `EdgeLSTM`, `StandardLSTM`, `CS_MSELoss`, `train_edge_lstm`

**New in v3.2:** `StandardLSTM`, a larger-capacity, deeper, dropout-regularized
LSTM that was NOT designed under edge-deployment constraints -- the
architectural counterpart to `EdgeLSTM` in the ablation study's
`{EdgeLSTM, StandardLSTM} x {MSE, CS-MSE}` grid. Same input/output
contract as `EdgeLSTM`, so it drops into the orchestrator and training
routines unmodified.

In [ ]:
%%writefile qrepeater_twin/models.py
"""
Component 2 -- `EdgeLSTM` and `CS_MSELoss` (parameterized for the sweep).

`CS_MSELoss` exposes `lambda_penalty` as the main hyperparameter of the
Pareto Frontier, instantiable dynamically: `CS_MSELoss(lambda_penalty=L)`.
The remaining terms (a moderate False Negative penalty and excess-discard
regularization) remain as stabilizers that prevent the model from
trivially collapsing at any point of the lambda sweep.

`train_edge_lstm` explicitly accepts a `seed` and calls `torch.manual_seed`
before any parameter initialization -- a prerequisite for the multi-seed
averaging implemented in `pareto_sweep.py` (each seed must produce a
genuinely different, reproducible weight initialization).
"""

from __future__ import annotations

import torch
import torch.nn as nn


class EdgeLSTM(nn.Module):
    """
    Lightweight recurrent neural network ("Edge LSTM"), designed for fast
    inference on resource-constrained edge hardware.

    Architecture:
        input   -> (batch, seq_len, 2)  [phase_deviation, temp_gradient]
        LSTM    -> compact hidden_size, few layers
        output  -> linear layer + sigmoid, producing F_hat(t) in [0, 1]
    """

    def __init__(self, input_size: int = 2, hidden_size: int = 16, num_layers: int = 1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)
        self.activation = nn.Sigmoid()  # ensures F_hat(t) is in [0, 1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        pred = self.activation(self.head(last_hidden))
        return pred


class StandardLSTM(nn.Module):
    """
    Conventional (non-edge) LSTM baseline: a larger-capacity, deeper
    recurrent network with dropout regularization, representing a model
    that was NOT designed under edge-deployment constraints (compact
    hidden size, single layer, minimal parameter count).

    This is the architectural counterpart to `EdgeLSTM` in the 2x2
    ablation grid implemented in `ablation.py`
    ({StandardLSTM, EdgeLSTM} x {MSE, CS_MSELoss}): holding the loss
    function fixed and swapping only this class for `EdgeLSTM` isolates
    the contribution of the *architecture* (capacity/depth) from the
    contribution of the *loss function*. Same input/output contract as
    `EdgeLSTM` (`(batch, seq_len, n_features) -> (batch, 1)` in [0, 1]),
    so it drops into `DigitalTwinOrchestrator`, `train_edge_lstm`, and
    `baselines.train_lstm_mse` completely unmodified.
    """

    def __init__(self, input_size: int = 2, hidden_size: int = 64,
                 num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            # nn.LSTM only applies inter-layer dropout when num_layers > 1;
            # with num_layers == 1 this is silently a no-op (matches
            # PyTorch's own documented behavior, not a bug here).
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)
        self.activation = nn.Sigmoid()  # ensures F_hat(t) is in [0, 1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.lstm(x)
        last_hidden = self.dropout(out[:, -1, :])
        pred = self.activation(self.head(last_hidden))
        return pred


class CS_MSELoss(nn.Module):
    """
    Cost-Sensitive Mean Squared Error (CS-MSE), parameterized as the "knob"
    of the Pareto Frontier.

    Terms:
        - lambda_penalty : SEVERE penalty on False Positives (F_true < threshold
                            <= F_pred). This is the hyperparameter swept over
                            in the optimization loop -- the higher it is, the
                            more conservative the model, the higher the QPU
                            efficiency, and the lower the throughput.
        - lambda_fn       : MODERATE penalty on False Negatives (F_pred <
                            threshold <= F_true), kept fixed during the sweep
                            to avoid the model fully collapsing into
                            "discard everything" at high lambda_penalty
                            values.
        - discard_penalty_weight / max_discard_rate : batch-level
          regularization that penalizes discard rates above
          max_discard_rate, reinforcing training stability across the
          whole sweep.
    """

    def __init__(self, threshold: float = 0.65, lambda_penalty: float = 10.0,
                 lambda_fn: float = 2.0, discard_penalty_weight: float = 5.0,
                 max_discard_rate: float = 0.60):
        super().__init__()
        self.threshold = threshold
        self.lambda_penalty = lambda_penalty
        self.lambda_fn = lambda_fn
        self.discard_penalty_weight = discard_penalty_weight
        self.max_discard_rate = max_discard_rate

    def forward(self, y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
        squared_error = (y_pred - y_true) ** 2

        is_false_positive = (y_true < self.threshold) & (y_pred >= self.threshold)
        is_false_negative = (y_true >= self.threshold) & (y_pred < self.threshold)

        weights = torch.ones_like(squared_error)
        weights = torch.where(is_false_positive, torch.full_like(squared_error, self.lambda_penalty), weights)
        weights = torch.where(is_false_negative, torch.full_like(squared_error, self.lambda_fn), weights)

        weighted_mse = (squared_error * weights).mean()

        # Excess-discard penalty (batch level), differentiable via sigmoid.
        soft_discard_indicator = torch.sigmoid((self.threshold - y_pred) * 50.0)
        discard_rate = soft_discard_indicator.mean()
        excess_discard = torch.clamp(discard_rate - self.max_discard_rate, min=0.0)
        discard_penalty = self.discard_penalty_weight * (excess_discard ** 2)

        return weighted_mse + discard_penalty


def train_edge_lstm(model: nn.Module, X_train: torch.Tensor, y_train: torch.Tensor,
                     threshold: float = 0.65, lambda_penalty: float = 10.0, lambda_fn: float = 2.0,
                     discard_penalty_weight: float = 5.0, max_discard_rate: float = 0.60,
                     epochs: int = 120, lr: float = 3e-3, device: torch.device = None,
                     seed: int = None, verbose: bool = False):
    """
    Single-batch (full-batch) training routine (compact dataset).

    `device` determines where the model (and implicitly the tensors, which
    must already be on the same device) is trained. Kept explicit to allow
    consistent GPU/CPU handling throughout the pipeline.

    When provided, `seed` is re-applied via `torch.manual_seed` at the start
    of this function, for reproducibility of the optimization loop itself
    (Adam's internal operation order, etc.). **Important**: since training
    here is full-batch (no DataLoader/shuffling), the only real source of
    variation between seeds is the model's weight initialization -- and
    that initialization must already have happened *before* this call,
    with the same seed applied immediately before `EdgeLSTM(...)` is
    instantiated. This is exactly the pattern (seed -> build model -> train)
    that `pareto_sweep.run_pareto_sweep` follows on every round of the
    multi-seed averaging, ensuring each round starts from a genuinely
    different, reproducible initialization.
    """
    if seed is not None:
        torch.manual_seed(seed)

    if device is not None:
        model = model.to(device)

    criterion = CS_MSELoss(threshold=threshold, lambda_penalty=lambda_penalty, lambda_fn=lambda_fn,
                            discard_penalty_weight=discard_penalty_weight,
                            max_discard_rate=max_discard_rate)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        if verbose and (epoch + 1) % 30 == 0:
            with torch.no_grad():
                discard_rate_now = (y_pred < threshold).float().mean().item()
            print(f"    Epoch {epoch + 1:3d}/{epochs} | CS-MSE Loss: {loss.item():.6f} | "
                  f"Discard rate (train): {discard_rate_now*100:.1f}%")
    return model


### 1.6 `qrepeater_twin/quantum_node.py` -- `QuantumRepeaterNode`

Implements the BBPSSW protocol under NISQ noise (depolarization +
$T_1/T_2$), with the circuit built and transpiled once per instance.
Unchanged from v2.

In [ ]:
%%writefile qrepeater_twin/quantum_node.py
"""
Component 3 -- Virtual Quantum Dataplane (`QuantumRepeaterNode`).

Implements the BBPSSW protocol under NISQ noise (depolarization + T1/T2),
with the "logical latency clock" applied via a thermal-relaxation channel
e^(-latency/T2). The circuit is built and transpiled exactly once per
instance (avoids thousands of redundant recompilations during the Pareto
sweep, since the circuit structure doesn't change across runs -- only the
simulator's noise model does).
"""

from __future__ import annotations

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error


class QuantumRepeaterNode:
    """
    Virtual quantum dataplane of a repeater node, emulated via Qiskit Aer.

    Implements the BBPSSW entanglement-purification circuit under a custom
    NISQ noise model (depolarization + T1/T2 relaxation), and exposes a
    "logical latency clock" that ages the quantum memory in proportion to
    the isolated classical inference time.
    """

    def __init__(self, T1: float = 50e-6, T2: float = 30e-6,
                 depol_prob: float = 0.01, shots: int = 512, seed: int = 7):
        assert T2 <= 2 * T1, "Physical constraint: T2 must be <= 2*T1"
        self.T1 = T1
        self.T2 = T2
        self.depol_prob = depol_prob
        self.shots = shots
        self.seed = seed

        self.base_noise_model = self._build_noise_model()
        self.simulator = AerSimulator(noise_model=self.base_noise_model, seed_simulator=seed)

        # The BBPSSW circuit is structural and doesn't change across runs --
        # only the simulator's noise model varies (via apply_latency_decay).
        # For this reason it is built and transpiled ONCE here, and reused
        # by run_purification() on every subsequent call, eliminating the
        # bottleneck of thousands of recompilations in the simulation loop /
        # Pareto sweep.
        self._circuit = self.build_bbpssw_circuit()
        self._compiled_circuit = transpile(self._circuit, self.simulator)

    def _build_noise_model(self, extra_relax_error=None) -> NoiseModel:
        """
        Builds the NISQ noise model: depolarization + T1/T2 relaxation on
        logical gates, and (optionally) an additional relaxation error
        mapped to the 'id' gate, used to represent aging due to classical
        latency.
        """
        noise_model = NoiseModel()

        error_1q = depolarizing_error(self.depol_prob, 1)
        error_2q = depolarizing_error(self.depol_prob * 2, 2)

        gate_time_1q = 50e-9
        gate_time_2q = 300e-9

        thermal_1q = thermal_relaxation_error(self.T1, self.T2, gate_time_1q)
        thermal_2q_single = thermal_relaxation_error(self.T1, self.T2, gate_time_2q)
        thermal_2q = thermal_2q_single.tensor(thermal_2q_single)

        # Composes depolarization + thermal relaxation into a single
        # QuantumError per gate, avoiding multiple calls to
        # add_all_qubit_quantum_error on the same instruction (which would
        # generate redundant composition warnings).
        combined_1q = error_1q.compose(thermal_1q)
        combined_2q = error_2q.compose(thermal_2q)

        noise_model.add_all_qubit_quantum_error(combined_1q, ["u1", "u2", "u3", "x", "h"])
        noise_model.add_all_qubit_quantum_error(combined_2q, ["cx"])

        if extra_relax_error is not None:
            noise_model.add_all_qubit_quantum_error(extra_relax_error, ["id"])

        return noise_model

    def apply_latency_decay(self, latency: float) -> AerSimulator:
        """
        Logical latency clock.

        Takes the computational time (in seconds) measured in isolation
        (strictly the EdgeLSTM forward pass, or 0.0 in the blind baseline)
        and builds a simulator whose noise model includes a thermal
        relaxation channel equivalent to that interval, applied to the
        'id' gate of the circuit (quantum memory sitting idle, waiting on
        the classical AI's decision). With latency=0.0 (the baseline case),
        the aging channel is effectively null, reflecting unconditional
        admission with no additional wait.
        """
        latency = max(latency, 0.0)
        aging_error = thermal_relaxation_error(self.T1, self.T2, latency) if latency > 0.0 else None
        aged_noise_model = self._build_noise_model(extra_relax_error=aging_error)
        aged_simulator = AerSimulator(noise_model=aged_noise_model, seed_simulator=self.seed)
        return aged_simulator

    @staticmethod
    def build_bbpssw_circuit() -> QuantumCircuit:
        """
        BBPSSW entanglement-purification circuit.

        Qubits 0, 1 -> Bell pair "A"; qubits 2, 3 -> Bell pair "B" (sacrificed).
        1) Creates the two Bell pairs.
        2) 'id' gate on all qubits: represents the wait in quantum memory
           (the target of latency-driven aging).
        3) Bilateral CNOTs of the BBPSSW protocol.
        4) Measures the control qubits (sacrificed pair) in the Z basis;
           matching outcomes (00 or 11) indicate successful purification.
        """
        qc = QuantumCircuit(4, 2, name="BBPSSW")

        for a, b in [(0, 1), (2, 3)]:
            qc.h(a)
            qc.cx(a, b)
        qc.barrier()

        for q in range(4):
            qc.id(q)
        qc.barrier()

        qc.cx(0, 2)
        qc.cx(1, 3)
        qc.barrier()

        qc.measure(2, 0)
        qc.measure(3, 1)

        return qc

    def run_purification(self, simulator: AerSimulator = None):
        """
        Runs the pre-compiled BBPSSW circuit on the given simulator (or on
        the base simulator, without aging, if none is provided).
        """
        sim = simulator if simulator is not None else self.simulator
        result = sim.run(self._compiled_circuit, shots=self.shots).result()
        counts = result.get_counts()

        success_counts = counts.get("00", 0) + counts.get("11", 0)
        success_rate = success_counts / self.shots
        return success_rate, counts


### 1.7 `qrepeater_twin/orchestrator.py` -- `DigitalTwinOrchestrator`

Runs the intelligent (predictive) and blind (reactive) simulation loops,
including the admission confusion matrix (TP/FP/TN/FN). Unchanged from
v3.1.

In [ ]:
%%writefile qrepeater_twin/orchestrator.py
"""
Component 4 -- Orchestrator (`DigitalTwinOrchestrator`).

Keeps two strictly separate methods, each with its own profiling regime:

    - run_intelligent      : timer isolated around `self.model(x)`,
                               measuring only tau_inf (the forward pass),
                               via `InferenceTimer` (CUDA Events on GPU,
                               `perf_counter` on CPU -- see timing.py).
                               Applies the admission control
                               (HALT_PURIFICATION vs PURIFY), and tracks
                               the resulting admission CONFUSION MATRIX
                               (TP/FP/TN/FN -- see below).
    - run_blind_baseline    : NEVER invokes the neural network. Classical
                               latency is forced to 0.0 and admission is
                               unconditional (its own degenerate confusion
                               matrix is reported for context: it always
                               "predicts admit", so FN == TN == 0 by
                               construction).

Admission confusion matrix
---------------------------
Ground truth ("good"/"bad" photon) is `true_fidelity >= threshold`; the
predicted label is the admission DECISION itself (`pred_fidelity >=
threshold` => PURIFY/"admit", i.e. NOT halted):

    TP : good photon, correctly admitted.
    FP : a DEAD photon admitted (F_true < threshold <= F_pred) -- wastes a
         purification attempt (QPU time/shots/energy) on a pair that was
         never going to be useful. This is exactly the error
         `CS_MSELoss.lambda_penalty` penalizes severely.
    FN : a GOOD photon discarded (F_pred < threshold <= F_true) -- a
         usable pair is thrown away, directly reducing throughput.
         Penalized moderately by `CS_MSELoss.lambda_fn`.
    TN : bad photon, correctly halted.

Both loops report `confusion_matrix: {"TP", "FP", "TN", "FN"}` in their
returned metrics dict; `metrics.compute_confusion_metrics` derives
precision/recall/FPR/FNR/F1 from it.
"""

from __future__ import annotations

import torch
import torch.nn as nn

from .quantum_node import QuantumRepeaterNode
from .timing import InferenceTimer


class DigitalTwinOrchestrator:
    """
    Central orchestrator of the Quantum Repeater Digital Twin.

    Keeps two strictly separate simulation loops to eliminate any
    cross-contamination in profiling between the intelligent (predictive)
    approach and the blind/reactive (baseline) approach.
    """

    def __init__(self, model: nn.Module, quantum_node: QuantumRepeaterNode,
                 threshold: float = 0.65, success_rate_cutoff: float = 0.5,
                 device: torch.device = None):
        self.model = model
        self.quantum_node = quantum_node
        self.threshold = threshold
        self.success_rate_cutoff = success_rate_cutoff
        self.device = device if device is not None else torch.device("cpu")
        self.log = []

    def run_intelligent(self, X_test: torch.Tensor, y_test: torch.Tensor) -> dict:
        """
        Simulation loop with predictive admission control (EdgeLSTM + CS_MSELoss).

        `InferenceTimer` wraps STRICTLY the `self.model(x_sample)` call --
        no other operation (scalar extraction via `.item()`, threshold
        comparison, call into the quantum dataplane) enters the timed
        window. On GPU, the measurement is done with `torch.cuda.Event`
        (hardware markers on the CUDA stream), avoiding the host-clock
        jitter inherent to `time.perf_counter()`; on CPU, `InferenceTimer`
        falls back to `perf_counter()`, since there is no asynchronous
        stream to instrument.

        Every cycle's admission decision (PURIFY/HALT) is compared against
        ground truth (`true_fidelity >= threshold`) and tallied into the
        confusion matrix described in the module docstring -- this happens
        for EVERY step, including halted ones, so `TP + FP + TN + FN ==
        total_steps` always holds.
        """
        assert self.model is not None, "run_intelligent requires a trained model."
        self.model.eval()

        results = []
        useful_pairs = 0
        halted = 0
        total_forward_latency = 0.0
        total_steps = len(X_test)
        confusion = {"TP": 0, "FP": 0, "TN": 0, "FN": 0}

        with torch.no_grad():
            for i in range(total_steps):
                x_sample = X_test[i:i + 1]
                true_fidelity = float(y_test[i].item())

                # --- Isolated profiling: times STRICTLY the forward pass ---
                with InferenceTimer(self.device) as timer:
                    pred_tensor = self.model(x_sample)
                tau_inf = timer.elapsed_s
                # --- End of timed window ---

                pred_fidelity = float(pred_tensor.item())
                total_forward_latency += tau_inf

                is_true_good = true_fidelity >= self.threshold
                is_pred_admit = pred_fidelity >= self.threshold
                if is_pred_admit and is_true_good:
                    confusion["TP"] += 1
                elif is_pred_admit and not is_true_good:
                    confusion["FP"] += 1  # dead photon admitted
                elif (not is_pred_admit) and is_true_good:
                    confusion["FN"] += 1  # good photon discarded
                else:
                    confusion["TN"] += 1

                if pred_fidelity < self.threshold:
                    halted += 1
                    results.append({
                        "step": i, "action": "HALT_PURIFICATION",
                        "pred_fidelity": pred_fidelity, "true_fidelity": true_fidelity,
                        "latency_s": tau_inf,
                    })
                    continue

                # Approved: dispatches the isolated classical latency to the
                # quantum node (memory aging) and runs the purification circuit.
                aged_simulator = self.quantum_node.apply_latency_decay(tau_inf)
                success_rate, _counts = self.quantum_node.run_purification(simulator=aged_simulator)

                is_useful = (success_rate >= self.success_rate_cutoff) and (true_fidelity >= self.threshold)
                if is_useful:
                    useful_pairs += 1

                results.append({
                    "step": i, "action": "PURIFY",
                    "pred_fidelity": pred_fidelity, "true_fidelity": true_fidelity,
                    "latency_s": tau_inf, "purification_success_rate": success_rate,
                    "useful": is_useful,
                })

        self.log = results
        return {
            "mode": "intelligent",
            "total_steps": total_steps,
            "useful_pairs": useful_pairs,
            "halted": halted,
            "attempted": total_steps - halted,
            "avg_classical_latency_s": total_forward_latency / max(total_steps, 1),
            "confusion_matrix": confusion,
        }

    def run_blind_baseline(self, X_test: torch.Tensor, y_test: torch.Tensor) -> dict:
        """
        Simulation loop for the blind/reactive (baseline) approach.

        Admission is UNCONDITIONAL: every window is purified, with no
        consultation of the predictive model whatsoever. The neural network
        is NEVER instantiated nor called in this routine -- there is no
        residual "background call". Classical latency is therefore
        correctly forced and recorded as 0.0 seconds (there is no inference
        wait to time).

        Because admission is unconditional, the confusion matrix is
        degenerate by construction: every cycle counts as "predicted
        admit", so `FN == TN == 0` always, `TP` counts truly-good photons
        purified (correctly, by luck of unconditional admission) and `FP`
        counts truly-dead photons purified anyway (wasted QPU time) --
        this is precisely the failure mode a predictive controller with an
        asymmetric (`CS_MSELoss`) penalty is meant to reduce.
        """
        results = []
        useful_pairs = 0
        total_steps = len(X_test)
        forced_latency = 0.0  # No AI inference => no memory wait.
        confusion = {"TP": 0, "FP": 0, "TN": 0, "FN": 0}

        for i in range(total_steps):
            true_fidelity = float(y_test[i].item())

            aged_simulator = self.quantum_node.apply_latency_decay(forced_latency)
            success_rate, _counts = self.quantum_node.run_purification(simulator=aged_simulator)

            is_useful = (success_rate >= self.success_rate_cutoff) and (true_fidelity >= self.threshold)
            if is_useful:
                useful_pairs += 1

            if true_fidelity >= self.threshold:
                confusion["TP"] += 1
            else:
                confusion["FP"] += 1  # dead photon admitted (unconditional admission)

            results.append({
                "step": i, "action": "PURIFY_BLIND",
                "true_fidelity": true_fidelity, "latency_s": forced_latency,
                "purification_success_rate": success_rate, "useful": is_useful,
            })

        self.log = results
        return {
            "mode": "blind",
            "total_steps": total_steps,
            "useful_pairs": useful_pairs,
            "halted": 0,
            "attempted": total_steps,
            "avg_classical_latency_s": forced_latency,
            "confusion_matrix": confusion,
        }


### 1.8 `qrepeater_twin/pareto_sweep.py` -- `run_pareto_sweep` (multi-seed averaging)

Central fix for the statistical fragility flaw: each `lambda_penalty` is
trained/evaluated over `len(seeds)` independent rounds, and the final
table reports mean +/- standard deviation per metric. Unchanged from
v3.1.

In [ ]:
%%writefile qrepeater_twin/pareto_sweep.py
"""
Pareto Frontier -- `lambda_penalty` sweep with multi-seed averaging.

Fixes the "Statistical Fragility" flaw: the original prototype trained the
EdgeLSTM with a single batch (full-batch) and a single random seed per
lambda value, merely *acknowledging* in markdown that individual points
could get stuck in local optima -- without actually mitigating the problem.

Here, each `lambda_penalty` value is trained and evaluated `len(seeds)`
times independently (a different seed each round, applied before the
EdgeLSTM's weight initialization). The point reported on the Pareto
Frontier is the mean across those rounds, accompanied by the standard
deviation -- which makes the training variance at each lambda visible
instead of hidden, drastically reducing the risk of making a production
decision (which lambda to deploy) based on a single, unrepresentative
local optimum.

Each row also reports, per lambda:
    - MAE / RMSE / R^2   : pure regression quality of F_hat(t) itself
                             (`metrics.evaluate_predictor_regression`),
                             computed independently of the admission loop.
    - FP / FN counts       : the admission confusion matrix
                               (`metrics.compute_confusion_metrics`) --
                               the direct, countable evidence for why
                               `CS_MSELoss.lambda_penalty` is worth
                               sweeping: FP ("dead photon admitted") should
                               fall as lambda grows, at some cost in FN
                               ("good photon discarded").
    - C_latencia = tau_inf / T2 : the dimensionless temporal-scale ratio
                                    (`metrics.compute_latency_ratio`)
                                    replacing a raw-millisecond comparison
                                    with a physical constraint against the
                                    qubit's own coherence time.
"""

from __future__ import annotations

import statistics as stats
from typing import List, Sequence

import pandas as pd
import torch

from .models import EdgeLSTM, train_edge_lstm
from .metrics import compute_confusion_metrics, compute_latency_ratio, evaluate_predictor_regression
from .orchestrator import DigitalTwinOrchestrator
from .quantum_node import QuantumRepeaterNode


def _mean_std(values: Sequence[float]) -> tuple:
    mean = stats.fmean(values)
    std = stats.pstdev(values) if len(values) > 1 else 0.0
    return mean, std


def run_pareto_sweep(lambda_values: list, X_train: torch.Tensor, y_train: torch.Tensor,
                      X_test: torch.Tensor, y_test: torch.Tensor, device: torch.device,
                      threshold: float = 0.65, epochs: int = 120, lr: float = 3e-3,
                      hidden_size: int = 16, T1: float = 50e-6, T2: float = 30e-6,
                      depol_prob: float = 0.01, shots: int = 512, quantum_seed: int = 7,
                      lambda_fn: float = 2.0, discard_penalty_weight: float = 5.0,
                      max_discard_rate: float = 0.60, seeds: List[int] = None):
    """
    Runs the Pareto Frontier sweep over the `lambda_penalty` hyperparameter
    of CS_MSELoss, with multi-seed averaging at every point.

    Parameters
    ----------
    seeds : list[int], optional
        Seeds used to repeat training/evaluation at each lambda value.
        Default: 5 seeds ([42, 43, 44, 45, 46]). Each round trains an
        EdgeLSTM from scratch (weight initialization determined by the
        seed) and runs the full Digital Twin over the test set; the
        results of the `len(seeds)` rounds are aggregated into mean ±
        standard deviation before composing that lambda's row in the
        final table.

    Returns
    -------
    results_df : pd.DataFrame
        Consolidated table with one row per lambda value, reporting the
        mean ± standard deviation of each metric across seeds.
    baseline_metrics : dict
        Blind/reactive baseline metrics, computed exactly once (they don't
        depend on lambda or seed, since the neural network is never
        invoked).
    per_seed_results : dict[float, list[dict]]
        Raw metrics from each individual round (lambda -> list of dicts),
        preserved for auditing/debugging and to allow recomputing other
        statistics (median, confidence intervals, etc.) without retraining.
    """
    if seeds is None:
        seeds = [42, 43, 44, 45, 46]

    # --- Blind/reactive baseline: computed exactly once, independent of lambda and seed ---
    print("Running blind/reactive baseline (unconditional admission, forced latency = 0.0)...")
    baseline_node = QuantumRepeaterNode(T1=T1, T2=T2, depol_prob=depol_prob, shots=shots, seed=quantum_seed)
    baseline_orchestrator = DigitalTwinOrchestrator(model=None, quantum_node=baseline_node,
                                                      threshold=threshold, device=device)
    baseline_metrics = baseline_orchestrator.run_blind_baseline(X_test, y_test)
    print(f"  Baseline: Attempts={baseline_metrics['attempted']} | "
          f"Useful pairs={baseline_metrics['useful_pairs']} | "
          f"Forced latency={baseline_metrics['avg_classical_latency_s']*1000:.4f} ms | "
          f"Confusion (unconditional admission) TP={baseline_metrics['confusion_matrix']['TP']} "
          f"FP={baseline_metrics['confusion_matrix']['FP']} (every dead photon is admitted, by construction)\n")

    rows = []
    per_seed_results = {}

    for lam in lambda_values:
        print(f"[lambda_penalty={lam}] training EdgeLSTM on {len(seeds)} seeds "
              f"({epochs} epochs each) ...")

        seed_runs = []
        for seed in seeds:
            # Seed applied BEFORE model construction: guarantees each round
            # starts from an independent weight initialization.
            torch.manual_seed(seed)
            model = EdgeLSTM(input_size=2, hidden_size=hidden_size, num_layers=1).to(device)
            model = train_edge_lstm(
                model, X_train, y_train,
                threshold=threshold, lambda_penalty=lam, lambda_fn=lambda_fn,
                discard_penalty_weight=discard_penalty_weight, max_discard_rate=max_discard_rate,
                epochs=epochs, lr=lr, device=device, seed=seed, verbose=False,
            )

            # Each round uses its own QuantumRepeaterNode with the SAME
            # quantum_seed: isolates the observed variation to the
            # EdgeLSTM's initialization/training, not to the quantum
            # simulator (which must remain comparable across seeds and
            # across lambdas).
            quantum_node = QuantumRepeaterNode(T1=T1, T2=T2, depol_prob=depol_prob,
                                                shots=shots, seed=quantum_seed)
            orchestrator = DigitalTwinOrchestrator(model=model, quantum_node=quantum_node,
                                                     threshold=threshold, device=device)
            metrics = orchestrator.run_intelligent(X_test, y_test)

            # Pure ML regression quality of F_hat(t) -- computed via one
            # full-batch forward pass over X_test, entirely decoupled from
            # the admission/quantum loop above (see metrics.py).
            regression_metrics = evaluate_predictor_regression(model, X_test, y_test, device=device)

            # Admission confusion matrix (this lambda's FP/FN trade-off --
            # the direct, countable justification for CS_MSELoss's
            # asymmetric penalty) and derived rates.
            confusion = metrics["confusion_matrix"]
            confusion_rates = compute_confusion_metrics(confusion)

            # Dimensionless temporal-scale ratio: replaces a raw-ms
            # latency comparison with a physical constraint against T2.
            latency_ratio_c = compute_latency_ratio(metrics["avg_classical_latency_s"], T2)

            yield_qpu_pct = (metrics["useful_pairs"] / max(metrics["attempted"], 1)) * 100.0
            deficit_surplus = metrics["useful_pairs"] - baseline_metrics["useful_pairs"]

            seed_runs.append({
                "seed": seed,
                "halted": metrics["halted"],
                "attempted": metrics["attempted"],
                "useful_pairs": metrics["useful_pairs"],
                "yield_qpu_pct": yield_qpu_pct,
                "deficit_surplus": deficit_surplus,
                "avg_inference_latency_ms": metrics["avg_classical_latency_s"] * 1000.0,
                "latency_ratio_c": latency_ratio_c,
                "tp": confusion["TP"], "fp": confusion["FP"],
                "tn": confusion["TN"], "fn": confusion["FN"],
                "precision": confusion_rates["precision"], "recall": confusion_rates["recall"],
                "mae": regression_metrics["mae"], "rmse": regression_metrics["rmse"],
                "r2": regression_metrics["r2"],
            })

        per_seed_results[lam] = seed_runs

        halted_mean, halted_std = _mean_std([r["halted"] for r in seed_runs])
        attempted_mean, attempted_std = _mean_std([r["attempted"] for r in seed_runs])
        useful_mean, useful_std = _mean_std([r["useful_pairs"] for r in seed_runs])
        yield_mean, yield_std = _mean_std([r["yield_qpu_pct"] for r in seed_runs])
        deficit_mean, deficit_std = _mean_std([r["deficit_surplus"] for r in seed_runs])
        latency_mean, latency_std = _mean_std([r["avg_inference_latency_ms"] for r in seed_runs])
        latency_ratio_mean, latency_ratio_std = _mean_std([r["latency_ratio_c"] for r in seed_runs])
        fp_mean, fp_std = _mean_std([r["fp"] for r in seed_runs])
        fn_mean, fn_std = _mean_std([r["fn"] for r in seed_runs])
        mae_mean, mae_std = _mean_std([r["mae"] for r in seed_runs])
        rmse_mean, rmse_std = _mean_std([r["rmse"] for r in seed_runs])
        r2_values = [r["r2"] for r in seed_runs if r["r2"] == r["r2"]]  # drop NaN
        r2_mean, r2_std = _mean_std(r2_values) if r2_values else (float("nan"), float("nan"))

        rows.append({
            "Lambda": lam,
            "N Seeds": len(seeds),
            "Cycles Saved (HALT)": f"{halted_mean:.1f} +/- {halted_std:.1f}",
            "QPU Attempts": f"{attempted_mean:.1f} +/- {attempted_std:.1f}",
            "Useful Pairs": f"{useful_mean:.1f} +/- {useful_std:.1f}",
            "QPU Yield (%)": f"{yield_mean:.2f} +/- {yield_std:.2f}",
            "SKR Deficit/Surplus": f"{deficit_mean:+.1f} +/- {deficit_std:.1f}",
            "Inference Latency (ms)": f"{latency_mean:.4f} +/- {latency_std:.4f}",
            "C_latencia (tau_inf/T2)": f"{latency_ratio_mean:.4f} +/- {latency_ratio_std:.4f}",
            "FP (dead photon admitted)": f"{fp_mean:.1f} +/- {fp_std:.1f}",
            "FN (good photon discarded)": f"{fn_mean:.1f} +/- {fn_std:.1f}",
            "MAE": f"{mae_mean:.4f} +/- {mae_std:.4f}",
            "RMSE": f"{rmse_mean:.4f} +/- {rmse_std:.4f}",
            "R2": f"{r2_mean:.4f} +/- {r2_std:.4f}",
        })

        print(f"  -> QPU Yield (mean +/- std) = {yield_mean:.2f}% +/- {yield_std:.2f}% | "
              f"Deficit/Surplus (mean) = {deficit_mean:+.1f} | "
              f"FP (mean) = {fp_mean:.1f} | FN (mean) = {fn_mean:.1f} | "
              f"C_latencia (mean) = {latency_ratio_mean:.4f} | "
              f"MAE (mean) = {mae_mean:.4f} | R2 (mean) = {r2_mean:.4f}\n")

    results_df = pd.DataFrame(rows, columns=[
        "Lambda", "N Seeds", "Cycles Saved (HALT)", "QPU Attempts",
        "Useful Pairs", "QPU Yield (%)", "SKR Deficit/Surplus",
        "Inference Latency (ms)", "C_latencia (tau_inf/T2)",
        "FP (dead photon admitted)", "FN (good photon discarded)",
        "MAE", "RMSE", "R2",
    ])
    return results_df, baseline_metrics, per_seed_results


### 1.9 `qrepeater_twin/baselines.py` -- LSTM+MSE, Random Forest, XGBoost, Transformer

Predictor baselines compared against EdgeLSTM+CS-MSE in
`model_comparison.py`. Unchanged from v3.

In [ ]:
%%writefile qrepeater_twin/baselines.py
"""
Component 5 -- Predictor baselines compared against the intelligent
admission controller (EdgeLSTM + CS_MSELoss).

Every baseline here ends up wrapped so it exposes the exact same runtime
contract the rest of the pipeline already relies on
(`orchestrator.DigitalTwinOrchestrator.run_intelligent`):

    - `.eval()`                       (no-op for non-torch models)
    - `model(x)` with `x` of shape (1, window_size, n_features)
      returning something with a scalar `.item()` in [0, 1]

This lets `DigitalTwinOrchestrator` -- and therefore `InferenceTimer`,
`QuantumRepeaterNode.apply_latency_decay`, and every downstream metric --
run *unmodified* regardless of whether the underlying predictor is a
recurrent network, a tree ensemble, or a Transformer. Comparability
across architectures was the whole point of adding these baselines: they
all get admitted into the digital twin loop through the same door.

Baselines implemented:

    1. LSTM + MSE        : `train_lstm_mse` -- the *same* `EdgeLSTM`
                             architecture as the intelligent controller,
                             trained with a plain, cost-insensitive
                             `nn.MSELoss`. Isolates the contribution of
                             `CS_MSELoss` itself (architecture held fixed).
    2. Random Forest      : `RandomForestFidelityModel` -- classical,
                             non-recurrent ensemble over the flattened
                             window.
    3. XGBoost            : `XGBoostFidelityModel` -- gradient-boosted
                             trees over the flattened window. Optional
                             dependency: raises a clear, catchable
                             `ImportError` (not a hard crash) if `xgboost`
                             isn't installed, so callers can skip it.
    4. Transformer        : `TinyTransformer` + `train_transformer` -- a
                             small Transformer encoder over the same input
                             window, trained with plain `nn.MSELoss`, as a
                             higher-capacity architectural baseline.
"""

from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn as nn


# ---------------------------------------------------------------------------
# 1) LSTM + plain MSE (same architecture as EdgeLSTM, cost-insensitive loss)
# ---------------------------------------------------------------------------

def train_lstm_mse(model: nn.Module, X_train: torch.Tensor, y_train: torch.Tensor,
                    epochs: int = 150, lr: float = 0.012, device: torch.device = None,
                    seed: int = None, verbose: bool = False) -> nn.Module:
    """
    Trains an `EdgeLSTM` (or any compatible module) with a plain
    `nn.MSELoss`, i.e. WITHOUT the False-Positive-severe, cost-sensitive
    weighting of `CS_MSELoss`.

    Kept architecturally identical to `models.train_edge_lstm` (same
    full-batch loop, same seed-before-init convention) so that any
    difference observed downstream (QPU yield, useful pairs, latency) is
    attributable to the *loss function*, not to incidental differences in
    the training procedure.
    """
    if seed is not None:
        torch.manual_seed(seed)

    if device is not None:
        model = model.to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        if verbose and (epoch + 1) % 30 == 0:
            print(f"    [LSTM+MSE] Epoch {epoch + 1:3d}/{epochs} | MSE Loss: {loss.item():.6f}")
    return model


# ---------------------------------------------------------------------------
# Shared helpers: window flattening for the non-recurrent baselines
# ---------------------------------------------------------------------------

def _flatten_windows(X: torch.Tensor) -> np.ndarray:
    """(batch, seq_len, n_features) -> (batch, seq_len * n_features), on CPU/numpy."""
    X_np = X.detach().cpu().numpy() if isinstance(X, torch.Tensor) else np.asarray(X)
    return X_np.reshape(X_np.shape[0], -1)


class _SklearnRegressorAdapter:
    """
    Wraps a fitted scikit-learn-style regressor (`.predict(X)`) so it can
    be dropped into `DigitalTwinOrchestrator.run_intelligent` exactly like
    a `torch.nn.Module`: same `.eval()` / callable contract, same
    (1, 1)-shaped, [0, 1]-clipped tensor output.

    These models have no notion of `torch` autograd or CUDA, so
    `InferenceTimer` in `run_intelligent` transparently falls back to
    `perf_counter()` for them (its CUDA-Events branch is only entered when
    `device.type == "cuda"`, and this adapter always returns a CPU
    tensor) -- still a fair, isolated latency measurement of exactly the
    `.predict(...)` call, nothing else.
    """

    def __init__(self, fitted_estimator, name: str):
        self._estimator = fitted_estimator
        self.name = name

    def eval(self) -> "_SklearnRegressorAdapter":
        return self  # stateless at inference time; kept for interface parity

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        x_flat = _flatten_windows(x)
        pred = self._estimator.predict(x_flat)
        pred = np.clip(pred, 0.0, 1.0)
        return torch.as_tensor(pred, dtype=torch.float32).reshape(-1, 1)


# ---------------------------------------------------------------------------
# 2) Random Forest
# ---------------------------------------------------------------------------

class RandomForestFidelityModel:
    """
    Random Forest regressor baseline for F_hat(t), trained on the
    flattened (window_size * n_features) feature vector.

    Unlike `EdgeLSTM`, this model has no notion of sequence order beyond
    whatever the tree splits pick up from the flattened positions -- a
    useful contrast against the recurrent baselines: does explicit
    temporal structure (LSTM) actually earn its keep over an
    order-agnostic ensemble with the same raw information?
    """

    def __init__(self, n_estimators: int = 200, max_depth: int = 8, seed: int = 42):
        from sklearn.ensemble import RandomForestRegressor

        self.model = RandomForestRegressor(
            n_estimators=n_estimators, max_depth=max_depth,
            random_state=seed, n_jobs=-1,
        )

    def fit(self, X_train: torch.Tensor, y_train: torch.Tensor) -> "RandomForestFidelityModel":
        X_flat = _flatten_windows(X_train)
        y_flat = _flatten_windows(y_train).ravel()
        self.model.fit(X_flat, y_flat)
        return self

    def as_orchestrator_model(self) -> _SklearnRegressorAdapter:
        """Wraps the fitted estimator for use with `DigitalTwinOrchestrator`."""
        return _SklearnRegressorAdapter(self.model, name="RandomForest")


def train_random_forest(X_train: torch.Tensor, y_train: torch.Tensor,
                         n_estimators: int = 200, max_depth: int = 8,
                         seed: int = 42) -> _SklearnRegressorAdapter:
    """Convenience one-shot: fit a Random Forest and return it orchestrator-ready."""
    rf = RandomForestFidelityModel(n_estimators=n_estimators, max_depth=max_depth, seed=seed)
    rf.fit(X_train, y_train)
    return rf.as_orchestrator_model()


# ---------------------------------------------------------------------------
# 3) XGBoost (optional dependency)
# ---------------------------------------------------------------------------

class XGBoostFidelityModel:
    """
    Gradient-boosted trees (XGBoost) baseline for F_hat(t), trained on the
    same flattened window representation as `RandomForestFidelityModel`.

    `xgboost` is an OPTIONAL dependency (not in `requirements.txt` by
    default): the import is attempted lazily, inside `__init__`, and
    raises a plain `ImportError` with an actionable message
    (`pip install xgboost`) rather than crashing the whole comparison run.
    Callers (see `model_comparison.run_model_comparison`) catch this and
    skip the XGBoost row, logging a warning instead of failing the sweep.
    """

    def __init__(self, n_estimators: int = 200, max_depth: int = 5,
                 learning_rate: float = 0.1, seed: int = 42):
        try:
            from xgboost import XGBRegressor
        except ImportError as exc:  # pragma: no cover - exercised only when xgboost is absent
            raise ImportError(
                "XGBoost baseline requested but the 'xgboost' package is not installed. "
                "Install it with `pip install xgboost` or set "
                "ComparisonConfig.include_xgboost=False to skip this baseline."
            ) from exc

        self.model = XGBRegressor(
            n_estimators=n_estimators, max_depth=max_depth,
            learning_rate=learning_rate, objective="reg:squarederror",
            random_state=seed, n_jobs=-1,
        )

    def fit(self, X_train: torch.Tensor, y_train: torch.Tensor) -> "XGBoostFidelityModel":
        X_flat = _flatten_windows(X_train)
        y_flat = _flatten_windows(y_train).ravel()
        self.model.fit(X_flat, y_flat)
        return self

    def as_orchestrator_model(self) -> _SklearnRegressorAdapter:
        return _SklearnRegressorAdapter(self.model, name="XGBoost")


def train_xgboost(X_train: torch.Tensor, y_train: torch.Tensor,
                   n_estimators: int = 200, max_depth: int = 5,
                   learning_rate: float = 0.1, seed: int = 42) -> _SklearnRegressorAdapter:
    """Convenience one-shot: fit an XGBoost regressor and return it orchestrator-ready.

    Raises `ImportError` if `xgboost` is not installed -- see
    `XGBoostFidelityModel`.
    """
    xgb = XGBoostFidelityModel(n_estimators=n_estimators, max_depth=max_depth,
                                learning_rate=learning_rate, seed=seed)
    xgb.fit(X_train, y_train)
    return xgb.as_orchestrator_model()


# ---------------------------------------------------------------------------
# 4) Transformer encoder
# ---------------------------------------------------------------------------

class _PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding (Vaswani et al., 2017)."""

    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32)
                              * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[: pe[:, 1::2].shape[1]])
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]


class TinyTransformer(nn.Module):
    """
    Small Transformer-encoder baseline for F_hat(t), operating on the same
    (batch, seq_len, n_features) windows as `EdgeLSTM`.

    Architecture:
        input projection (Linear: n_features -> d_model)
        -> sinusoidal positional encoding
        -> `num_layers` x TransformerEncoderLayer (self-attention + FFN)
        -> mean-pool over the sequence dimension
        -> linear head + sigmoid, producing F_hat(t) in [0, 1]

    A higher-capacity, attention-based architectural baseline against
    which the compact, recurrent `EdgeLSTM` (designed for cheap edge
    inference) can be compared on the yield/latency/energy trade-off,
    not just on raw predictive accuracy.
    """

    def __init__(self, input_size: int = 2, d_model: int = 32, nhead: int = 4,
                 num_layers: int = 2, dim_feedforward: int = 64, dropout: float = 0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.pos_encoding = _PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, 1)
        self.activation = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.input_proj(x)
        h = self.pos_encoding(h)
        h = self.encoder(h)
        pooled = h.mean(dim=1)
        pred = self.activation(self.head(pooled))
        return pred


def train_transformer(model: nn.Module, X_train: torch.Tensor, y_train: torch.Tensor,
                       epochs: int = 150, lr: float = 0.005, device: torch.device = None,
                       seed: int = None, verbose: bool = False) -> nn.Module:
    """
    Trains a `TinyTransformer` with a plain `nn.MSELoss`, following the
    same full-batch, seed-before-init convention as `train_edge_lstm` /
    `train_lstm_mse` so results stay directly comparable.
    """
    if seed is not None:
        torch.manual_seed(seed)

    if device is not None:
        model = model.to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        if verbose and (epoch + 1) % 30 == 0:
            print(f"    [Transformer] Epoch {epoch + 1:3d}/{epochs} | MSE Loss: {loss.item():.6f}")
    return model


### 1.10 `qrepeater_twin/metrics/` -- prediction, quantum, performance, and decision metrics

**Restructured in v3.2:** what used to be a single `metrics.py` is now a
package with one module per responsibility. `from qrepeater_twin.metrics
import ...` keeps working exactly as before -- `metrics/__init__.py`
re-exports every public name from all four submodules.

#### 1.10.1 `qrepeater_twin/metrics/__init__.py` -- re-exports for backward compatibility

In [ ]:
%%writefile qrepeater_twin/metrics/__init__.py
"""
Component 6 -- Derived operational metrics, now split into four
independent, single-responsibility submodules for maintainability and
scientific clarity:

    - `prediction.py`  : pure ML regression quality of F_hat(t)
                          (MAE/RMSE/R^2) AND temporal (event-based)
                          analysis of threshold-crossing timing --
                          "when" the predictor detects degradation
                          relative to when it actually happens.
    - `quantum.py`      : QPU-time economy relative to the blind baseline,
                          and descriptive statistics of the true fidelity
                          trajectory itself (degradation event counts,
                          etc.).
    - `performance.py`  : throughput, the dimensionless latency ratio
                          C_latencia = tau_inf / T2, and illustrative
                          energy accounting.
    - `decision.py`     : the controller's admission decision evaluated
                          as a binary classifier (confusion matrix ->
                          precision/recall/F1), plus the multi-criteria
                          decision matrix used to rank predictor models.

This `__init__.py` re-exports every public name from all four submodules
under `qrepeater_twin.metrics`, so existing code (`from .metrics import
compute_regression_metrics, ...`) keeps working completely unmodified --
the split is purely an internal reorganization, not a breaking API
change.
"""

from .prediction import (
    compute_regression_metrics,
    evaluate_predictor_regression,
    predict_sequence,
    find_threshold_crossings,
    match_crossings,
    compute_temporal_prediction_metrics,
    extract_fidelity_arrays_from_log,
    compute_controller_decision_timing,
)
from .quantum import (
    compute_qpu_economy,
    compute_fidelity_statistics,
)
from .performance import (
    compute_latency_ratio,
    compute_throughput,
    compute_energy_report,
)
from .decision import (
    compute_confusion_metrics,
    classify_controller_bias,
    build_decision_matrix,
)

__all__ = [
    # prediction.py
    "compute_regression_metrics",
    "evaluate_predictor_regression",
    "predict_sequence",
    "find_threshold_crossings",
    "match_crossings",
    "compute_temporal_prediction_metrics",
    "extract_fidelity_arrays_from_log",
    "compute_controller_decision_timing",
    # quantum.py
    "compute_qpu_economy",
    "compute_fidelity_statistics",
    # performance.py
    "compute_latency_ratio",
    "compute_throughput",
    "compute_energy_report",
    # decision.py
    "compute_confusion_metrics",
    "classify_controller_bias",
    "build_decision_matrix",
]


#### 1.10.2 `qrepeater_twin/metrics/prediction.py` -- MAE/RMSE/R^2 + temporal crossing-timing analysis (NEW)

The core deliverable of task 1: pure regression quality of F_hat(t)
(`compute_regression_metrics`), plus event-based temporal accuracy
(`find_threshold_crossings`, `match_crossings`,
`compute_temporal_prediction_metrics`,
`compute_controller_decision_timing`) -- does the predictor's threshold
crossing happen at the right TIME, not just the right VALUE?

In [ ]:
%%writefile qrepeater_twin/metrics/prediction.py
"""
Prediction metrics: pure Machine-Learning regression quality of the
F_hat(t) predictor (MAE / RMSE / R^2), plus temporal analysis of *when*
the predictor detects channel degradation relative to when it actually
happens.

Everything in this module is computed on the predictor's raw output
sequence, entirely BEFORE and decoupled from any admission-control logic
or the quantum dataplane -- these are metrics of the forecaster itself,
not of the final HALT/PURIFY decision (that is `metrics.decision`'s job).

Two complementary views of predictive quality:

    1. Point-wise regression accuracy (MAE/RMSE/R^2) -- "how close is
       F_hat(t) to F(t), on average, across the whole test window?"
    2. Temporal (event-based) accuracy -- "when the channel actually
       degrades below the admission threshold, does the predictor see it
       coming at the right TIME, or does it lag behind / hallucinate
       degradation events that never happen?" A model can have excellent
       point-wise MAE yet still be systematically early or late at the
       one moment that matters operationally: the threshold crossing that
       triggers a HALT decision.
"""

from __future__ import annotations

from typing import List, Sequence, Tuple

import numpy as np
import pandas as pd
import torch


# ---------------------------------------------------------------------------
# Pure Machine-Learning regression metrics (MAE / RMSE / R^2)
# ---------------------------------------------------------------------------

def _to_numpy(x) -> np.ndarray:
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def compute_regression_metrics(y_true, y_pred) -> dict:
    """
    Pure regression-quality metrics of the F_hat(t) predictor: Mean
    Absolute Error (MAE), Root Mean Squared Error (RMSE), and the
    coefficient of determination (R^2).

    Computed BEFORE any admission-control logic (threshold comparison,
    HALT/PURIFY decision) is applied -- this is the predictor evaluated as
    a plain regressor against the true fidelity trajectory, independent of
    the digital-twin/quantum simulation loop entirely. `y_true`/`y_pred`
    accept `torch.Tensor` or array-like of any shape (flattened
    internally); a torch tensor with `requires_grad=True` is safely
    detached first.

    R^2 is defined as `1 - SS_res / SS_tot` and is left as `float('nan')`
    on the degenerate case `SS_tot == 0` (constant `y_true`), rather than
    silently reporting a spurious/undefined value.
    """
    y_true_np = _to_numpy(y_true).ravel().astype(float)
    y_pred_np = _to_numpy(y_pred).ravel().astype(float)
    if y_true_np.shape != y_pred_np.shape:
        raise ValueError(
            f"compute_regression_metrics: shape mismatch after flattening "
            f"({y_true_np.shape} vs {y_pred_np.shape})."
        )

    errors = y_pred_np - y_true_np
    mae = float(np.mean(np.abs(errors)))
    rmse = float(np.sqrt(np.mean(errors ** 2)))

    ss_res = float(np.sum(errors ** 2))
    ss_tot = float(np.sum((y_true_np - y_true_np.mean()) ** 2))
    r2 = (1.0 - ss_res / ss_tot) if ss_tot > 0 else float("nan")

    return {"mae": mae, "rmse": rmse, "r2": r2}


def evaluate_predictor_regression(model, X_test: torch.Tensor, y_test: torch.Tensor,
                                   device: torch.device = None) -> dict:
    """
    Runs ONE full-batch forward pass of `model` over `X_test` (no
    admission logic, no quantum dataplane involved) and reports
    `compute_regression_metrics` against `y_test`.

    Works uniformly for a trained `EdgeLSTM`/`StandardLSTM`/`TinyTransformer`
    (`torch.nn.Module`, moved to `device` and run under
    `torch.no_grad()`) and for the tree-ensemble baselines wrapped by
    `baselines._SklearnRegressorAdapter` (which ignores `device` and
    always returns a CPU tensor) -- both expose the same `.eval()` /
    callable contract, so this function never needs to know which kind of
    predictor it was handed.
    """
    model.eval()
    X = X_test.to(device) if device is not None else X_test
    with torch.no_grad():
        y_pred = model(X)
    return compute_regression_metrics(y_test, y_pred)


def predict_sequence(model, X_test: torch.Tensor, device: torch.device = None) -> np.ndarray:
    """
    Runs ONE full-batch forward pass of `model` over `X_test` and returns
    the flattened prediction sequence as a plain 1-D numpy array -- the
    shared entry point `compute_temporal_prediction_metrics` and
    `evaluate_predictor_regression` both build on, so the model is only
    ever invoked once per evaluation.
    """
    model.eval()
    X = X_test.to(device) if device is not None else X_test
    with torch.no_grad():
        y_pred = model(X)
    return _to_numpy(y_pred).ravel().astype(float)


# ---------------------------------------------------------------------------
# Temporal (event-based) prediction analysis: threshold-crossing timing
# ---------------------------------------------------------------------------

def find_threshold_crossings(series, threshold: float, direction: str = "falling") -> np.ndarray:
    """
    Locates every point where `series` crosses `threshold`, returning
    FRACTIONAL (linearly-interpolated) time indices rather than only the
    nearest integer sample -- e.g. a crossing found between samples 12 and
    13, three-quarters of the way from 12 to 13, is reported as `12.75`.
    This sub-sample precision matters because the "instant of degradation"
    is a continuous physical event that the discrete sampling grid only
    approximates; rounding to the nearest sample would inject up to +/-0.5
    samples of pure discretization error into every timing comparison
    below.

    `direction`:
        - "falling" : `series[i] >= threshold` then `series[i+1] < threshold`
                       (channel degrading below the admission threshold --
                       the event of interest for HALT/PURIFY timing).
        - "rising"  : `series[i] < threshold` then `series[i+1] >= threshold`
                       (channel recovering above the threshold).

    Returns a 1-D numpy array of fractional indices, sorted ascending
    (empty if `series` never crosses `threshold` in the requested
    direction).
    """
    if direction not in ("falling", "rising"):
        raise ValueError(f"find_threshold_crossings: direction must be 'falling' or 'rising', got {direction!r}.")

    s = np.asarray(series, dtype=float).ravel()
    crossings = []
    for i in range(len(s) - 1):
        a, b = s[i], s[i + 1]
        if direction == "falling":
            crossed = (a >= threshold) and (b < threshold)
        else:
            crossed = (a < threshold) and (b >= threshold)
        if not crossed:
            continue
        denom = b - a
        frac = (threshold - a) / denom if denom != 0 else 0.0
        frac = min(max(frac, 0.0), 1.0)  # numerical safety
        crossings.append(i + frac)

    return np.asarray(crossings, dtype=float)


def _interpolate_at(series, frac_index: float) -> float:
    """Linearly interpolates `series` at a (possibly fractional) index."""
    s = np.asarray(series, dtype=float).ravel()
    i0 = int(np.floor(frac_index))
    i1 = min(i0 + 1, len(s) - 1)
    i0 = min(max(i0, 0), len(s) - 1)
    w = frac_index - i0
    return float(s[i0] * (1.0 - w) + s[i1] * w)


def match_crossings(true_crossings: Sequence[float], pred_crossings: Sequence[float],
                     max_distance: float = None) -> Tuple[List[Tuple[int, int, float]], List[int], List[int]]:
    """
    Greedy nearest-neighbor 1-to-1 matching between two sorted lists of
    crossing positions (fractional time indices).

    Repeatedly picks the globally closest still-unmatched (true, pred)
    pair, ties broken by array order, until no pair remains within
    `max_distance` (or all crossings on one side are exhausted). This is a
    standard greedy approximation to the assignment problem -- adequate
    here because crossing events are typically sparse and well-separated
    relative to the sampling grid, so the greedy and optimal (Hungarian)
    assignments coincide in the overwhelming majority of cases, at a
    fraction of the implementation complexity.

    Returns
    -------
    matches : list[(true_idx, pred_idx, distance)]
        Indices into `true_crossings`/`pred_crossings` (not the crossing
        values themselves), plus the matched absolute time distance.
    missed_true_idx : list[int]
        Indices of true crossings with NO matching predicted crossing
        within `max_distance` -- degradation events the predictor missed
        entirely.
    false_pred_idx : list[int]
        Indices of predicted crossings with no corresponding true
        crossing -- degradation events the predictor "hallucinated".
    """
    true_crossings = list(true_crossings)
    pred_crossings = list(pred_crossings)

    candidates = []
    for ti, t in enumerate(true_crossings):
        for pi, p in enumerate(pred_crossings):
            dist = abs(p - t)
            if max_distance is not None and dist > max_distance:
                continue
            candidates.append((dist, ti, pi))
    candidates.sort(key=lambda c: c[0])

    matched_true, matched_pred = set(), set()
    matches: List[Tuple[int, int, float]] = []
    for dist, ti, pi in candidates:
        if ti in matched_true or pi in matched_pred:
            continue
        matched_true.add(ti)
        matched_pred.add(pi)
        matches.append((ti, pi, dist))

    missed_true_idx = [i for i in range(len(true_crossings)) if i not in matched_true]
    false_pred_idx = [i for i in range(len(pred_crossings)) if i not in matched_pred]
    return matches, missed_true_idx, false_pred_idx


def compute_temporal_prediction_metrics(y_true, y_pred, threshold: float = 0.65,
                                         dt: float = None, max_match_distance: float = None,
                                         direction: str = "falling") -> dict:
    """
    Event-based temporal accuracy of the predictor's threshold crossings:
    does F_hat(t) cross below `threshold` at the same TIME the true F(t)
    does?

    For each matched (true, predicted) crossing pair, the signed timing
    error is:

        timing_error = predicted_crossing_index - true_crossing_index

    with the sign convention:
        - timing_error > 0 : the predictor crosses LATE -- it detects
          degradation *after* it has already happened (a lagging /
          delayed controller decision: HALT arrives too late to avoid
          admitting an already-dead photon).
        - timing_error < 0 : the predictor crosses EARLY -- it anticipates
          degradation before it actually occurs (a controller that HALTs
          too conservatively, ahead of the real event).

    Also reports, per matched pair, the VALUE-space error at the true
    crossing instant: `pred_value_at_true_crossing - threshold` (positive
    => the predictor still thought the channel was fine exactly when it
    wasn't; negative => the predictor had already flagged degradation by
    then), a complementary view to the purely temporal error above.

    Parameters
    ----------
    y_true, y_pred : array-like or torch.Tensor
        The TRUE and PREDICTED fidelity sequences, in chronological order
        (one point per test-window step -- e.g. `y_test`/`predict_sequence(...)`).
    threshold : float
        Admission threshold (same value used by `CS_MSELoss`/the
        orchestrator).
    dt : float, optional
        Seconds per step (e.g. `ComparisonConfig.cycle_time_s`). When
        given, every "_steps" quantity below also gets a "_s" (seconds)
        counterpart.
    max_match_distance : float, optional
        Maximum allowed distance (in fractional steps) for a (true, pred)
        crossing pair to be considered a match; unmatched crossings beyond
        this are reported as missed events / false alarms instead of
        being forced into a nonsensical pairing. `None` (default) allows
        any distance.
    direction : {"falling", "rising"}
        Which crossing direction to analyze (default "falling" --
        degradation events, the ones that drive HALT decisions).

    Returns
    -------
    dict with:
        n_true_events, n_pred_events, n_matched, n_missed_events, n_false_alarms
        mean_timing_error_steps, std_timing_error_steps, median_timing_error_steps
        mean_abs_timing_error_steps
        mean_value_error_at_crossing   (predicted-fidelity gap at the true crossing instant)
        timing_errors_steps            : raw list, one per matched pair (for histograms)
        matched_pairs_df                : pd.DataFrame, one row per matched pair
                                           (true_step, pred_step, timing_error_steps,
                                           value_error_at_crossing[, *_s columns])
        (all "_steps" fields get "_s" seconds counterparts when `dt` is given)
    """
    y_true_np = _to_numpy(y_true).ravel().astype(float)
    y_pred_np = _to_numpy(y_pred).ravel().astype(float)
    if y_true_np.shape != y_pred_np.shape:
        raise ValueError(
            f"compute_temporal_prediction_metrics: shape mismatch after flattening "
            f"({y_true_np.shape} vs {y_pred_np.shape})."
        )

    true_crossings = find_threshold_crossings(y_true_np, threshold, direction=direction)
    pred_crossings = find_threshold_crossings(y_pred_np, threshold, direction=direction)

    matches, missed_true_idx, false_pred_idx = match_crossings(
        true_crossings, pred_crossings, max_distance=max_match_distance,
    )

    rows = []
    for ti, pi, _dist in matches:
        t_cross, p_cross = true_crossings[ti], pred_crossings[pi]
        timing_error = p_cross - t_cross
        value_error = _interpolate_at(y_pred_np, t_cross) - threshold
        row = {
            "true_step": t_cross,
            "pred_step": p_cross,
            "timing_error_steps": timing_error,
            "value_error_at_crossing": value_error,
        }
        if dt is not None:
            row["timing_error_s"] = timing_error * dt
        rows.append(row)

    matched_pairs_df = pd.DataFrame(rows, columns=(
        ["true_step", "pred_step", "timing_error_steps", "value_error_at_crossing"]
        + (["timing_error_s"] if dt is not None else [])
    ))

    timing_errors = [r["timing_error_steps"] for r in rows]
    value_errors = [r["value_error_at_crossing"] for r in rows]

    result = {
        "n_true_events": len(true_crossings),
        "n_pred_events": len(pred_crossings),
        "n_matched": len(matches),
        "n_missed_events": len(missed_true_idx),
        "n_false_alarms": len(false_pred_idx),
        "mean_timing_error_steps": float(np.mean(timing_errors)) if timing_errors else float("nan"),
        "std_timing_error_steps": float(np.std(timing_errors)) if timing_errors else float("nan"),
        "median_timing_error_steps": float(np.median(timing_errors)) if timing_errors else float("nan"),
        "mean_abs_timing_error_steps": float(np.mean(np.abs(timing_errors))) if timing_errors else float("nan"),
        "mean_value_error_at_crossing": float(np.mean(value_errors)) if value_errors else float("nan"),
        "timing_errors_steps": timing_errors,
        "matched_pairs_df": matched_pairs_df,
    }
    if dt is not None:
        result["mean_timing_error_s"] = result["mean_timing_error_steps"] * dt
        result["std_timing_error_s"] = result["std_timing_error_steps"] * dt
        result["mean_abs_timing_error_s"] = result["mean_abs_timing_error_steps"] * dt

    return result


def extract_fidelity_arrays_from_log(log: List[dict]) -> Tuple[np.ndarray, np.ndarray]:
    """
    Extracts chronologically-ordered `(true_fidelity, pred_fidelity)`
    arrays from `DigitalTwinOrchestrator.run_intelligent`'s returned log
    (`orchestrator.log`, or the `results` list it builds internally).

    Every entry in that log -- whether the action was `HALT_PURIFICATION`
    or `PURIFY` -- carries both `pred_fidelity` and `true_fidelity` (the
    admission decision only gates whether the QUANTUM circuit runs, not
    whether the classical prediction is recorded), so this always
    recovers the full, unbroken prediction timeline. Raises `ValueError`
    if any entry lacks `pred_fidelity` -- this happens for
    `run_blind_baseline`'s log (`PURIFY_BLIND` actions), which never runs
    a predictor at all and therefore has no prediction timeline to
    analyze.
    """
    if not log:
        raise ValueError("extract_fidelity_arrays_from_log: log is empty.")
    ordered = sorted(log, key=lambda r: r["step"])
    if any("pred_fidelity" not in r for r in ordered):
        raise ValueError(
            "extract_fidelity_arrays_from_log: at least one log entry has no 'pred_fidelity' "
            "(this happens for the blind baseline's log, which never runs a predictor -- "
            "temporal prediction analysis only applies to DigitalTwinOrchestrator.run_intelligent)."
        )
    true_arr = np.array([r["true_fidelity"] for r in ordered], dtype=float)
    pred_arr = np.array([r["pred_fidelity"] for r in ordered], dtype=float)
    return true_arr, pred_arr


def compute_controller_decision_timing(log: List[dict], threshold: float = 0.65,
                                        dt: float = None, max_match_distance: float = None) -> dict:
    """
    Convenience wrapper: extracts the true/predicted fidelity timeline
    from a `DigitalTwinOrchestrator.run_intelligent` log via
    `extract_fidelity_arrays_from_log`, then runs
    `compute_temporal_prediction_metrics` on it.

    Because the admission decision (HALT vs PURIFY) is itself just
    `pred_fidelity >= threshold`, a "falling" crossing of `pred_fidelity`
    below `threshold` IS the controller's HALT decision boundary -- so
    `timing_error_steps > 0` here means the controller's HALT decision
    arrived LATE (after the channel had already degraded: it anticipated
    and correctly, or belatedly, admitted a bad cycle) and
    `timing_error_steps < 0` means the controller HALTed EARLY (ahead of
    the actual degradation event -- an anticipatory / conservative
    decision). This directly answers "anticipation or delay of the
    controller's decision" in terms already tied to the concrete
    HALT/PURIFY log, without requiring the caller to re-derive prediction
    arrays by hand.
    """
    true_arr, pred_arr = extract_fidelity_arrays_from_log(log)
    return compute_temporal_prediction_metrics(
        true_arr, pred_arr, threshold=threshold, dt=dt,
        max_match_distance=max_match_distance, direction="falling",
    )


#### 1.10.3 `qrepeater_twin/metrics/quantum.py` -- QPU economy, fidelity trajectory statistics

In [ ]:
%%writefile qrepeater_twin/metrics/quantum.py
"""
Quantum metrics: QPU-time economy relative to the blind/reactive
baseline, and descriptive statistics of the channel's true fidelity
trajectory itself (mean/spread, fraction below threshold, count of
degradation/recovery events).

`useful_pairs` and `QPU Yield (%)` (useful_pairs / attempted) are already
reported directly by `DigitalTwinOrchestrator.run_intelligent` /
`run_pareto_sweep` / `model_comparison.run_model_comparison`; this module
adds the metrics that are DERIVED from those, plus the physical channel
statistics needed to characterize a WDM/Ornstein-Uhlenbeck run
independently of any predictor.
"""

from __future__ import annotations

import numpy as np

from .prediction import find_threshold_crossings


def compute_qpu_economy(metrics: dict, baseline_metrics: dict, shots_per_attempt: int = None) -> dict:
    """
    QPU-time economy relative to the blind/reactive baseline.

    The blind baseline purifies unconditionally on every cycle
    (`baseline_metrics["attempted"] == baseline_metrics["total_steps"]`).
    A predictive controller instead HALTs low-fidelity cycles, so its
    `attempted` count -- and therefore the number of BBPSSW circuits
    actually dispatched to the QPU -- is lower. This function reports both
    the absolute and percentage reduction in QPU attempts (a direct proxy
    for QPU-time budget saved), alongside the resulting useful-pair
    deficit/surplus.

    `shots_per_attempt` (typically `QuantumConfig.shots`) is optional: when
    given, the avoided attempts are also translated into avoided QPU shots
    (`qpu_shots_saved`); when omitted, `qpu_shots_saved` is left as `None`.
    """
    baseline_attempted = max(baseline_metrics["attempted"], 1)
    cycles_saved = baseline_metrics["attempted"] - metrics["attempted"]
    cycles_saved_pct = (cycles_saved / baseline_attempted) * 100.0
    shots_saved = cycles_saved * shots_per_attempt if shots_per_attempt is not None else None
    return {
        "qpu_cycles_saved": cycles_saved,
        "qpu_cycles_saved_pct": cycles_saved_pct,
        "qpu_shots_saved": shots_saved,
        "useful_pairs_deficit_surplus": metrics["useful_pairs"] - baseline_metrics["useful_pairs"],
    }


def compute_fidelity_statistics(y_true, threshold: float = 0.65) -> dict:
    """
    Descriptive statistics of the TRUE fidelity trajectory F(t) itself --
    independent of any predictor -- characterizing how challenging a given
    channel run actually is:

        - mean / std / min / max fidelity over the window.
        - pct_below_threshold : fraction of samples that are already
          "dead" photons by the admission threshold.
        - n_degradation_events / n_recovery_events : count of falling /
          rising threshold crossings (via `find_threshold_crossings`) --
          how often the channel actually flips state, as opposed to
          drifting slowly around the threshold once. A channel with many
          degradation events in a short window is intrinsically harder to
          predict than one with a single, gradual decline.
    """
    y = np.asarray(y_true, dtype=float).ravel()
    if y.size == 0:
        raise ValueError("compute_fidelity_statistics: y_true is empty.")

    degradation_events = find_threshold_crossings(y, threshold, direction="falling")
    recovery_events = find_threshold_crossings(y, threshold, direction="rising")

    return {
        "mean_fidelity": float(np.mean(y)),
        "std_fidelity": float(np.std(y)),
        "min_fidelity": float(np.min(y)),
        "max_fidelity": float(np.max(y)),
        "pct_below_threshold": float(np.mean(y < threshold) * 100.0),
        "n_degradation_events": int(len(degradation_events)),
        "n_recovery_events": int(len(recovery_events)),
    }


#### 1.10.4 `qrepeater_twin/metrics/performance.py` -- throughput, C_latencia, energy accounting

In [ ]:
%%writefile qrepeater_twin/metrics/performance.py
"""
Performance metrics: throughput, inference/classical latency expressed as
a dimensionless physical ratio against the qubit's own coherence time, and
an illustrative energy accounting comparing a predictive controller
against the blind/reactive baseline.
"""

from __future__ import annotations

from ..config import EnergyConfig


# ---------------------------------------------------------------------------
# Dimensionless temporal-scale ratio: C_latencia = tau_inf / T2
# ---------------------------------------------------------------------------

def compute_latency_ratio(avg_classical_latency_s: float, T2: float) -> float:
    """
    C_latencia = tau_inf / T2 -- the classical inference latency expressed
    as a FRACTION of the qubit's T2 coherence time, replacing a raw
    millisecond comparison with a dimensionless physical constraint.

    A millisecond figure alone says nothing about whether the classical
    decision-making overhead actually matters to the quantum hardware it
    gates: 100 microseconds is negligible for a T2 of seconds, and
    catastrophic for a T2 of microseconds. C_latencia << 1 means the
    forward pass is effectively "free" relative to decoherence; C_latencia
    approaching or exceeding 1 means the qubit has already substantially
    (or fully) decohered by the time the admission decision is made,
    undermining the entire premise of predictive admission control.

    Raises `ValueError` if `T2 <= 0` (undefined ratio).
    """
    if T2 <= 0:
        raise ValueError(f"compute_latency_ratio: T2 must be positive, got {T2!r}.")
    return avg_classical_latency_s / T2


# ---------------------------------------------------------------------------
# Throughput
# ---------------------------------------------------------------------------

def compute_throughput(metrics: dict, cycle_time_s: float = 1e-3) -> dict:
    """
    Useful-pair throughput, in pairs/second.

    `cycle_time_s` is the fixed repetition period of one WDM
    entanglement-generation attempt (the channel keeps ticking at this
    rate regardless of the admission decision -- HALT just skips the
    purification step for that cycle, it doesn't stop the clock). Total
    wall-clock time is therefore:

        total_time_s = total_steps * cycle_time_s + total_classical_latency_s

    where `total_classical_latency_s = avg_classical_latency_s *
    total_steps` adds back the classical inference overhead paid on every
    cycle (zero for the blind baseline, which never invokes a predictor).

    Returns a dict with `total_time_s` and `throughput_pairs_per_s`.
    """
    total_steps = max(metrics["total_steps"], 1)
    total_classical_latency_s = metrics["avg_classical_latency_s"] * total_steps
    total_time_s = total_steps * cycle_time_s + total_classical_latency_s
    throughput = metrics["useful_pairs"] / total_time_s if total_time_s > 0 else 0.0
    return {
        "total_time_s": total_time_s,
        "throughput_pairs_per_s": throughput,
    }


# ---------------------------------------------------------------------------
# Energy accounting
# ---------------------------------------------------------------------------

def _energy_per_attempt_j(energy_cfg: EnergyConfig, shots: int) -> float:
    """
    Illustrative quantum-side energy of ONE BBPSSW purification attempt
    (see `EnergyConfig` docstring for the caveat that these are
    order-of-magnitude coefficients, not datasheet values):

        E_attempt = shots * (n_1q * E_1q + n_2q * E_2q + E_shot_overhead)
    """
    per_shot = (
        energy_cfg.gates_1q_per_attempt * energy_cfg.joules_per_1q_gate
        + energy_cfg.gates_2q_per_attempt * energy_cfg.joules_per_2q_gate
        + energy_cfg.joules_per_shot_overhead
    )
    return shots * per_shot


def compute_energy_report(metrics: dict, baseline_metrics: dict, shots: int,
                           energy_cfg: EnergyConfig = None) -> dict:
    """
    Illustrative total-energy comparison (Joules) between a predictive
    controller and the blind baseline, over the same evaluation window.

    Two components, summed per cycle:

        - Quantum energy : paid ONLY on cycles where a purification
          attempt is actually dispatched (`attempted`), at
          `_energy_per_attempt_j(...)` each.
        - Classical energy : paid on EVERY cycle. For a predictive
          controller, `classical_inference_power_w * avg_classical_latency_s`
          per cycle (the edge accelerator actively running the predictor);
          for the blind baseline (no predictor invoked),
          `classical_idle_power_w * cycle-equivalent latency` -- using the
          predictor's own average latency as the reference cycle length so
          the two totals are computed over comparable wall-clock time.

    Returns quantum/classical/total energy (J) for `metrics`, the
    equivalent total for the baseline, and the resulting percentage
    saved (positive = the predictor used less total energy than blind
    purification).
    """
    energy_cfg = energy_cfg or EnergyConfig()
    e_attempt = _energy_per_attempt_j(energy_cfg, shots)

    quantum_j = metrics["attempted"] * e_attempt
    classical_j = (
        metrics["total_steps"] * metrics["avg_classical_latency_s"]
        * energy_cfg.classical_inference_power_w
    )
    total_j = quantum_j + classical_j

    baseline_quantum_j = baseline_metrics["attempted"] * e_attempt
    # Reference idle window: the predictor's own average latency, so the
    # baseline's classical term is on the same time basis as `metrics`
    # rather than implicitly zero.
    baseline_classical_j = (
        baseline_metrics["total_steps"] * metrics["avg_classical_latency_s"]
        * energy_cfg.classical_idle_power_w
    )
    baseline_total_j = baseline_quantum_j + baseline_classical_j

    energy_saved_j = baseline_total_j - total_j
    energy_saved_pct = (energy_saved_j / baseline_total_j * 100.0) if baseline_total_j > 0 else 0.0

    return {
        "quantum_energy_j": quantum_j,
        "classical_energy_j": classical_j,
        "total_energy_j": total_j,
        "baseline_total_energy_j": baseline_total_j,
        "energy_saved_j": energy_saved_j,
        "energy_saved_pct": energy_saved_pct,
    }


#### 1.10.5 `qrepeater_twin/metrics/decision.py` -- confusion-matrix classification metrics, decision matrix

In [ ]:
%%writefile qrepeater_twin/metrics/decision.py
"""
Decision metrics: the controller's admission decision (HALT vs PURIFY)
evaluated as a binary CLASSIFICATION problem against ground truth (the
photon actually being usable or not), plus the multi-criteria decision
matrix used to rank and select which predictor to deploy.
"""

from __future__ import annotations

from typing import Dict, List

import pandas as pd


# ---------------------------------------------------------------------------
# Admission confusion matrix (derived rates)
# ---------------------------------------------------------------------------

def compute_confusion_metrics(confusion: dict) -> dict:
    """
    Derives precision, recall (sensitivity), specificity, False Positive
    Rate, False Negative Rate, and F1 from the admission confusion matrix
    `{"TP", "FP", "TN", "FN"}` reported by
    `DigitalTwinOrchestrator.run_intelligent` / `run_blind_baseline`.

    The controller's decision is treated as a binary CLASSIFIER: ground
    truth ("good"/"bad" photon) is `true_fidelity >= threshold`; the
    predicted label is the admission DECISION itself
    (`pred_fidelity >= threshold` => PURIFY/"admit"):

        TP : good photon, correctly admitted.
        FP : DEAD photon admitted (F_true < threshold <= F_pred) -- wastes
             a purification attempt (QPU time/shots/energy) on a pair that
             was never going to be useful. This is exactly the error
             `CS_MSELoss.lambda_penalty` penalizes severely. A high FP
             count/rate indicates the controller is "accepting degraded
             states" it should have rejected.
        FN : GOOD photon discarded (F_pred < threshold <= F_true) -- a
             usable pair is thrown away, directly reducing throughput.
             Penalized moderately by `CS_MSELoss.lambda_fn`. A high FN
             count/rate indicates the controller is "rejecting still-usable
             states" -- i.e. being excessively conservative.
        TN : bad photon, correctly halted.

    Returns precision/recall/specificity/FPR/FNR/F1, each `float('nan')`
    when its denominator is zero (e.g. `FPR` is undefined if there were no
    negative/"bad" ground-truth cases at all in the evaluation window).
    """
    tp, fp, tn, fn = confusion["TP"], confusion["FP"], confusion["TN"], confusion["FN"]

    def _safe_div(numerator, denominator):
        return (numerator / denominator) if denominator > 0 else float("nan")

    precision = _safe_div(tp, tp + fp)
    recall = _safe_div(tp, tp + fn)  # a.k.a. sensitivity / True Positive Rate
    specificity = _safe_div(tn, tn + fp)  # True Negative Rate
    fpr = _safe_div(fp, fp + tn)  # dead-photon-admitted rate
    fnr = _safe_div(fn, fn + tp)  # good-photon-discarded rate
    f1 = _safe_div(2 * precision * recall, precision + recall) if (precision == precision and recall == recall) else float("nan")

    return {
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "fpr": fpr,
        "fnr": fnr,
        "f1": f1,
    }


def classify_controller_bias(confusion_rates: dict, fpr_high: float = 0.15, fnr_high: float = 0.15) -> str:
    """
    Human-readable qualitative verdict from `compute_confusion_metrics`'s
    output, answering directly: is the controller (a) accepting degraded
    states, (b) rejecting still-usable states, or (c) excessively
    conservative overall?

    Thresholds `fpr_high`/`fnr_high` (default 15%) are deliberately
    simple, inspectable cutoffs -- NOT a claim of statistical
    significance -- meant to turn a table of numbers into an immediate,
    qualitative reading; always inspect the raw `fpr`/`fnr` values
    alongside this verdict for the actual magnitude.
    """
    fpr, fnr = confusion_rates["fpr"], confusion_rates["fnr"]
    fpr_flag = (fpr == fpr) and (fpr > fpr_high)
    fnr_flag = (fnr == fnr) and (fnr > fnr_high)

    if fpr_flag and fnr_flag:
        return (f"Controller shows BOTH a high false-positive rate ({fpr*100:.1f}% > {fpr_high*100:.0f}%, "
                f"accepting degraded states) AND a high false-negative rate ({fnr*100:.1f}% > {fnr_high*100:.0f}%, "
                f"rejecting still-usable states) -- decision quality is poor in both directions.")
    if fpr_flag:
        return (f"Controller is ACCEPTING DEGRADED STATES too often: false-positive rate "
                f"{fpr*100:.1f}% exceeds the {fpr_high*100:.0f}% threshold (dead photons are being "
                f"admitted, wasting QPU time on doomed purification attempts).")
    if fnr_flag:
        return (f"Controller is EXCESSIVELY CONSERVATIVE: false-negative rate {fnr*100:.1f}% exceeds "
                f"the {fnr_high*100:.0f}% threshold (still-usable photons are being rejected, "
                f"unnecessarily reducing throughput).")
    return (f"Controller decision quality is within the inspected bounds: false-positive rate "
            f"{fpr*100:.1f}% and false-negative rate {fnr*100:.1f}% are both <= {max(fpr_high, fnr_high)*100:.0f}%.")


# ---------------------------------------------------------------------------
# Multi-criteria decision matrix
# ---------------------------------------------------------------------------

# Criteria where a LOWER raw value is better (inverted before weighting).
# `latency_ratio_c` (C_latencia = tau_inf / T2) replaces a raw
# `inference_latency_ms` criterion -- see `performance.compute_latency_ratio`.
_COST_CRITERIA = {"latency_ratio_c", "total_energy_j"}


def build_decision_matrix(rows: List[dict], weights: Dict[str, float],
                           model_key: str = "Model") -> pd.DataFrame:
    """
    Multi-criteria decision matrix: normalizes every weighted criterion
    to [0, 1] (min-max, direction-aware -- see `_COST_CRITERIA`) across
    the candidate models in `rows`, then computes a weighted composite
    score in [0, 1] (higher is better).

    Parameters
    ----------
    rows : list[dict]
        One dict per candidate model, each containing `model_key` plus
        every key present in `weights` (raw, un-normalized values -- e.g.
        `qpu_yield_pct`, `throughput_pairs_per_s`, `qpu_cycles_saved_pct`,
        `energy_saved_pct`, `latency_ratio_c`).
    weights : dict[str, float]
        Criterion -> weight (non-negative; renormalized to sum to 1
        internally, so callers don't need to pre-normalize).
    model_key : str
        Column name identifying each row's model (default "Model").

    Returns
    -------
    pd.DataFrame
        One row per model, with a `<criterion>_norm` column per weighted
        criterion, a `Decision Score` column (weighted sum, higher =
        better), and a `Rank` column (1 = best), sorted by rank.
    """
    if not rows:
        return pd.DataFrame(columns=[model_key, "Decision Score", "Rank"])

    total_weight = sum(max(w, 0.0) for w in weights.values())
    if total_weight <= 0:
        raise ValueError("build_decision_matrix: weights must sum to a positive value.")
    norm_weights = {k: max(w, 0.0) / total_weight for k, w in weights.items()}

    df = pd.DataFrame(rows)
    scored = df.copy()

    for criterion in norm_weights:
        if criterion not in df.columns:
            raise KeyError(f"build_decision_matrix: missing criterion '{criterion}' in rows.")
        col = df[criterion].astype(float)
        lo, hi = col.min(), col.max()
        span = hi - lo
        if span == 0:
            normalized = pd.Series([1.0] * len(col), index=col.index)
        else:
            normalized = (col - lo) / span
            if criterion in _COST_CRITERIA:
                normalized = 1.0 - normalized
        scored[f"{criterion}_norm"] = normalized

    scored["Decision Score"] = sum(
        scored[f"{criterion}_norm"] * w for criterion, w in norm_weights.items()
    )
    scored["Rank"] = scored["Decision Score"].rank(ascending=False, method="min").astype(int)

    ordered_cols = [model_key] + [f"{c}_norm" for c in norm_weights] + ["Decision Score", "Rank"]
    remaining_cols = [c for c in scored.columns if c not in ordered_cols]
    scored = scored[ordered_cols + remaining_cols]
    return scored.sort_values("Rank").reset_index(drop=True)


### 1.11 `qrepeater_twin/model_comparison.py` -- `run_model_comparison`

Trains/evaluates EdgeLSTM+CS-MSE against every baseline over the same
seeds, reporting regression accuracy, the confusion matrix, throughput,
QPU economy, energy, and C_latencia -- then ranks them with the decision
matrix. Unchanged from v3.1.

In [ ]:
%%writefile qrepeater_twin/model_comparison.py
"""
Component 7 -- Cross-architecture model comparison
(`run_model_comparison`).

Extends the single-hyperparameter Pareto sweep (`pareto_sweep.py`, which
only varies `lambda_penalty` for a fixed `EdgeLSTM + CS_MSELoss`) to a
comparison ACROSS predictor architectures/objectives:

    - EdgeLSTM + CS_MSELoss  (at one representative `lambda_penalty`,
                                `ComparisonConfig.representative_lambda`)
    - LSTM + MSE              (`baselines.train_lstm_mse`)
    - Random Forest            (`baselines.train_random_forest`)
    - XGBoost                   (`baselines.train_xgboost`, skipped with a
                                  warning if not installed)
    - Transformer                (`baselines.train_transformer`)
    - Blind/reactive baseline     (`orchestrator.run_blind_baseline`,
                                    computed once, same as in
                                    `pareto_sweep.py`)

Every predictor is trained/evaluated over the SAME `seeds` and run
through the SAME `DigitalTwinOrchestrator.run_intelligent` loop against
the SAME `QuantumRepeaterNode` configuration -- so the resulting
metrics are directly comparable, and `metrics.build_decision_matrix` can
rank them on equal footing. Per model (mean +/- std across seeds), this
reports:

    - Pure ML regression quality of F_hat(t): MAE, RMSE, R^2
      (`metrics.evaluate_predictor_regression`), computed BEFORE any
      admission logic, decoupled from the digital-twin/quantum loop.
    - The admission confusion matrix: TP, FP ("dead photon admitted"),
      TN, FN ("good photon discarded") -- `metrics.compute_confusion_metrics`
      -- the direct, countable justification for `CS_MSELoss`'s asymmetric
      penalty.
    - Throughput, QPU economy, energy (`metrics.compute_throughput` /
      `compute_qpu_economy` / `compute_energy_report`).
    - The dimensionless temporal-scale ratio C_latencia = tau_inf / T2
      (`metrics.compute_latency_ratio`), used (instead of a raw
      millisecond figure) as the latency criterion in the decision matrix.

Finally, `run_model_comparison` automatically runs
`sensitivity.run_weight_sensitivity_analysis` on the resulting decision
rows (+/- `ComparisonConfig.sensitivity_perturbation_pct`, 10% by
default), to certify that the #1-ranked model is not an artifact of the
particular weights chosen for `ComparisonConfig.decision_weights`.
"""

from __future__ import annotations

import statistics as stats
import warnings
from typing import List, Sequence

import pandas as pd
import torch

from .baselines import TinyTransformer, train_lstm_mse, train_random_forest, train_transformer
from .config import BaselineConfig, ComparisonConfig, EnergyConfig, QuantumConfig, TrainConfig
from .models import EdgeLSTM, train_edge_lstm
from .metrics import (
    build_decision_matrix,
    compute_confusion_metrics,
    compute_energy_report,
    compute_latency_ratio,
    compute_qpu_economy,
    compute_throughput,
    evaluate_predictor_regression,
)
from .orchestrator import DigitalTwinOrchestrator
from .quantum_node import QuantumRepeaterNode
from .sensitivity import run_weight_sensitivity_analysis, summarize_robustness


def _mean_std(values: Sequence[float]) -> tuple:
    mean = stats.fmean(values)
    std = stats.pstdev(values) if len(values) > 1 else 0.0
    return mean, std


def _mean_std_dropna(values: Sequence[float]) -> tuple:
    """Same as `_mean_std`, but drops NaN entries first (e.g. degenerate R^2)."""
    clean = [v for v in values if v == v]  # NaN != NaN
    return _mean_std(clean) if clean else (float("nan"), float("nan"))


def _build_predictor(name: str, seed: int, device: torch.device,
                      X_train: torch.Tensor, y_train: torch.Tensor,
                      train_cfg: TrainConfig, baseline_cfg: BaselineConfig,
                      representative_lambda: float):
    """
    Trains one predictor of `name` for one `seed` and returns an object
    compatible with `DigitalTwinOrchestrator` (a trained `nn.Module`, or a
    `_SklearnRegressorAdapter` for the tree-ensemble baselines).

    Raises `ImportError` for "XGBoost" when `xgboost` isn't installed --
    callers are expected to catch this and skip the model (see
    `run_model_comparison`).
    """
    torch.manual_seed(seed)

    if name == "EdgeLSTM+CS-MSE":
        model = EdgeLSTM(input_size=2, hidden_size=train_cfg.hidden_size, num_layers=1).to(device)
        return train_edge_lstm(
            model, X_train, y_train, threshold=train_cfg.threshold,
            lambda_penalty=representative_lambda, lambda_fn=train_cfg.lambda_fn,
            discard_penalty_weight=train_cfg.discard_penalty_weight,
            max_discard_rate=train_cfg.max_discard_rate,
            epochs=train_cfg.epochs, lr=train_cfg.lr, device=device, seed=seed,
        )

    if name == "LSTM+MSE":
        model = EdgeLSTM(input_size=2, hidden_size=baseline_cfg.lstm_mse_hidden_size, num_layers=1).to(device)
        return train_lstm_mse(model, X_train, y_train, epochs=baseline_cfg.lstm_mse_epochs,
                               lr=baseline_cfg.lstm_mse_lr, device=device, seed=seed)

    if name == "RandomForest":
        return train_random_forest(X_train, y_train, n_estimators=baseline_cfg.rf_n_estimators,
                                    max_depth=baseline_cfg.rf_max_depth, seed=seed)

    if name == "XGBoost":
        from .baselines import train_xgboost  # local import: surfaces ImportError to the caller
        return train_xgboost(X_train, y_train, n_estimators=baseline_cfg.xgb_n_estimators,
                              max_depth=baseline_cfg.xgb_max_depth,
                              learning_rate=baseline_cfg.xgb_learning_rate, seed=seed)

    if name == "Transformer":
        model = TinyTransformer(
            input_size=2, d_model=baseline_cfg.transformer_d_model,
            nhead=baseline_cfg.transformer_nhead, num_layers=baseline_cfg.transformer_num_layers,
            dim_feedforward=baseline_cfg.transformer_dim_feedforward,
        ).to(device)
        return train_transformer(model, X_train, y_train, epochs=baseline_cfg.transformer_epochs,
                                  lr=baseline_cfg.transformer_lr, device=device, seed=seed)

    raise ValueError(f"Unknown predictor name: {name!r}")


def run_model_comparison(X_train: torch.Tensor, y_train: torch.Tensor,
                          X_test: torch.Tensor, y_test: torch.Tensor, device: torch.device,
                          train_cfg: TrainConfig = None, quantum_cfg: QuantumConfig = None,
                          baseline_cfg: BaselineConfig = None, energy_cfg: EnergyConfig = None,
                          comparison_cfg: ComparisonConfig = None,
                          model_names: List[str] = None):
    """
    Trains/evaluates every predictor baseline (plus the blind baseline)
    over `comparison_cfg.seeds`, and reports regression accuracy
    (MAE/RMSE/R^2), the admission confusion matrix (FP/FN/TP/TN),
    throughput, QPU economy, energy, the dimensionless latency ratio
    C_latencia, a ranked multi-criteria decision matrix, and a +/-10%
    weight-sensitivity analysis of that ranking.

    Parameters
    ----------
    model_names : list[str], optional
        Subset/order of predictors to include. Default: all five
        (`["EdgeLSTM+CS-MSE", "LSTM+MSE", "RandomForest", "XGBoost",
        "Transformer"]`). "XGBoost" is silently skipped (with a printed
        warning) if `xgboost` isn't installed or
        `comparison_cfg.include_xgboost` is `False`.

    Returns
    -------
    results_df : pd.DataFrame
        One row per model (mean +/- std across seeds): regression
        accuracy, confusion matrix, QPU yield, useful pairs, throughput,
        QPU cycles saved, energy saved, and C_latencia.
    baseline_metrics : dict
        Blind/reactive baseline metrics, including its own (degenerate)
        confusion matrix (see `orchestrator.run_blind_baseline`).
    decision_matrix_df : pd.DataFrame
        Output of `metrics.build_decision_matrix` -- normalized criteria,
        weighted `Decision Score`, and `Rank` (1 = recommended model),
        built from the SAME mean values reported in `results_df`.
    per_model_seed_results : dict[str, list[dict]]
        Raw per-seed metrics for each model, preserved for auditing.
    sensitivity_results : tuple
        `(summary_df, trials_df, verdict)` from
        `sensitivity.run_weight_sensitivity_analysis` run on
        `decision_matrix_df`'s underlying rows, +/-
        `comparison_cfg.sensitivity_perturbation_pct` -- certifies whether
        the #1 ranking is independent of the exact decision weights.
    """
    train_cfg = train_cfg or TrainConfig()
    quantum_cfg = quantum_cfg or QuantumConfig()
    baseline_cfg = baseline_cfg or BaselineConfig()
    energy_cfg = energy_cfg or EnergyConfig()
    comparison_cfg = comparison_cfg or ComparisonConfig()
    model_names = model_names or ["EdgeLSTM+CS-MSE", "LSTM+MSE", "RandomForest", "XGBoost", "Transformer"]

    # --- Blind/reactive baseline: computed exactly once ---
    print("Running blind/reactive baseline (unconditional admission, forced latency = 0.0)...")
    baseline_node = QuantumRepeaterNode(T1=quantum_cfg.T1, T2=quantum_cfg.T2, depol_prob=quantum_cfg.depol_prob,
                                         shots=quantum_cfg.shots, seed=quantum_cfg.seed)
    baseline_orchestrator = DigitalTwinOrchestrator(model=None, quantum_node=baseline_node,
                                                      threshold=train_cfg.threshold, device=device)
    baseline_metrics = baseline_orchestrator.run_blind_baseline(X_test, y_test)
    baseline_confusion = baseline_metrics["confusion_matrix"]
    print(f"  Baseline: Attempts={baseline_metrics['attempted']} | "
          f"Useful pairs={baseline_metrics['useful_pairs']} | "
          f"Confusion: TP={baseline_confusion['TP']} FP={baseline_confusion['FP']} "
          f"(every dead photon is admitted, by construction)\n")

    rows = []
    per_model_seed_results = {}

    for name in model_names:
        if name == "XGBoost" and not comparison_cfg.include_xgboost:
            print("[XGBoost] skipped (ComparisonConfig.include_xgboost=False).\n")
            continue

        print(f"[{name}] training on {len(comparison_cfg.seeds)} seeds ...")
        seed_runs = []
        skipped = False

        for seed in comparison_cfg.seeds:
            try:
                model = _build_predictor(name, seed, device, X_train, y_train,
                                          train_cfg, baseline_cfg, comparison_cfg.representative_lambda)
            except ImportError as exc:
                warnings.warn(f"[{name}] skipped: {exc}")
                skipped = True
                break

            # --- Pure ML regression quality of F_hat(t): BEFORE any
            # admission logic, decoupled from the quantum/admission loop. ---
            regression_metrics = evaluate_predictor_regression(model, X_test, y_test, device=device)

            quantum_node = QuantumRepeaterNode(T1=quantum_cfg.T1, T2=quantum_cfg.T2,
                                                depol_prob=quantum_cfg.depol_prob,
                                                shots=quantum_cfg.shots, seed=quantum_cfg.seed)
            orchestrator = DigitalTwinOrchestrator(model=model, quantum_node=quantum_node,
                                                     threshold=train_cfg.threshold, device=device)
            metrics = orchestrator.run_intelligent(X_test, y_test)

            confusion = metrics["confusion_matrix"]
            confusion_rates = compute_confusion_metrics(confusion)
            throughput = compute_throughput(metrics, cycle_time_s=comparison_cfg.cycle_time_s)
            qpu_economy = compute_qpu_economy(metrics, baseline_metrics, shots_per_attempt=quantum_cfg.shots)
            energy = compute_energy_report(metrics, baseline_metrics, shots=quantum_cfg.shots, energy_cfg=energy_cfg)
            latency_ratio_c = compute_latency_ratio(metrics["avg_classical_latency_s"], quantum_cfg.T2)

            yield_qpu_pct = (metrics["useful_pairs"] / max(metrics["attempted"], 1)) * 100.0

            seed_runs.append({
                "seed": seed,
                "useful_pairs": metrics["useful_pairs"],
                "attempted": metrics["attempted"],
                "halted": metrics["halted"],
                "qpu_yield_pct": yield_qpu_pct,
                "deficit_surplus": qpu_economy["useful_pairs_deficit_surplus"],
                "mae": regression_metrics["mae"],
                "rmse": regression_metrics["rmse"],
                "r2": regression_metrics["r2"],
                "tp": confusion["TP"], "fp": confusion["FP"],
                "tn": confusion["TN"], "fn": confusion["FN"],
                "precision": confusion_rates["precision"], "recall": confusion_rates["recall"],
                "inference_latency_ms": metrics["avg_classical_latency_s"] * 1000.0,
                "latency_ratio_c": latency_ratio_c,
                "throughput_pairs_per_s": throughput["throughput_pairs_per_s"],
                "qpu_cycles_saved": qpu_economy["qpu_cycles_saved"],
                "qpu_cycles_saved_pct": qpu_economy["qpu_cycles_saved_pct"],
                "total_energy_j": energy["total_energy_j"],
                "energy_saved_pct": energy["energy_saved_pct"],
            })

        if skipped or not seed_runs:
            continue

        per_model_seed_results[name] = seed_runs

        def _agg(key):
            return _mean_std([r[key] for r in seed_runs])

        yield_mean, yield_std = _agg("qpu_yield_pct")
        deficit_mean, deficit_std = _agg("deficit_surplus")
        mae_mean, mae_std = _agg("mae")
        rmse_mean, rmse_std = _agg("rmse")
        r2_mean, r2_std = _mean_std_dropna([r["r2"] for r in seed_runs])
        fp_mean, fp_std = _agg("fp")
        fn_mean, fn_std = _agg("fn")
        precision_mean, precision_std = _mean_std_dropna([r["precision"] for r in seed_runs])
        recall_mean, recall_std = _mean_std_dropna([r["recall"] for r in seed_runs])
        latency_mean, latency_std = _agg("inference_latency_ms")
        latency_ratio_mean, latency_ratio_std = _agg("latency_ratio_c")
        throughput_mean, throughput_std = _agg("throughput_pairs_per_s")
        cycles_saved_pct_mean, cycles_saved_pct_std = _agg("qpu_cycles_saved_pct")
        energy_mean, energy_std = _agg("total_energy_j")
        energy_saved_pct_mean, energy_saved_pct_std = _agg("energy_saved_pct")
        useful_mean, useful_std = _agg("useful_pairs")

        rows.append({
            "Model": name,
            "N Seeds": len(seed_runs),
            "Useful Pairs": f"{useful_mean:.1f} +/- {useful_std:.1f}",
            "QPU Yield (%)": f"{yield_mean:.2f} +/- {yield_std:.2f}",
            "SKR Deficit/Surplus": f"{deficit_mean:+.1f} +/- {deficit_std:.1f}",
            "MAE": f"{mae_mean:.4f} +/- {mae_std:.4f}",
            "RMSE": f"{rmse_mean:.4f} +/- {rmse_std:.4f}",
            "R2": f"{r2_mean:.4f} +/- {r2_std:.4f}",
            "FP (dead photon admitted)": f"{fp_mean:.1f} +/- {fp_std:.1f}",
            "FN (good photon discarded)": f"{fn_mean:.1f} +/- {fn_std:.1f}",
            "Precision": f"{precision_mean:.4f} +/- {precision_std:.4f}",
            "Recall": f"{recall_mean:.4f} +/- {recall_std:.4f}",
            "Throughput (pairs/s)": f"{throughput_mean:.2f} +/- {throughput_std:.2f}",
            "QPU Cycles Saved (%)": f"{cycles_saved_pct_mean:.2f} +/- {cycles_saved_pct_std:.2f}",
            "Energy (J)": f"{energy_mean:.6f} +/- {energy_std:.6f}",
            "Energy Saved (%)": f"{energy_saved_pct_mean:+.2f} +/- {energy_saved_pct_std:.2f}",
            "Inference Latency (ms)": f"{latency_mean:.4f} +/- {latency_std:.4f}",
            "C_latencia (tau_inf/T2)": f"{latency_ratio_mean:.4f} +/- {latency_ratio_std:.4f}",
            # Raw means, kept alongside the formatted strings above for
            # `build_decision_matrix` (which needs numeric values).
            "_qpu_yield_pct": yield_mean,
            "_throughput_pairs_per_s": throughput_mean,
            "_qpu_cycles_saved_pct": cycles_saved_pct_mean,
            "_energy_saved_pct": energy_saved_pct_mean,
            "_latency_ratio_c": latency_ratio_mean,
        })

        print(f"  -> QPU Yield (mean) = {yield_mean:.2f}% | MAE (mean) = {mae_mean:.4f} | "
              f"R2 (mean) = {r2_mean:.4f} | FP (mean) = {fp_mean:.1f} | FN (mean) = {fn_mean:.1f} | "
              f"Throughput (mean) = {throughput_mean:.2f} pairs/s | "
              f"QPU cycles saved (mean) = {cycles_saved_pct_mean:.2f}% | "
              f"Energy saved (mean) = {energy_saved_pct_mean:+.2f}% | "
              f"C_latencia (mean) = {latency_ratio_mean:.4f}\n")

    results_df = pd.DataFrame(rows, columns=[
        "Model", "N Seeds", "Useful Pairs", "QPU Yield (%)", "SKR Deficit/Surplus",
        "MAE", "RMSE", "R2", "FP (dead photon admitted)", "FN (good photon discarded)",
        "Precision", "Recall", "Throughput (pairs/s)", "QPU Cycles Saved (%)",
        "Energy (J)", "Energy Saved (%)", "Inference Latency (ms)", "C_latencia (tau_inf/T2)",
    ])

    decision_rows = [{
        "Model": r["Model"],
        "qpu_yield_pct": r["_qpu_yield_pct"],
        "throughput_pairs_per_s": r["_throughput_pairs_per_s"],
        "qpu_cycles_saved_pct": r["_qpu_cycles_saved_pct"],
        "energy_saved_pct": r["_energy_saved_pct"],
        "latency_ratio_c": r["_latency_ratio_c"],
    } for r in rows]
    decision_matrix_df = build_decision_matrix(decision_rows, comparison_cfg.decision_weights, model_key="Model")

    # --- Weight sensitivity analysis: is the #1 ranking an artifact of
    # the particular weights in comparison_cfg.decision_weights? ---
    print("Running decision-weight sensitivity analysis "
          f"(+/-{comparison_cfg.sensitivity_perturbation_pct*100:.0f}%, "
          f"{2**len(comparison_cfg.decision_weights)} vertex combinations)...")
    sensitivity_summary_df, sensitivity_trials_df = run_weight_sensitivity_analysis(
        decision_rows, comparison_cfg.decision_weights,
        perturbation_pct=comparison_cfg.sensitivity_perturbation_pct, model_key="Model",
    )
    sensitivity_verdict = summarize_robustness(sensitivity_summary_df, model_key="Model")
    print(f"  {sensitivity_verdict}\n")
    sensitivity_results = (sensitivity_summary_df, sensitivity_trials_df, sensitivity_verdict)

    return results_df, baseline_metrics, decision_matrix_df, per_model_seed_results, sensitivity_results


### 1.12 `qrepeater_twin/ablation.py` -- `run_ablation_study` (NEW: 2x2 factorial ablation)

The core deliverable of task 2: trains/evaluates the full
`{EdgeLSTM, StandardLSTM} x {MSE, CS-MSE}` grid and decomposes the
results into Architecture Effect, Loss Effect, and Interaction Effect --
the standard 2-factor decomposition, answering exactly the three
questions posed for the ablation study.

In [ ]:
%%writefile qrepeater_twin/ablation.py
"""
Component 9 -- Ablation study (`run_ablation_study`).

Isolates the individual and combined contribution of the two components
proposed by this work -- the `EdgeLSTM` architecture and the `CS_MSELoss`
cost-sensitive loss function -- via a full 2x2 factorial grid:

    +-------------+------------------+------------------+
    |             | MSE (plain)      | CS-MSE            |
    +-------------+------------------+------------------+
    | StandardLSTM| StandardLSTM+MSE | StandardLSTM+CS-MSE|
    | EdgeLSTM    | EdgeLSTM+MSE     | EdgeLSTM+CS-MSE     |
    +-------------+------------------+------------------+

Every cell is trained/evaluated under the EXACT SAME multi-seed protocol,
`DigitalTwinOrchestrator` loop, and `QuantumRepeaterNode` configuration as
`pareto_sweep.run_pareto_sweep` / `model_comparison.run_model_comparison`,
so results are directly, fairly comparable across cells.

2x2 factorial decomposition
----------------------------
For any numeric metric Y (e.g. `qpu_yield_pct`, `mae`, `fp`), let
`Y(A, L)` denote its mean value (across seeds) at architecture `A` in
{EdgeLSTM, StandardLSTM} and loss `L` in {MSE, CS-MSE}. This module
reports the standard factorial main-effects/interaction decomposition:

    Architecture effect = ((Y(Edge,MSE) + Y(Edge,CS-MSE)) / 2)
                         - ((Y(Std,MSE)  + Y(Std,CS-MSE))  / 2)

        "Holding the loss function fixed (averaged over both), how much
        does swapping StandardLSTM for EdgeLSTM change Y?" Answers
        "What is the impact of the EdgeLSTM architecture?"

    Loss effect         = ((Y(Edge,CS-MSE) + Y(Std,CS-MSE)) / 2)
                         - ((Y(Edge,MSE)    + Y(Std,MSE))    / 2)

        "Holding the architecture fixed (averaged over both), how much
        does swapping MSE for CS-MSE change Y?" Answers "What is the
        impact of the CS-MSE cost function?"

    Interaction effect  = (Y(Edge,CS-MSE) - Y(Edge,MSE))
                         - (Y(Std,CS-MSE)  - Y(Std,MSE))

        "Is the loss function's effect on Y THE SAME regardless of
        architecture, or does EdgeLSTM benefit from CS-MSE by a different
        amount than StandardLSTM does?" A near-zero interaction means the
        two components' contributions are simply additive (the combined
        EdgeLSTM+CS-MSE gain is fully explained by summing the two main
        effects); a large interaction means the components have a
        genuine synergistic (or antagonistic) combined effect that
        neither one alone would predict. Answers "Does the gain come from
        the COMBINATION of the two components?"

This is exactly the classical 2-factor ANOVA effect decomposition, here
applied to point-estimate means (not to per-seed variance/significance --
`decomposition_df` reports the per-seed spread of each cell alongside the
effects so the reader can judge whether the differences are large
relative to seed noise, without this module performing a formal
hypothesis test).
"""

from __future__ import annotations

import statistics as stats
from typing import List, Sequence

import pandas as pd
import torch

from .baselines import train_lstm_mse
from .config import AblationConfig, ComparisonConfig, EnergyConfig, QuantumConfig, TrainConfig
from .models import EdgeLSTM, StandardLSTM, train_edge_lstm
from .metrics import (
    compute_confusion_metrics,
    compute_energy_report,
    compute_latency_ratio,
    compute_qpu_economy,
    compute_throughput,
    evaluate_predictor_regression,
)
from .orchestrator import DigitalTwinOrchestrator
from .quantum_node import QuantumRepeaterNode

ARCHITECTURES = ["EdgeLSTM", "StandardLSTM"]
LOSSES = ["MSE", "CS-MSE"]

# Metrics where a LOWER value is better -- used only to phrase the
# human-readable interpretation strings; the raw effect numbers in
# `decomposition_df` are unaffected by this and always follow the sign
# convention documented in the module docstring (Edge/CS-MSE side minus
# StandardLSTM/MSE side).
_LOWER_IS_BETTER = {"mae", "rmse", "fp", "fn"}


def _mean_std(values: Sequence[float]) -> tuple:
    mean = stats.fmean(values)
    std = stats.pstdev(values) if len(values) > 1 else 0.0
    return mean, std


def _mean_std_dropna(values: Sequence[float]) -> tuple:
    clean = [v for v in values if v == v]  # NaN != NaN
    return _mean_std(clean) if clean else (float("nan"), float("nan"))


def _build_and_train(architecture: str, loss: str, seed: int, device: torch.device,
                      X_train: torch.Tensor, y_train: torch.Tensor,
                      train_cfg: TrainConfig, ablation_cfg: AblationConfig):
    """Builds one (architecture, loss) grid cell's model and trains it for one seed."""
    torch.manual_seed(seed)

    if architecture == "EdgeLSTM":
        model = EdgeLSTM(input_size=2, hidden_size=train_cfg.hidden_size, num_layers=1).to(device)
    elif architecture == "StandardLSTM":
        model = StandardLSTM(
            input_size=2, hidden_size=ablation_cfg.standard_lstm_hidden_size,
            num_layers=ablation_cfg.standard_lstm_num_layers,
            dropout=ablation_cfg.standard_lstm_dropout,
        ).to(device)
    else:
        raise ValueError(f"Unknown architecture: {architecture!r}")

    if loss == "MSE":
        return train_lstm_mse(model, X_train, y_train, epochs=ablation_cfg.epochs,
                               lr=ablation_cfg.lr, device=device, seed=seed)
    if loss == "CS-MSE":
        return train_edge_lstm(
            model, X_train, y_train, threshold=train_cfg.threshold,
            lambda_penalty=ablation_cfg.representative_lambda, lambda_fn=train_cfg.lambda_fn,
            discard_penalty_weight=train_cfg.discard_penalty_weight,
            max_discard_rate=train_cfg.max_discard_rate,
            epochs=ablation_cfg.epochs, lr=ablation_cfg.lr, device=device, seed=seed,
        )
    raise ValueError(f"Unknown loss: {loss!r}")


def run_ablation_study(X_train: torch.Tensor, y_train: torch.Tensor,
                        X_test: torch.Tensor, y_test: torch.Tensor, device: torch.device,
                        train_cfg: TrainConfig = None, quantum_cfg: QuantumConfig = None,
                        ablation_cfg: AblationConfig = None, energy_cfg: EnergyConfig = None,
                        comparison_cfg: ComparisonConfig = None):
    """
    Trains/evaluates all four cells of the `{StandardLSTM, EdgeLSTM} x
    {MSE, CS-MSE}` grid over `ablation_cfg.seeds`, reporting the same rich
    metrics suite as `model_comparison.run_model_comparison` per cell
    (mean +/- std across seeds), plus the 2x2 factorial decomposition for
    every metric in `ablation_cfg.headline_metrics`.

    Returns
    -------
    results_df : pd.DataFrame
        One row per grid cell (`Architecture`, `Loss`), same metric
        columns as `run_model_comparison`'s `results_df`.
    decomposition_df : pd.DataFrame
        One row per headline metric: `Architecture Effect`, `Loss Effect`,
        `Interaction Effect` (raw, signed -- see module docstring), plus a
        human-readable `Interpretation` string.
    baseline_metrics : dict
        Blind/reactive baseline metrics, computed once.
    per_cell_seed_results : dict[(str, str), list[dict]]
        Raw per-seed metrics for each (architecture, loss) cell, keyed by
        e.g. `("EdgeLSTM", "CS-MSE")`, preserved for auditing.
    """
    train_cfg = train_cfg or TrainConfig()
    quantum_cfg = quantum_cfg or QuantumConfig()
    ablation_cfg = ablation_cfg or AblationConfig()
    energy_cfg = energy_cfg or EnergyConfig()
    comparison_cfg = comparison_cfg or ComparisonConfig()

    print("Running blind/reactive baseline (unconditional admission, forced latency = 0.0)...")
    baseline_node = QuantumRepeaterNode(T1=quantum_cfg.T1, T2=quantum_cfg.T2, depol_prob=quantum_cfg.depol_prob,
                                         shots=quantum_cfg.shots, seed=quantum_cfg.seed)
    baseline_orchestrator = DigitalTwinOrchestrator(model=None, quantum_node=baseline_node,
                                                      threshold=train_cfg.threshold, device=device)
    baseline_metrics = baseline_orchestrator.run_blind_baseline(X_test, y_test)
    print(f"  Baseline: Attempts={baseline_metrics['attempted']} | "
          f"Useful pairs={baseline_metrics['useful_pairs']}\n")

    rows = []
    per_cell_seed_results = {}
    cell_metric_means = {}  # (architecture, loss) -> {metric_name: mean_value}

    for architecture in ARCHITECTURES:
        for loss in LOSSES:
            cell_name = f"{architecture}+{loss}"
            print(f"[{cell_name}] training on {len(ablation_cfg.seeds)} seeds "
                  f"({ablation_cfg.epochs} epochs each) ...")

            seed_runs = []
            for seed in ablation_cfg.seeds:
                model = _build_and_train(architecture, loss, seed, device, X_train, y_train,
                                          train_cfg, ablation_cfg)

                regression_metrics = evaluate_predictor_regression(model, X_test, y_test, device=device)

                quantum_node = QuantumRepeaterNode(T1=quantum_cfg.T1, T2=quantum_cfg.T2,
                                                    depol_prob=quantum_cfg.depol_prob,
                                                    shots=quantum_cfg.shots, seed=quantum_cfg.seed)
                orchestrator = DigitalTwinOrchestrator(model=model, quantum_node=quantum_node,
                                                         threshold=train_cfg.threshold, device=device)
                metrics = orchestrator.run_intelligent(X_test, y_test)

                confusion = metrics["confusion_matrix"]
                confusion_rates = compute_confusion_metrics(confusion)
                throughput = compute_throughput(metrics, cycle_time_s=ablation_cfg.cycle_time_s)
                qpu_economy = compute_qpu_economy(metrics, baseline_metrics, shots_per_attempt=quantum_cfg.shots)
                energy = compute_energy_report(metrics, baseline_metrics, shots=quantum_cfg.shots, energy_cfg=energy_cfg)
                latency_ratio_c = compute_latency_ratio(metrics["avg_classical_latency_s"], quantum_cfg.T2)

                yield_qpu_pct = (metrics["useful_pairs"] / max(metrics["attempted"], 1)) * 100.0

                seed_runs.append({
                    "seed": seed,
                    "useful_pairs": metrics["useful_pairs"],
                    "attempted": metrics["attempted"],
                    "halted": metrics["halted"],
                    "qpu_yield_pct": yield_qpu_pct,
                    "deficit_surplus": qpu_economy["useful_pairs_deficit_surplus"],
                    "mae": regression_metrics["mae"],
                    "rmse": regression_metrics["rmse"],
                    "r2": regression_metrics["r2"],
                    "tp": confusion["TP"], "fp": confusion["FP"],
                    "tn": confusion["TN"], "fn": confusion["FN"],
                    "precision": confusion_rates["precision"], "recall": confusion_rates["recall"],
                    "f1": confusion_rates["f1"],
                    "inference_latency_ms": metrics["avg_classical_latency_s"] * 1000.0,
                    "latency_ratio_c": latency_ratio_c,
                    "throughput_pairs_per_s": throughput["throughput_pairs_per_s"],
                    "qpu_cycles_saved_pct": qpu_economy["qpu_cycles_saved_pct"],
                    "total_energy_j": energy["total_energy_j"],
                    "energy_saved_pct": energy["energy_saved_pct"],
                })

            per_cell_seed_results[(architecture, loss)] = seed_runs

            def _agg(key):
                return _mean_std([r[key] for r in seed_runs])

            yield_mean, yield_std = _agg("qpu_yield_pct")
            deficit_mean, deficit_std = _agg("deficit_surplus")
            mae_mean, mae_std = _agg("mae")
            rmse_mean, rmse_std = _agg("rmse")
            r2_mean, r2_std = _mean_std_dropna([r["r2"] for r in seed_runs])
            fp_mean, fp_std = _agg("fp")
            fn_mean, fn_std = _agg("fn")
            precision_mean, precision_std = _mean_std_dropna([r["precision"] for r in seed_runs])
            recall_mean, recall_std = _mean_std_dropna([r["recall"] for r in seed_runs])
            f1_mean, f1_std = _mean_std_dropna([r["f1"] for r in seed_runs])
            latency_mean, latency_std = _agg("inference_latency_ms")
            latency_ratio_mean, latency_ratio_std = _agg("latency_ratio_c")
            throughput_mean, throughput_std = _agg("throughput_pairs_per_s")
            cycles_saved_pct_mean, cycles_saved_pct_std = _agg("qpu_cycles_saved_pct")
            energy_mean, energy_std = _agg("total_energy_j")
            energy_saved_pct_mean, energy_saved_pct_std = _agg("energy_saved_pct")
            useful_mean, useful_std = _agg("useful_pairs")

            cell_metric_means[(architecture, loss)] = {
                "qpu_yield_pct": yield_mean, "mae": mae_mean, "rmse": rmse_mean, "r2": r2_mean,
                "fp": fp_mean, "fn": fn_mean, "precision": precision_mean, "recall": recall_mean,
                "f1": f1_mean, "throughput_pairs_per_s": throughput_mean,
                "qpu_cycles_saved_pct": cycles_saved_pct_mean, "energy_saved_pct": energy_saved_pct_mean,
                "latency_ratio_c": latency_ratio_mean, "useful_pairs": useful_mean,
            }

            rows.append({
                "Architecture": architecture,
                "Loss": loss,
                "Model": cell_name,
                "N Seeds": len(seed_runs),
                "Useful Pairs": f"{useful_mean:.1f} +/- {useful_std:.1f}",
                "QPU Yield (%)": f"{yield_mean:.2f} +/- {yield_std:.2f}",
                "SKR Deficit/Surplus": f"{deficit_mean:+.1f} +/- {deficit_std:.1f}",
                "MAE": f"{mae_mean:.4f} +/- {mae_std:.4f}",
                "RMSE": f"{rmse_mean:.4f} +/- {rmse_std:.4f}",
                "R2": f"{r2_mean:.4f} +/- {r2_std:.4f}",
                "FP (dead photon admitted)": f"{fp_mean:.1f} +/- {fp_std:.1f}",
                "FN (good photon discarded)": f"{fn_mean:.1f} +/- {fn_std:.1f}",
                "Precision": f"{precision_mean:.4f} +/- {precision_std:.4f}",
                "Recall": f"{recall_mean:.4f} +/- {recall_std:.4f}",
                "F1": f"{f1_mean:.4f} +/- {f1_std:.4f}",
                "Throughput (pairs/s)": f"{throughput_mean:.2f} +/- {throughput_std:.2f}",
                "QPU Cycles Saved (%)": f"{cycles_saved_pct_mean:.2f} +/- {cycles_saved_pct_std:.2f}",
                "Energy (J)": f"{energy_mean:.6f} +/- {energy_std:.6f}",
                "Energy Saved (%)": f"{energy_saved_pct_mean:+.2f} +/- {energy_saved_pct_std:.2f}",
                "Inference Latency (ms)": f"{latency_mean:.4f} +/- {latency_std:.4f}",
                "C_latencia (tau_inf/T2)": f"{latency_ratio_mean:.4f} +/- {latency_ratio_std:.4f}",
            })

            print(f"  -> QPU Yield (mean) = {yield_mean:.2f}% | MAE (mean) = {mae_mean:.4f} | "
                  f"FP (mean) = {fp_mean:.1f} | FN (mean) = {fn_mean:.1f}\n")

    results_df = pd.DataFrame(rows, columns=[
        "Architecture", "Loss", "Model", "N Seeds", "Useful Pairs", "QPU Yield (%)",
        "SKR Deficit/Surplus", "MAE", "RMSE", "R2", "FP (dead photon admitted)",
        "FN (good photon discarded)", "Precision", "Recall", "F1", "Throughput (pairs/s)",
        "QPU Cycles Saved (%)", "Energy (J)", "Energy Saved (%)", "Inference Latency (ms)",
        "C_latencia (tau_inf/T2)",
    ])

    decomposition_df = _build_decomposition_df(cell_metric_means, ablation_cfg.headline_metrics)

    return results_df, decomposition_df, baseline_metrics, per_cell_seed_results


def _build_decomposition_df(cell_metric_means: dict, headline_metrics: List[str]) -> pd.DataFrame:
    """
    Builds the 2x2 factorial decomposition table (Architecture Effect /
    Loss Effect / Interaction Effect + a human-readable interpretation)
    for every metric in `headline_metrics`, from the four cells' mean
    values -- see the module docstring for the exact formulas.
    """
    rows = []
    for metric in headline_metrics:
        y_edge_mse = cell_metric_means[("EdgeLSTM", "MSE")][metric]
        y_edge_cs = cell_metric_means[("EdgeLSTM", "CS-MSE")][metric]
        y_std_mse = cell_metric_means[("StandardLSTM", "MSE")][metric]
        y_std_cs = cell_metric_means[("StandardLSTM", "CS-MSE")][metric]

        architecture_effect = ((y_edge_mse + y_edge_cs) / 2.0) - ((y_std_mse + y_std_cs) / 2.0)
        loss_effect = ((y_edge_cs + y_std_cs) / 2.0) - ((y_edge_mse + y_std_mse) / 2.0)
        interaction_effect = (y_edge_cs - y_edge_mse) - (y_std_cs - y_std_mse)

        lower_is_better = metric in _LOWER_IS_BETTER

        def _verdict(effect: float) -> str:
            if effect == 0:
                return "has no effect on"
            improves = (effect < 0) == lower_is_better
            return "improves" if improves else "worsens"

        arch_verdict = _verdict(architecture_effect)
        loss_verdict = _verdict(loss_effect)

        interpretation = (
            f"EdgeLSTM (vs StandardLSTM) {arch_verdict} '{metric}' (architecture effect = "
            f"{architecture_effect:+.4g}). CS-MSE (vs plain MSE) {loss_verdict} '{metric}' "
            f"(loss effect = {loss_effect:+.4g}). Interaction = {interaction_effect:+.4g}: "
            + ("effects are approximately additive (no strong synergy/antagonism detected)."
               if abs(interaction_effect) < 0.1 * max(abs(architecture_effect), abs(loss_effect), 1e-9)
               else "effects are NOT purely additive -- CS-MSE's benefit depends on which "
                    "architecture it is paired with, i.e. part of the gain comes from the "
                    "COMBINATION of the two components, not from either alone.")
        )

        rows.append({
            "Metric": metric,
            "EdgeLSTM+MSE": y_edge_mse,
            "EdgeLSTM+CS-MSE": y_edge_cs,
            "StandardLSTM+MSE": y_std_mse,
            "StandardLSTM+CS-MSE": y_std_cs,
            "Architecture Effect": architecture_effect,
            "Loss Effect": loss_effect,
            "Interaction Effect": interaction_effect,
            "Interpretation": interpretation,
        })

    return pd.DataFrame(rows, columns=[
        "Metric", "EdgeLSTM+MSE", "EdgeLSTM+CS-MSE", "StandardLSTM+MSE", "StandardLSTM+CS-MSE",
        "Architecture Effect", "Loss Effect", "Interaction Effect", "Interpretation",
    ])


### 1.13 `qrepeater_twin/sensitivity.py` -- decision-weight sensitivity analysis

Automatic +/-10% perturbation of every decision-matrix weight, reporting
how often each model ranks #1 across all vertex combinations. Unchanged
from v3.1.

In [ ]:
%%writefile qrepeater_twin/sensitivity.py
"""
Component 8 -- Decision-matrix weight sensitivity analysis
(`run_weight_sensitivity_analysis`).

Purpose: prove that the model ranked #1 by `metrics.build_decision_matrix`
(expected: `EdgeLSTM+CS-MSE`) wins REGARDLESS of the exact numeric weights
chosen for the decision criteria -- i.e. the conclusion is not an
artifact of having picked one particular, debatable set of weights.

Method and why it is exhaustive, not a spot-check
---------------------------------------------------
`build_decision_matrix` computes, for weights `w`, `Decision Score_m(w) =
sum_i (w_i / sum(w)) * norm_i,m` for each model `m`. Dividing by `sum(w)`
is a positive rescaling common to every model, so it can change the
*rank ordering* between two models `A`, `B` only if it flips the sign of

    Decision Score_A(w) - Decision Score_B(w) = (1 / sum(w)) * D_AB(w),
    where D_AB(w) = sum_i w_i * (norm_i,A - norm_i,B)  -- LINEAR in w.

`D_AB` is an affine (linear) function of the weight vector `w`. A linear
function over a bounded convex polytope attains BOTH its minimum and its
maximum at the polytope's vertices (standard linear-programming result).
Here the polytope is the hypercube `w_i in [(1 - pct) * base_i, (1 + pct)
* base_i]` for each criterion `i`, whose vertices are exactly the `2**n`
corner combinations of "-pct" / "+pct" per criterion. Consequently:

    Evaluating the decision matrix at all 2**n vertex combinations is
    SUFFICIENT -- not merely illustrative -- to certify whether the #1
    ranking changes anywhere inside the +/-pct box. If the same model
    wins at every vertex, `D_AB(w)` cannot change sign anywhere inside the
    box for any losing model `B` (its minimum over the box is already
    non-negative at the vertices), so that model wins EVERYWHERE inside
    the box, not just at the sampled points.

This is why the analysis below enumerates all `2**n` sign combinations
rather than randomly sampling perturbed weights.
"""

from __future__ import annotations

import itertools
from typing import Dict, List

import pandas as pd

from .metrics import build_decision_matrix


def run_weight_sensitivity_analysis(decision_rows: List[dict], base_weights: Dict[str, float],
                                     perturbation_pct: float = 0.10,
                                     model_key: str = "Model") -> tuple:
    """
    Combinatorial +/- `perturbation_pct` sensitivity analysis over the
    decision-matrix weights (see module docstring for why checking all
    `2**n` vertices is exhaustive, not a sample).

    Parameters
    ----------
    decision_rows : list[dict]
        Same input `build_decision_matrix` expects: one dict per candidate
        model, with `model_key` plus every criterion key present in
        `base_weights`.
    base_weights : dict[str, float]
        The nominal (unperturbed) decision weights, e.g.
        `ComparisonConfig.decision_weights`.
    perturbation_pct : float
        Fractional perturbation applied to each weight independently, in
        both directions (default 0.10, i.e. +/-10%).
    model_key : str
        Column identifying each row's model (default "Model").

    Returns
    -------
    summary_df : pd.DataFrame
        One row per candidate model: how many of the `2**n` vertex trials
        it won (`Wins`), the win rate (%), and its min/max Decision Score
        across all trials -- sorted by win rate, descending.
    trials_df : pd.DataFrame
        One row per vertex trial: the perturbed weight used for each
        criterion, the winning model, and its Decision Score.
    """
    if not decision_rows:
        raise ValueError("run_weight_sensitivity_analysis: decision_rows must be non-empty.")
    if not (0.0 <= perturbation_pct < 1.0):
        raise ValueError(f"run_weight_sensitivity_analysis: perturbation_pct must be in [0, 1), "
                          f"got {perturbation_pct!r}.")

    criteria = list(base_weights.keys())
    n = len(criteria)
    vertex_signs = list(itertools.product([-1, 1], repeat=n))

    trial_records = []
    winner_counts: Dict[str, int] = {}
    model_scores: Dict[str, list] = {}

    for signs in vertex_signs:
        perturbed_weights = {
            c: base_weights[c] * (1.0 + s * perturbation_pct)
            for c, s in zip(criteria, signs)
        }
        dm = build_decision_matrix(decision_rows, perturbed_weights, model_key=model_key)
        winner_row = dm.iloc[0]
        winner = winner_row[model_key]

        winner_counts[winner] = winner_counts.get(winner, 0) + 1
        trial_records.append({
            **{f"w[{c}]": perturbed_weights[c] for c in criteria},
            "Winner": winner,
            "Winner Decision Score": winner_row["Decision Score"],
        })
        for _, row in dm.iterrows():
            model_scores.setdefault(row[model_key], []).append(row["Decision Score"])

    n_trials = len(vertex_signs)
    summary_rows = []
    for model, scores in model_scores.items():
        wins = winner_counts.get(model, 0)
        summary_rows.append({
            model_key: model,
            "Wins (of 2^n vertices)": wins,
            "N Vertices": n_trials,
            "Win Rate (%)": wins / n_trials * 100.0,
            "Min Decision Score": min(scores),
            "Max Decision Score": max(scores),
        })

    summary_df = pd.DataFrame(summary_rows).sort_values(
        "Win Rate (%)", ascending=False
    ).reset_index(drop=True)
    trials_df = pd.DataFrame(trial_records)
    return summary_df, trials_df


def summarize_robustness(summary_df: pd.DataFrame, model_key: str = "Model") -> str:
    """
    Human-readable one-line verdict from `run_weight_sensitivity_analysis`'s
    `summary_df`: names the top model and states whether it won 100% of
    the `2**n` vertex trials (fully robust to +/-`perturbation_pct` weight
    bias) or only some fraction of them (rank is weight-dependent).
    """
    if summary_df.empty:
        return "No models to summarize."

    top = summary_df.iloc[0]
    model, win_rate = top[model_key], top["Win Rate (%)"]
    if win_rate >= 100.0 - 1e-9:
        return (f"'{model}' wins Rank #1 in 100% of the vertex trials -- the ranking is "
                f"PROVABLY INDEPENDENT of the exact decision-matrix weights within the "
                f"tested +/- range (see module docstring for why vertex enumeration is "
                f"exhaustive, not a sample).")
    return (f"'{model}' only wins Rank #1 in {win_rate:.1f}% of the vertex trials -- the "
            f"ranking IS sensitive to the exact decision-matrix weights within the tested "
            f"+/- range; inspect `trials_df` to see which criteria drive the flips.")


### 1.14 `qrepeater_twin/plotting.py` -- chart generators (NEW)

One function per result table produced by this project (Pareto frontier,
model/ablation comparison bars, confusion matrix, decision matrix,
sensitivity summary, temporal-error histogram, fidelity timeline with
crossings, ablation interaction plot). Every function returns a
`matplotlib.figure.Figure` and optionally saves it to disk -- shared code
between the notebook cells below and `experiment_tracking.py`.

In [ ]:
%%writefile qrepeater_twin/plotting.py
"""
Component 10 -- Plotting helpers.

Every function here takes the plain `pd.DataFrame`/`dict` outputs already
produced by `pareto_sweep.py`, `model_comparison.py`, `ablation.py`,
`sensitivity.py`, and `metrics.prediction`, and returns a
`matplotlib.figure.Figure` -- never calls `plt.show()` itself, so the
caller (notebook cell or `experiment_tracking.ExperimentRun`) decides
whether to display it inline, save it to disk, or both.

Each function accepts an optional `save_path`: when given, the figure is
also written to disk (`fig.savefig(save_path, dpi=150,
bbox_inches="tight")`) before being returned.

`_parse_mean_std_column` is the shared adapter that turns this project's
display-formatted `"12.34 +/- 1.02"` string columns (see `pareto_sweep.py`
/ `model_comparison.py` / `ablation.py`) back into numeric
`(means, stds)` arrays for plotting with error bars, so callers never need
to re-run an experiment just to get numbers instead of strings.
"""

from __future__ import annotations

import re
from typing import Sequence

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

_MEAN_STD_RE = re.compile(r"^\s*([+-]?[\d.eE+-]+)\s*\+/-\s*([\d.eE+-]+)\s*$")


def _parse_mean_std_column(series: pd.Series) -> tuple:
    """
    Parses a `"<mean> +/- <std>"` string column (as produced throughout
    this project's `results_df` tables) into two numpy float arrays
    `(means, stds)`. Raises `ValueError` on the first entry that doesn't
    match the expected format.
    """
    means, stds = [], []
    for value in series:
        m = _MEAN_STD_RE.match(str(value))
        if not m:
            raise ValueError(f"_parse_mean_std_column: could not parse {value!r} as '<mean> +/- <std>'.")
        means.append(float(m.group(1)))
        stds.append(float(m.group(2)))
    return np.array(means), np.array(stds)


def _finish(fig, save_path: str = None):
    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig


# ---------------------------------------------------------------------------
# Pareto sweep
# ---------------------------------------------------------------------------

def plot_pareto_frontier(results_df: pd.DataFrame, x_col: str = "Lambda",
                          metric_cols: Sequence[str] = ("QPU Yield (%)", "MAE"),
                          save_path: str = None):
    """
    Line plot of one or more mean +/- std metric columns against
    `x_col` (default `lambda_penalty`), one subplot per metric, sharing
    the x-axis. Designed for `pareto_sweep.run_pareto_sweep`'s
    `results_df`.
    """
    x = results_df[x_col].astype(float).to_numpy()
    n = len(metric_cols)
    fig, axes = plt.subplots(n, 1, figsize=(7, 3.2 * n), sharex=True)
    if n == 1:
        axes = [axes]

    for ax, col in zip(axes, metric_cols):
        means, stds = _parse_mean_std_column(results_df[col])
        ax.errorbar(x, means, yerr=stds, marker="o", capsize=3)
        ax.set_ylabel(col)
        ax.grid(alpha=0.3)

    axes[-1].set_xlabel(x_col)
    fig.suptitle("Pareto Frontier: mean +/- std across seeds")
    return _finish(fig, save_path)


# ---------------------------------------------------------------------------
# Model comparison / ablation bar charts
# ---------------------------------------------------------------------------

def plot_model_comparison_bars(results_df: pd.DataFrame, metric_col: str = "QPU Yield (%)",
                                model_col: str = "Model", save_path: str = None):
    """
    Bar chart of `metric_col` (a `"<mean> +/- <std>"` column) across
    models/grid cells, with error bars. Works for both
    `model_comparison.run_model_comparison`'s and
    `ablation.run_ablation_study`'s `results_df`.
    """
    means, stds = _parse_mean_std_column(results_df[metric_col])
    labels = results_df[model_col].astype(str).to_numpy()

    fig, ax = plt.subplots(figsize=(max(6, 1.1 * len(labels)), 4.5))
    x = np.arange(len(labels))
    ax.bar(x, means, yerr=stds, capsize=4)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_ylabel(metric_col)
    ax.set_title(f"{metric_col} by model (mean +/- std across seeds)")
    ax.grid(alpha=0.3, axis="y")
    return _finish(fig, save_path)


def plot_ablation_interaction(decomposition_df: pd.DataFrame, metric: str, save_path: str = None):
    """
    Classic 2x2 factorial interaction plot for one metric: x-axis = loss
    function (MSE, CS-MSE), one line per architecture (EdgeLSTM,
    StandardLSTM). Parallel lines indicate a purely additive
    (non-interacting) effect; non-parallel lines make the interaction
    effect visually explicit -- the same quantity reported numerically
    in `decomposition_df["Interaction Effect"]`.
    """
    row = decomposition_df[decomposition_df["Metric"] == metric]
    if row.empty:
        raise ValueError(f"plot_ablation_interaction: metric {metric!r} not found in decomposition_df.")
    row = row.iloc[0]

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    x = [0, 1]  # MSE, CS-MSE
    ax.plot(x, [row["EdgeLSTM+MSE"], row["EdgeLSTM+CS-MSE"]], marker="o", label="EdgeLSTM")
    ax.plot(x, [row["StandardLSTM+MSE"], row["StandardLSTM+CS-MSE"]], marker="o", label="StandardLSTM")
    ax.set_xticks(x)
    ax.set_xticklabels(["MSE", "CS-MSE"])
    ax.set_ylabel(metric)
    ax.set_title(f"Architecture x Loss interaction: {metric}")
    ax.legend()
    ax.grid(alpha=0.3)
    return _finish(fig, save_path)


# ---------------------------------------------------------------------------
# Confusion matrix / decision metrics
# ---------------------------------------------------------------------------

def plot_confusion_matrix(confusion: dict, title: str = "Admission Confusion Matrix", save_path: str = None):
    """
    2x2 heatmap of the admission confusion matrix
    `{"TP", "FP", "TN", "FN"}` (see `orchestrator.py` / `metrics.decision`
    for the ground-truth/predicted-label convention).
    """
    matrix = np.array([[confusion["TP"], confusion["FN"]],
                        [confusion["FP"], confusion["TN"]]])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Predicted Admit", "Predicted Halt"])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["True Good", "True Bad"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(matrix[i, j]), ha="center", va="center",
                     color="white" if matrix[i, j] > matrix.max() / 2 else "black")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return _finish(fig, save_path)


def plot_decision_matrix(decision_matrix_df: pd.DataFrame, model_col: str = "Model", save_path: str = None):
    """
    Horizontal bar chart of `Decision Score` by model, ordered by `Rank`
    (rank 1 at the top). Designed for `metrics.decision.build_decision_matrix`'s
    output.
    """
    df = decision_matrix_df.sort_values("Rank")
    fig, ax = plt.subplots(figsize=(6.5, max(3, 0.6 * len(df))))
    y = np.arange(len(df))
    ax.barh(y, df["Decision Score"].to_numpy())
    ax.set_yticks(y)
    ax.set_yticklabels(df[model_col].astype(str).to_numpy())
    ax.invert_yaxis()  # rank 1 on top
    ax.set_xlabel("Decision Score (higher = better)")
    ax.set_title("Multi-criteria decision matrix ranking")
    ax.grid(alpha=0.3, axis="x")
    return _finish(fig, save_path)


# ---------------------------------------------------------------------------
# Sensitivity analysis
# ---------------------------------------------------------------------------

def plot_sensitivity_summary(summary_df: pd.DataFrame, model_col: str = "Model", save_path: str = None):
    """
    Bar chart of `Win Rate (%)` per model from
    `sensitivity.run_weight_sensitivity_analysis`'s `summary_df` -- how
    often each model ranks #1 across the `2**n` +/-pct weight-perturbation
    vertices.
    """
    df = summary_df.sort_values("Win Rate (%)", ascending=False)
    fig, ax = plt.subplots(figsize=(max(6, 1.1 * len(df)), 4.5))
    x = np.arange(len(df))
    ax.bar(x, df["Win Rate (%)"].to_numpy())
    ax.set_xticks(x)
    ax.set_xticklabels(df[model_col].astype(str).to_numpy(), rotation=30, ha="right")
    ax.set_ylabel("Win Rate (%) across weight-perturbation vertices")
    ax.set_ylim(0, 105)
    ax.set_title("Decision-weight sensitivity: Rank #1 win rate")
    ax.grid(alpha=0.3, axis="y")
    return _finish(fig, save_path)


# ---------------------------------------------------------------------------
# Temporal prediction analysis
# ---------------------------------------------------------------------------

def plot_temporal_prediction_error(temporal_metrics: dict, save_path: str = None):
    """
    Histogram of matched threshold-crossing timing errors (in steps) from
    `metrics.prediction.compute_temporal_prediction_metrics` /
    `compute_controller_decision_timing`'s output. A vertical line at 0
    marks perfect timing; bars to the right of 0 are LATE detections
    (positive `timing_error_steps`), bars to the left are EARLY /
    anticipatory ones.
    """
    errors = temporal_metrics.get("timing_errors_steps", [])
    fig, ax = plt.subplots(figsize=(6, 4))
    if errors:
        ax.hist(errors, bins=min(20, max(5, len(errors))), color="tab:blue", alpha=0.8)
    ax.axvline(0.0, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("Timing error (steps); + = late/lag, - = early/anticipation")
    ax.set_ylabel("Count of matched degradation events")
    n_missed = temporal_metrics.get("n_missed_events", 0)
    n_false = temporal_metrics.get("n_false_alarms", 0)
    ax.set_title(f"Threshold-crossing timing error (missed={n_missed}, false alarms={n_false})")
    ax.grid(alpha=0.3)
    return _finish(fig, save_path)


def plot_fidelity_timeseries_with_crossings(y_true, y_pred, threshold: float,
                                             true_crossings=None, pred_crossings=None,
                                             save_path: str = None):
    """
    Line plot of the true vs. predicted fidelity trajectory, the
    admission threshold, and (optionally) marked degradation-event
    crossings for both series -- a diagnostic figure pairing directly
    with `metrics.prediction.compute_temporal_prediction_metrics`.
    """
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    t = np.arange(len(y_true))

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(t, y_true, label="True fidelity F(t)", color="tab:blue")
    ax.plot(t, y_pred, label="Predicted fidelity F_hat(t)", color="tab:orange", alpha=0.85)
    ax.axhline(threshold, color="black", linestyle="--", linewidth=1, label=f"Threshold = {threshold}")

    if true_crossings is not None and len(true_crossings) > 0:
        ax.scatter(true_crossings, [threshold] * len(true_crossings),
                   marker="v", color="tab:blue", s=60, zorder=5, label="True degradation events")
    if pred_crossings is not None and len(pred_crossings) > 0:
        ax.scatter(pred_crossings, [threshold] * len(pred_crossings),
                   marker="^", color="tab:orange", s=60, zorder=5, label="Predicted degradation events")

    ax.set_xlabel("Time step")
    ax.set_ylabel("Fidelity")
    ax.set_title("Fidelity trajectory: true vs. predicted, with degradation crossings")
    ax.legend(loc="best", fontsize=8)
    ax.grid(alpha=0.3)
    return _finish(fig, save_path)


### 1.15 `qrepeater_twin/experiment_tracking.py` -- automatic experiment pipeline (NEW)

The core deliverable of task 3: `ExperimentRun` creates a timestamped
output directory and exposes `save_config` / `save_table` / `save_figure`
/ `save_metrics` / `finalize`; `track_pareto_sweep_experiment` /
`track_model_comparison_experiment` / `track_ablation_experiment` are
ready-to-use wrappers that record a full, reproducible experiment (every
config dataclass used, every result table, every figure) with a single
function call.

In [ ]:
%%writefile qrepeater_twin/experiment_tracking.py
"""
Component 11 -- Automatic experiment-tracking pipeline (`ExperimentRun`).

Every run of `pareto_sweep.run_pareto_sweep`, `model_comparison.run_model_comparison`,
or `ablation.run_ablation_study` produces, on its own, only in-memory
Python objects (`DataFrame`s, `dict`s) -- nothing is written to disk, and
nothing records WHICH configuration (model, hyperparameters, epochs,
seeds, simulator/channel parameters) produced a given table of numbers.
Re-deriving that afterward from notebook scroll-back is exactly the
failure mode an automated experiment pipeline exists to prevent.

`ExperimentRun` wraps ONE experiment execution end-to-end: it creates a
timestamped output directory, then exposes small, focused `save_*`
methods that any of the three experiment types (or a bespoke script) can
call as they produce results. `track_pareto_sweep_experiment` /
`track_model_comparison_experiment` / `track_ablation_experiment` are
thin, ready-to-use wrappers around the three main experiment entry
points, each calling the matching `save_*` methods with the config
objects and result tables already in scope -- so a full, reproducible
experiment record is a single function call away, not something bolted
on after the fact.

Directory layout produced by one experiment (`<base_dir>/<name>_<timestamp>/`):

    config.json     -- every dataclass config used (asdict), plus the
                        device string and any extra metadata passed in.
    metrics.json     -- scalar/dict metrics not already inside a table
                         (e.g. the blind baseline's metrics dict).
    *.csv             -- every results/decomposition/decision-matrix
                          table, one file per table.
    *.png              -- every figure generated for this experiment.
    manifest.json       -- lists every file written above, plus the
                            experiment name/timestamp, so the whole
                            directory can be audited or re-loaded without
                            re-running anything.
"""

from __future__ import annotations

import dataclasses
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict

import numpy as np
import pandas as pd


def _json_default(obj):
    """`json.dumps(..., default=_json_default)`: makes numpy scalars,
    numpy arrays, and torch devices JSON-serializable without requiring
    every caller to pre-convert its metrics dict by hand."""
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return str(obj)  # last resort: torch.device, etc.


class ExperimentRun:
    """
    One tracked experiment execution: a timestamped directory plus
    small, focused methods to save configuration, tables, figures, and
    scalar metrics into it, and a `finalize()` that writes a manifest
    listing everything produced.

    Directory name: `<base_dir>/<name>_<YYYYmmdd_HHMMSS_ffffff>/` -- the
    microsecond-resolution timestamp guarantees repeated runs of the same
    experiment `name` never overwrite each other (even when created
    back-to-back within the same second, e.g. in a notebook loop), so
    every historical run stays on disk and reproducible.
    """

    def __init__(self, name: str, base_dir: str = "experiments"):
        self.name = name
        self.timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
        self.dir = Path(base_dir) / f"{name}_{self.timestamp}"
        self.dir.mkdir(parents=True, exist_ok=True)
        self._manifest: Dict[str, Any] = {
            "name": name,
            "timestamp": self.timestamp,
            "directory": str(self.dir),
            "files": [],
        }

    def _register(self, filename: str, kind: str) -> Path:
        path = self.dir / filename
        self._manifest["files"].append({"filename": filename, "kind": kind})
        return path

    def save_config(self, config: Dict[str, Any], filename: str = "config.json") -> Path:
        """
        Saves a configuration dict to JSON. Dataclass instances anywhere
        in `config` (e.g. `SimConfig(...)`, `TrainConfig(...)`) are
        converted via `dataclasses.asdict` automatically, so callers can
        pass config OBJECTS directly rather than pre-serializing them:

            exp.save_config({"sim_config": sim_cfg, "train_config": train_cfg, "device": str(device)})
        """
        serializable = {
            k: (dataclasses.asdict(v) if dataclasses.is_dataclass(v) else v)
            for k, v in config.items()
        }
        path = self._register(filename, "config")
        with open(path, "w") as f:
            json.dump(serializable, f, indent=2, default=_json_default)
        return path

    def save_table(self, df: pd.DataFrame, filename: str) -> Path:
        """Saves a `pd.DataFrame` to CSV (`filename` should end in `.csv`)."""
        path = self._register(filename, "table")
        df.to_csv(path, index=False)
        return path

    def save_metrics(self, metrics: Dict[str, Any], filename: str = "metrics.json") -> Path:
        """Saves a scalar/nested-dict metrics object to JSON."""
        path = self._register(filename, "metrics")
        with open(path, "w") as f:
            json.dump(metrics, f, indent=2, default=_json_default)
        return path

    def save_figure(self, fig, filename: str) -> Path:
        """Saves a `matplotlib.figure.Figure` to PNG (`filename` should end in `.png`)."""
        path = self._register(filename, "figure")
        fig.savefig(path, dpi=150, bbox_inches="tight")
        return path

    def finalize(self) -> Path:
        """Writes `manifest.json`, listing every artifact produced by this run."""
        path = self.dir / "manifest.json"
        with open(path, "w") as f:
            json.dump(self._manifest, f, indent=2, default=_json_default)
        return path


# ---------------------------------------------------------------------------
# Ready-to-use trackers for the three main experiment types
# ---------------------------------------------------------------------------

def track_pareto_sweep_experiment(name: str, results_df: pd.DataFrame, baseline_metrics: dict,
                                   sim_cfg, train_cfg, quantum_cfg, sweep_cfg, device,
                                   base_dir: str = "experiments") -> ExperimentRun:
    """
    Records one `pareto_sweep.run_pareto_sweep` execution: every config
    dataclass used, the resulting Pareto-frontier table, the baseline
    metrics, and a Pareto-frontier plot (QPU Yield and MAE vs. lambda).
    """
    from . import plotting  # local import: keeps matplotlib optional for callers who never plot

    exp = ExperimentRun(name, base_dir=base_dir)
    exp.save_config({
        "experiment_kind": "pareto_sweep",
        "device": str(device),
        "sim_config": sim_cfg, "train_config": train_cfg,
        "quantum_config": quantum_cfg, "sweep_config": sweep_cfg,
    })
    exp.save_table(results_df, "pareto_frontier.csv")
    exp.save_metrics(baseline_metrics, "baseline_metrics.json")
    fig = plotting.plot_pareto_frontier(results_df)
    exp.save_figure(fig, "pareto_frontier.png")
    exp.finalize()
    return exp


def track_model_comparison_experiment(name: str, results_df: pd.DataFrame, baseline_metrics: dict,
                                       decision_matrix_df: pd.DataFrame, sensitivity_results: tuple,
                                       sim_cfg, train_cfg, quantum_cfg, baseline_cfg,
                                       comparison_cfg, device, base_dir: str = "experiments") -> ExperimentRun:
    """
    Records one `model_comparison.run_model_comparison` execution: every
    config dataclass used, the per-model results table, the decision
    matrix, the sensitivity-analysis summary, and three plots (QPU Yield
    bar chart, decision-matrix ranking, sensitivity win-rate).
    """
    from . import plotting

    sensitivity_summary_df, sensitivity_trials_df, sensitivity_verdict = sensitivity_results

    exp = ExperimentRun(name, base_dir=base_dir)
    exp.save_config({
        "experiment_kind": "model_comparison",
        "device": str(device),
        "sim_config": sim_cfg, "train_config": train_cfg, "quantum_config": quantum_cfg,
        "baseline_config": baseline_cfg, "comparison_config": comparison_cfg,
    })
    exp.save_table(results_df, "model_comparison.csv")
    exp.save_table(decision_matrix_df, "decision_matrix.csv")
    exp.save_table(sensitivity_summary_df, "sensitivity_summary.csv")
    exp.save_table(sensitivity_trials_df, "sensitivity_trials.csv")
    exp.save_metrics({**baseline_metrics, "sensitivity_verdict": sensitivity_verdict}, "baseline_metrics.json")

    exp.save_figure(plotting.plot_model_comparison_bars(results_df), "qpu_yield_by_model.png")
    exp.save_figure(plotting.plot_decision_matrix(decision_matrix_df), "decision_matrix.png")
    exp.save_figure(plotting.plot_sensitivity_summary(sensitivity_summary_df), "sensitivity_summary.png")
    exp.finalize()
    return exp


def track_ablation_experiment(name: str, results_df: pd.DataFrame, decomposition_df: pd.DataFrame,
                               baseline_metrics: dict, sim_cfg, train_cfg, quantum_cfg,
                               ablation_cfg, device, base_dir: str = "experiments") -> ExperimentRun:
    """
    Records one `ablation.run_ablation_study` execution: every config
    dataclass used, the per-cell results table, the 2x2 factorial
    decomposition table, a per-cell QPU-Yield bar chart, and one
    architecture x loss interaction plot per headline metric in
    `ablation_cfg.headline_metrics`.
    """
    from . import plotting

    exp = ExperimentRun(name, base_dir=base_dir)
    exp.save_config({
        "experiment_kind": "ablation",
        "device": str(device),
        "sim_config": sim_cfg, "train_config": train_cfg,
        "quantum_config": quantum_cfg, "ablation_config": ablation_cfg,
    })
    exp.save_table(results_df, "ablation_results.csv")
    exp.save_table(decomposition_df, "ablation_decomposition.csv")
    exp.save_metrics(baseline_metrics, "baseline_metrics.json")

    exp.save_figure(plotting.plot_model_comparison_bars(results_df, metric_col="QPU Yield (%)",
                                                          model_col="Model"), "qpu_yield_by_cell.png")
    for metric in decomposition_df["Metric"]:
        fig = plotting.plot_ablation_interaction(decomposition_df, metric)
        exp.save_figure(fig, f"interaction_{metric}.png")
    exp.finalize()
    return exp


### 1.16 `qrepeater_twin/cli.py` -- `main()` and the command-line entry point

In [ ]:
%%writefile qrepeater_twin/cli.py
"""
Command-line entry point.

Previously, `main()` lived inside the notebook and could only be run cell
by cell, inside a Jupyter kernel. Here it is an ordinary, importable
library function (`from qrepeater_twin.cli import main`) and is also
runnable via `python -m qrepeater_twin.cli` from a plain terminal --
no notebook required.
"""

from __future__ import annotations

import argparse

import numpy as np
import torch

from .channel_simulator import WDMChannelSimulator
from .config import (
    AblationConfig,
    BaselineConfig,
    ComparisonConfig,
    EnergyConfig,
    QuantumConfig,
    SimConfig,
    SweepConfig,
    TrainConfig,
)
from .pareto_sweep import run_pareto_sweep
from .model_comparison import run_model_comparison
from .ablation import run_ablation_study


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def main(sim_cfg: SimConfig = None, train_cfg: TrainConfig = None,
          quantum_cfg: QuantumConfig = None, sweep_cfg: SweepConfig = None,
          device: torch.device = None, base_seed: int = 42,
          run_baseline_comparison: bool = False,
          baseline_cfg: BaselineConfig = None, energy_cfg: EnergyConfig = None,
          comparison_cfg: ComparisonConfig = None,
          run_ablation: bool = False, ablation_cfg: AblationConfig = None):
    """
    Full pipeline of the Quantum Repeater Digital Twin (v3.2, decomposed).

    1. Generates and preprocesses the synthetic dataset (WDMChannelSimulator).
    2. Moves tensors to the selected device (CPU/GPU).
    3. Runs the (multi-seed) Pareto Frontier sweep over lambda_penalty,
       including the one-time computation of the blind/reactive baseline.
    4. Prints the consolidated metrics table.
    5. (Optional, `run_baseline_comparison=True`) Runs the cross-architecture
       baseline comparison (LSTM+MSE, Random Forest, XGBoost, Transformer)
       with regression accuracy (MAE/RMSE/R^2), the admission confusion
       matrix (FP/FN/TP/TN), throughput/QPU-economy/energy accounting, the
       dimensionless C_latencia = tau_inf/T2 ratio, the ranked decision
       matrix, and an automatic +/-10% decision-weight sensitivity
       analysis.
    6. (Optional, `run_ablation=True`) Runs the 2x2 factorial ablation
       study ({EdgeLSTM, StandardLSTM} x {MSE, CS-MSE}), reporting the
       architecture/loss/interaction effect decomposition.
    """
    sim_cfg = sim_cfg or SimConfig()
    train_cfg = train_cfg or TrainConfig()
    quantum_cfg = quantum_cfg or QuantumConfig()
    sweep_cfg = sweep_cfg or SweepConfig()
    baseline_cfg = baseline_cfg or BaselineConfig()
    energy_cfg = energy_cfg or EnergyConfig()
    comparison_cfg = comparison_cfg or ComparisonConfig()
    ablation_cfg = ablation_cfg or AblationConfig()
    device = device or get_device()

    torch.manual_seed(base_seed)
    np.random.seed(base_seed)

    print("=" * 88)
    print(" QUANTUM REPEATER DIGITAL TWIN -- PARETO FRONTIER (CS_MSELoss) ".center(88, "="))
    print("=" * 88)
    print(f"\nDevice: {device}")

    # -----------------------------------------------------------------
    # 1) Data generation and preprocessing
    # -----------------------------------------------------------------
    print("\n[1/3] Generating synthetic dataset (Ornstein-Uhlenbeck) ...")
    wdm_sim = WDMChannelSimulator(n_steps=sim_cfg.n_steps, dt=sim_cfg.dt, seed=sim_cfg.seed)
    df = wdm_sim.generate_dataset()
    X_train, y_train, X_test, y_test, feat_scaler = wdm_sim.preprocess(
        df, window_size=sim_cfg.window_size, test_size=sim_cfg.test_size,
    )
    print(f"    Total samples: {len(df)} | Training windows: {len(X_train)} | Test windows: {len(X_test)}")
    print(f"    Fraction of true fidelity below threshold {train_cfg.threshold}: "
          f"{(df['fidelity'] < train_cfg.threshold).mean() * 100:.1f}%")

    # -----------------------------------------------------------------
    # 2) Device handling: moves tensors to the selected device
    # -----------------------------------------------------------------
    X_train, y_train = X_train.to(device), y_train.to(device)
    X_test, y_test = X_test.to(device), y_test.to(device)

    # -----------------------------------------------------------------
    # 3) Pareto Frontier sweep over lambda_penalty (multi-seed averaging)
    # -----------------------------------------------------------------
    print(f"\n[2/3] Running the Pareto Frontier for lambda_penalty = "
          f"{sweep_cfg.lambda_values} with {len(sweep_cfg.seeds)} seeds/point "
          f"({sweep_cfg.seeds}) ...\n")

    results_df, baseline_metrics, per_seed_results = run_pareto_sweep(
        sweep_cfg.lambda_values, X_train, y_train, X_test, y_test, device=device,
        threshold=train_cfg.threshold, epochs=train_cfg.epochs, lr=train_cfg.lr,
        hidden_size=train_cfg.hidden_size,
        T1=quantum_cfg.T1, T2=quantum_cfg.T2, depol_prob=quantum_cfg.depol_prob,
        shots=quantum_cfg.shots, quantum_seed=quantum_cfg.seed,
        lambda_fn=train_cfg.lambda_fn, discard_penalty_weight=train_cfg.discard_penalty_weight,
        max_discard_rate=train_cfg.max_discard_rate, seeds=sweep_cfg.seeds,
    )

    # -----------------------------------------------------------------
    # Final report
    # -----------------------------------------------------------------
    print("[3/3] Consolidating results ...\n")
    print("=" * 88)
    print(" BASELINE (Blind/Reactive Purification) ".center(88, "="))
    print("=" * 88)
    print(f"  Total cycles evaluated    : {baseline_metrics['total_steps']}")
    print(f"  Purification attempts     : {baseline_metrics['attempted']} (unconditional admission)")
    print(f"  Useful pairs obtained     : {baseline_metrics['useful_pairs']}")
    print(f"  Forced classical latency  : {baseline_metrics['avg_classical_latency_s']*1000:.4f} ms "
          f"(neural network never invoked)")
    print(f"  Confusion (degenerate)    : TP={baseline_metrics['confusion_matrix']['TP']} "
          f"FP={baseline_metrics['confusion_matrix']['FP']} "
          f"(every dead photon is admitted, by construction -- FN=TN=0)")

    print("\n" + "=" * 88)
    print(" PARETO FRONTIER: CS_MSELoss(lambda_penalty) -- mean +/- std (multi-seed) "
          .center(88, "="))
    print("=" * 88)
    print(results_df.to_string(index=False))
    print("=" * 88)

    comparison_results = None
    if run_baseline_comparison:
        # -----------------------------------------------------------------
        # 4) (Optional) Cross-architecture baseline comparison
        # -----------------------------------------------------------------
        print("\n[4/5] Running cross-architecture baseline comparison "
              f"(LSTM+MSE, Random Forest, XGBoost, Transformer) with "
              f"{len(comparison_cfg.seeds)} seeds/model ...\n")

        comp_results_df, comp_baseline_metrics, decision_matrix_df, per_model_seed_results, sensitivity_results = run_model_comparison(
            X_train, y_train, X_test, y_test, device=device,
            train_cfg=train_cfg, quantum_cfg=quantum_cfg, baseline_cfg=baseline_cfg,
            energy_cfg=energy_cfg, comparison_cfg=comparison_cfg,
        )

        print("\n" + "=" * 88)
        print(" BASELINE COMPARISON: regression accuracy / confusion / throughput / QPU economy / energy "
              .center(88, "="))
        print("=" * 88)
        print(comp_results_df.to_string(index=False))

        print("\n" + "=" * 88)
        print(" DECISION MATRIX (weighted, higher Decision Score = better; Rank 1 = recommended) "
              .center(88, "="))
        print("=" * 88)
        print(decision_matrix_df.to_string(index=False))
        print("=" * 88)

        sensitivity_summary_df, sensitivity_trials_df, sensitivity_verdict = sensitivity_results
        print("\n" + "=" * 88)
        print(f" DECISION-WEIGHT SENSITIVITY (+/-{comparison_cfg.sensitivity_perturbation_pct*100:.0f}%, "
              f"{2**len(comparison_cfg.decision_weights)} vertex combinations) ".center(88, "="))
        print("=" * 88)
        print(sensitivity_summary_df.to_string(index=False))
        print(f"\n  Verdict: {sensitivity_verdict}")
        print("=" * 88)

        comparison_results = (comp_results_df, comp_baseline_metrics, decision_matrix_df,
                               per_model_seed_results, sensitivity_results)

    ablation_results = None
    if run_ablation:
        # -----------------------------------------------------------------
        # 5) (Optional) 2x2 factorial ablation study
        # -----------------------------------------------------------------
        print(f"\n[5/5] Running 2x2 factorial ablation study "
              f"({{EdgeLSTM, StandardLSTM}} x {{MSE, CS-MSE}}) with "
              f"{len(ablation_cfg.seeds)} seeds/cell ...\n")

        ablation_results_df, decomposition_df, ablation_baseline_metrics, per_cell_seed_results = run_ablation_study(
            X_train, y_train, X_test, y_test, device=device,
            train_cfg=train_cfg, quantum_cfg=quantum_cfg, ablation_cfg=ablation_cfg,
            energy_cfg=energy_cfg, comparison_cfg=comparison_cfg,
        )

        print("\n" + "=" * 88)
        print(" ABLATION STUDY: {EdgeLSTM, StandardLSTM} x {MSE, CS-MSE} ".center(88, "="))
        print("=" * 88)
        print(ablation_results_df.to_string(index=False))

        print("\n" + "=" * 88)
        print(" ABLATION DECOMPOSITION: architecture effect / loss effect / interaction ".center(88, "="))
        print("=" * 88)
        print(decomposition_df[["Metric", "Architecture Effect", "Loss Effect", "Interaction Effect"]].to_string(index=False))
        for _, row in decomposition_df.iterrows():
            print(f"\n  [{row['Metric']}] {row['Interpretation']}")
        print("=" * 88)

        ablation_results = (ablation_results_df, decomposition_df, ablation_baseline_metrics, per_cell_seed_results)

    if not run_baseline_comparison and not run_ablation:
        return results_df, baseline_metrics, per_seed_results
    if not run_ablation:
        return results_df, baseline_metrics, per_seed_results, comparison_results
    return results_df, baseline_metrics, per_seed_results, comparison_results, ablation_results


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Quantum Repeater Digital Twin -- Pareto Frontier")
    parser.add_argument("--epochs", type=int, default=150, help="Training epochs per round (seed x lambda).")
    parser.add_argument("--lr", type=float, default=0.012, help="Adam learning rate.")
    parser.add_argument("--hidden-size", type=int, default=16, help="EdgeLSTM hidden state size.")
    parser.add_argument("--seeds", type=int, nargs="+", default=[42, 43, 44, 45, 46],
                         help="Seeds used in multi-seed averaging (one training round per seed x lambda).")
    parser.add_argument("--lambda-values", type=float, nargs="+", default=[1.0, 2.0, 5.0, 10.0, 20.0, 50.0],
                         help="lambda_penalty values swept over the Pareto Frontier.")
    parser.add_argument("--compare-baselines", action="store_true",
                         help="Also run the cross-architecture baseline comparison "
                              "(LSTM+MSE, Random Forest, XGBoost, Transformer) with "
                              "throughput/QPU-economy/energy metrics and the decision matrix.")
    parser.add_argument("--no-xgboost", action="store_true",
                         help="Skip the XGBoost baseline even if the 'xgboost' package is installed.")
    parser.add_argument("--representative-lambda", type=float, default=10.0,
                         help="lambda_penalty used for the EdgeLSTM+CS-MSE row in --compare-baselines.")
    parser.add_argument("--sensitivity-pct", type=float, default=0.10,
                         help="+/- fractional perturbation applied to each decision-matrix weight "
                              "in the automatic weight-sensitivity analysis (default 0.10 = +/-10%%).")
    parser.add_argument("--run-ablation", action="store_true",
                         help="Also run the 2x2 factorial ablation study "
                              "({EdgeLSTM, StandardLSTM} x {MSE, CS-MSE}), reporting the "
                              "architecture/loss/interaction effect decomposition.")
    return parser


if __name__ == "__main__":
    args = build_arg_parser().parse_args()
    main(
        train_cfg=TrainConfig(epochs=args.epochs, lr=args.lr, hidden_size=args.hidden_size),
        sweep_cfg=SweepConfig(lambda_values=args.lambda_values, seeds=args.seeds),
        run_baseline_comparison=args.compare_baselines,
        comparison_cfg=ComparisonConfig(
            representative_lambda=args.representative_lambda,
            seeds=args.seeds,
            include_xgboost=not args.no_xgboost,
            sensitivity_perturbation_pct=args.sensitivity_pct,
        ),
        run_ablation=args.run_ablation,
        ablation_cfg=AblationConfig(seeds=args.seeds, representative_lambda=args.representative_lambda),
    )


### 1.17 Tests (`tests/`)

`pytest` suite covering every module above, including the new temporal
prediction-timing logic, the ablation factorial decomposition, the
plotting functions, and the experiment-tracking pipeline.

In [ ]:
%%writefile tests/__init__.py


In [ ]:
%%writefile tests/test_channel_simulator.py
import numpy as np

from qrepeater_twin import WDMChannelSimulator


def test_generate_dataset_shape_and_bounds():
    sim = WDMChannelSimulator(n_steps=500, dt=0.01, seed=1)
    df = sim.generate_dataset()

    assert len(df) == 500
    assert set(df.columns) == {"phase_deviation", "temp_gradient", "fidelity"}
    assert (df["phase_deviation"] >= 0).all()
    assert (df["temp_gradient"] >= 0).all()
    assert (df["fidelity"] >= 0).all() and (df["fidelity"] <= 1).all()


def test_generate_dataset_is_deterministic_given_seed():
    df_a = WDMChannelSimulator(n_steps=200, seed=7).generate_dataset()
    df_b = WDMChannelSimulator(n_steps=200, seed=7).generate_dataset()
    assert np.allclose(df_a.values, df_b.values)


def test_preprocess_shapes():
    sim = WDMChannelSimulator(n_steps=500, dt=0.01, seed=1)
    df = sim.generate_dataset()
    X_train, y_train, X_test, y_test, scaler = sim.preprocess(df, window_size=20, test_size=0.2)

    n_windows = len(df) - 20
    expected_train = int(n_windows * 0.8)
    expected_test = n_windows - expected_train

    assert X_train.shape == (expected_train, 20, 2)
    assert y_train.shape == (expected_train, 1)
    assert X_test.shape == (expected_test, 20, 2)
    assert y_test.shape == (expected_test, 1)


In [ ]:
%%writefile tests/test_models_and_timing.py
import torch

from qrepeater_twin import CS_MSELoss, EdgeLSTM, InferenceTimer, StandardLSTM, train_edge_lstm


def test_edge_lstm_output_shape_and_range():
    torch.manual_seed(0)
    model = EdgeLSTM(input_size=2, hidden_size=8, num_layers=1)
    x = torch.rand(4, 20, 2)
    out = model(x)
    assert out.shape == (4, 1)
    assert torch.all(out >= 0.0) and torch.all(out <= 1.0)


def test_standard_lstm_output_shape_and_range():
    torch.manual_seed(0)
    model = StandardLSTM(input_size=2, hidden_size=32, num_layers=2, dropout=0.1)
    x = torch.rand(4, 20, 2)
    out = model(x)
    assert out.shape == (4, 1)
    assert torch.all(out >= 0.0) and torch.all(out <= 1.0)


def test_standard_lstm_single_layer_ignores_inter_layer_dropout():
    # nn.LSTM only accepts dropout > 0 when num_layers > 1; StandardLSTM
    # must silently zero it out for num_layers == 1 rather than raising.
    torch.manual_seed(0)
    model = StandardLSTM(input_size=2, hidden_size=16, num_layers=1, dropout=0.5)
    x = torch.rand(2, 20, 2)
    out = model(x)
    assert out.shape == (2, 1)


def test_standard_lstm_has_more_parameters_than_edge_lstm_by_default():
    # The whole point of StandardLSTM as the ablation's architecture
    # counterpart is that it is NOT edge-optimized (larger capacity).
    edge = EdgeLSTM(input_size=2, hidden_size=16, num_layers=1)
    standard = StandardLSTM(input_size=2, hidden_size=64, num_layers=2, dropout=0.1)
    n_params_edge = sum(p.numel() for p in edge.parameters())
    n_params_standard = sum(p.numel() for p in standard.parameters())
    assert n_params_standard > n_params_edge


def test_cs_mse_loss_penalizes_false_positive_more_than_false_negative():
    threshold = 0.65
    loss = CS_MSELoss(threshold=threshold, lambda_penalty=10.0, lambda_fn=2.0,
                       discard_penalty_weight=0.0, max_discard_rate=1.0)

    # False positive: ground truth below threshold, prediction above.
    y_true_fp = torch.tensor([[0.5]])
    y_pred_fp = torch.tensor([[0.9]])

    # False negative: ground truth above threshold, prediction below (same absolute error).
    y_true_fn = torch.tensor([[0.9]])
    y_pred_fn = torch.tensor([[0.5]])

    loss_fp = loss(y_pred_fp, y_true_fp).item()
    loss_fn = loss(y_pred_fn, y_true_fn).item()

    assert loss_fp > loss_fn


def test_train_edge_lstm_reduces_loss():
    torch.manual_seed(0)
    X = torch.rand(16, 20, 2)
    y = torch.rand(16, 1)

    model = EdgeLSTM(input_size=2, hidden_size=8, num_layers=1)
    criterion = CS_MSELoss()
    with torch.no_grad():
        loss_before = criterion(model(X), y).item()

    model = train_edge_lstm(model, X, y, epochs=50, lr=1e-2, seed=0)

    with torch.no_grad():
        loss_after = criterion(model(X), y).item()

    assert loss_after < loss_before


def test_inference_timer_cpu_returns_nonnegative_elapsed():
    device = torch.device("cpu")
    model = EdgeLSTM(input_size=2, hidden_size=8, num_layers=1)
    x = torch.rand(1, 20, 2)

    with InferenceTimer(device) as timer:
        _ = model(x)

    assert timer.elapsed_s >= 0.0
    assert timer.use_cuda_events is False


In [ ]:
%%writefile tests/test_baselines.py
import pytest
import torch

from qrepeater_twin import EdgeLSTM, TinyTransformer
from qrepeater_twin.baselines import (
    RandomForestFidelityModel,
    train_lstm_mse,
    train_random_forest,
    train_transformer,
    train_xgboost,
)


def test_train_lstm_mse_reduces_loss():
    torch.manual_seed(0)
    X = torch.rand(16, 20, 2)
    y = torch.rand(16, 1)

    model = EdgeLSTM(input_size=2, hidden_size=8, num_layers=1)
    criterion = torch.nn.MSELoss()
    with torch.no_grad():
        loss_before = criterion(model(X), y).item()

    model = train_lstm_mse(model, X, y, epochs=50, lr=1e-2, seed=0)

    with torch.no_grad():
        loss_after = criterion(model(X), y).item()

    assert loss_after < loss_before


def test_random_forest_fits_and_predicts_in_unit_range():
    torch.manual_seed(0)
    X_train = torch.rand(32, 20, 2)
    y_train = torch.rand(32, 1)

    rf_model = RandomForestFidelityModel(n_estimators=20, max_depth=4, seed=0)
    rf_model.fit(X_train, y_train)
    orchestrator_model = rf_model.as_orchestrator_model()

    x_sample = torch.rand(1, 20, 2)
    orchestrator_model.eval()
    pred = orchestrator_model(x_sample)

    assert pred.shape == (1, 1)
    assert torch.all(pred >= 0.0) and torch.all(pred <= 1.0)


def test_train_random_forest_convenience_function():
    torch.manual_seed(0)
    X_train = torch.rand(32, 20, 2)
    y_train = torch.rand(32, 1)

    model = train_random_forest(X_train, y_train, n_estimators=10, max_depth=3, seed=0)
    pred = model(torch.rand(1, 20, 2))
    assert pred.shape == (1, 1)


def test_train_xgboost_skips_cleanly_when_not_installed():
    pytest.importorskip("xgboost", reason="xgboost is an optional dependency for this baseline")
    torch.manual_seed(0)
    X_train = torch.rand(32, 20, 2)
    y_train = torch.rand(32, 1)
    model = train_xgboost(X_train, y_train, n_estimators=10, max_depth=3, seed=0)
    pred = model(torch.rand(1, 20, 2))
    assert pred.shape == (1, 1)


def test_tiny_transformer_output_shape_and_range():
    torch.manual_seed(0)
    model = TinyTransformer(input_size=2, d_model=8, nhead=2, num_layers=1, dim_feedforward=16)
    x = torch.rand(4, 20, 2)
    out = model(x)
    assert out.shape == (4, 1)
    assert torch.all(out >= 0.0) and torch.all(out <= 1.0)


def test_train_transformer_reduces_loss():
    torch.manual_seed(0)
    X = torch.rand(16, 20, 2)
    y = torch.rand(16, 1)

    model = TinyTransformer(input_size=2, d_model=8, nhead=2, num_layers=1, dim_feedforward=16)
    criterion = torch.nn.MSELoss()
    with torch.no_grad():
        loss_before = criterion(model(X), y).item()

    model = train_transformer(model, X, y, epochs=60, lr=5e-3, seed=0)

    with torch.no_grad():
        loss_after = criterion(model(X), y).item()

    assert loss_after < loss_before


In [ ]:
%%writefile tests/test_metrics.py
import math

import pytest
import torch

from qrepeater_twin.config import EnergyConfig
from qrepeater_twin.metrics import (
    build_decision_matrix,
    compute_confusion_metrics,
    compute_energy_report,
    compute_latency_ratio,
    compute_qpu_economy,
    compute_regression_metrics,
    compute_throughput,
    evaluate_predictor_regression,
)


def _baseline_metrics():
    return {"total_steps": 780, "useful_pairs": 300, "halted": 0,
            "attempted": 780, "avg_classical_latency_s": 0.0}


def _intelligent_metrics():
    return {"total_steps": 780, "useful_pairs": 280, "halted": 300,
            "attempted": 480, "avg_classical_latency_s": 0.00012}


# ---------------------------------------------------------------------------
# compute_regression_metrics / evaluate_predictor_regression
# ---------------------------------------------------------------------------

def test_compute_regression_metrics_matches_manual_computation():
    y_true = [0.9, 0.8, 0.5, 0.3, 0.95]
    y_pred = [0.85, 0.82, 0.55, 0.25, 0.90]

    result = compute_regression_metrics(y_true, y_pred)

    errors = [p - t for p, t in zip(y_pred, y_true)]
    mae_expected = sum(abs(e) for e in errors) / len(errors)
    rmse_expected = math.sqrt(sum(e ** 2 for e in errors) / len(errors))

    assert result["mae"] == pytest.approx(mae_expected)
    assert result["rmse"] == pytest.approx(rmse_expected)
    assert 0.0 < result["r2"] <= 1.0


def test_compute_regression_metrics_perfect_prediction():
    y = [0.9, 0.8, 0.5, 0.3, 0.95]
    result = compute_regression_metrics(y, y)
    assert result["mae"] == pytest.approx(0.0, abs=1e-12)
    assert result["rmse"] == pytest.approx(0.0, abs=1e-12)
    assert result["r2"] == pytest.approx(1.0)


def test_compute_regression_metrics_constant_y_true_gives_nan_r2():
    result = compute_regression_metrics([0.5, 0.5, 0.5], [0.4, 0.5, 0.6])
    assert math.isnan(result["r2"])


def test_compute_regression_metrics_accepts_torch_tensors():
    y_true = torch.tensor([[0.9], [0.8], [0.5]])
    y_pred = torch.tensor([[0.85], [0.82], [0.55]])
    result = compute_regression_metrics(y_true, y_pred)
    assert result["mae"] > 0.0


def test_compute_regression_metrics_shape_mismatch_raises():
    with pytest.raises(ValueError):
        compute_regression_metrics([0.1, 0.2], [0.1, 0.2, 0.3])


def test_evaluate_predictor_regression_uses_model_forward_pass():
    torch.manual_seed(0)

    class _ConstantPredictor:
        """Minimal stand-in for a trained model: always predicts 0.7."""

        def eval(self):
            return self

        def __call__(self, x):
            return torch.full((x.shape[0], 1), 0.7)

    X_test = torch.rand(10, 20, 2)
    y_test = torch.full((10, 1), 0.7)

    result = evaluate_predictor_regression(_ConstantPredictor(), X_test, y_test)
    assert result["mae"] == pytest.approx(0.0, abs=1e-6)
    assert result["r2"] == pytest.approx(1.0)


# ---------------------------------------------------------------------------
# compute_confusion_metrics
# ---------------------------------------------------------------------------

def test_compute_confusion_metrics_matches_manual_rates():
    confusion = {"TP": 80, "FP": 20, "TN": 150, "FN": 30}
    result = compute_confusion_metrics(confusion)

    assert result["precision"] == pytest.approx(80 / 100)
    assert result["recall"] == pytest.approx(80 / 110)
    assert result["specificity"] == pytest.approx(150 / 170)
    assert result["fpr"] == pytest.approx(20 / 170)
    assert result["fnr"] == pytest.approx(30 / 110)
    assert 0.0 < result["f1"] < 1.0


def test_compute_confusion_metrics_degenerate_zero_positives_precision_is_nan():
    # No admissions were ever predicted (TP == FP == 0): precision is
    # undefined (0/0), not zero.
    confusion = {"TP": 0, "FP": 0, "TN": 10, "FN": 0}
    result = compute_confusion_metrics(confusion)
    assert math.isnan(result["precision"])
    assert math.isnan(result["recall"])  # TP + FN == 0 too


def test_compute_confusion_metrics_blind_baseline_shape():
    # The blind baseline's degenerate confusion matrix: FN == TN == 0
    # always (unconditional admission -- see orchestrator.run_blind_baseline).
    confusion = {"TP": 300, "FP": 480, "TN": 0, "FN": 0}
    result = compute_confusion_metrics(confusion)
    assert result["recall"] == pytest.approx(1.0)  # no good photon ever discarded
    assert result["specificity"] == pytest.approx(0.0)  # TN=0, FP=480 -> 0/480
    assert result["fnr"] == pytest.approx(0.0)  # FN=0, TP=300 -> 0/300


# ---------------------------------------------------------------------------
# compute_latency_ratio (C_latencia = tau_inf / T2)
# ---------------------------------------------------------------------------

def test_compute_latency_ratio_matches_manual_division():
    ratio = compute_latency_ratio(1e-4, 30e-6)
    assert ratio == pytest.approx(1e-4 / 30e-6)


def test_compute_latency_ratio_is_dimensionless_and_scale_invariant():
    # Scaling both tau_inf and T2 by the same factor leaves C_latencia unchanged.
    r1 = compute_latency_ratio(1e-4, 30e-6)
    r2 = compute_latency_ratio(2e-4, 60e-6)
    assert r1 == pytest.approx(r2)


def test_compute_latency_ratio_zero_t2_raises():
    with pytest.raises(ValueError):
        compute_latency_ratio(1e-4, 0.0)


def test_compute_latency_ratio_negative_t2_raises():
    with pytest.raises(ValueError):
        compute_latency_ratio(1e-4, -30e-6)


# ---------------------------------------------------------------------------
# compute_throughput / compute_qpu_economy / compute_energy_report
# ---------------------------------------------------------------------------

def test_compute_throughput_positive_and_bounded_by_blind():
    baseline = compute_throughput(_baseline_metrics(), cycle_time_s=1e-3)
    intelligent = compute_throughput(_intelligent_metrics(), cycle_time_s=1e-3)

    assert baseline["throughput_pairs_per_s"] > 0
    assert intelligent["throughput_pairs_per_s"] > 0
    # The predictive controller adds classical latency on every cycle
    # (blind never pays it), so for equal cycle_time_s its total_time_s
    # is >= the blind baseline's.
    assert intelligent["total_time_s"] >= baseline["total_time_s"]


def test_compute_qpu_economy_reports_positive_savings_when_halting():
    economy = compute_qpu_economy(_intelligent_metrics(), _baseline_metrics(), shots_per_attempt=512)

    assert economy["qpu_cycles_saved"] == 300
    assert economy["qpu_cycles_saved_pct"] == pytest.approx(300 / 780 * 100.0)
    assert economy["qpu_shots_saved"] == 300 * 512
    assert economy["useful_pairs_deficit_surplus"] == -20


def test_compute_qpu_economy_without_shots_leaves_shots_saved_none():
    economy = compute_qpu_economy(_intelligent_metrics(), _baseline_metrics())
    assert economy["qpu_shots_saved"] is None


def test_compute_energy_report_zero_cycles_saved_means_equal_quantum_energy():
    # A "predictor" that never halts (attempted == baseline attempted)
    # must show identical *quantum*-side energy to the baseline; only the
    # classical inference term can differ.
    always_purify = dict(_intelligent_metrics())
    always_purify["attempted"] = 780
    always_purify["halted"] = 0

    report = compute_energy_report(always_purify, _baseline_metrics(), shots=512)
    baseline_quantum_only = compute_energy_report(_baseline_metrics(), _baseline_metrics(), shots=512)
    assert report["quantum_energy_j"] == pytest.approx(baseline_quantum_only["quantum_energy_j"])


def test_compute_energy_report_custom_energy_config_scales_linearly():
    cfg_low = EnergyConfig(joules_per_1q_gate=1e-9, joules_per_2q_gate=1e-9, joules_per_shot_overhead=0.0,
                            classical_inference_power_w=0.0, classical_idle_power_w=0.0)
    cfg_high = EnergyConfig(joules_per_1q_gate=2e-9, joules_per_2q_gate=2e-9, joules_per_shot_overhead=0.0,
                             classical_inference_power_w=0.0, classical_idle_power_w=0.0)

    metrics = _intelligent_metrics()
    baseline = _baseline_metrics()

    report_low = compute_energy_report(metrics, baseline, shots=512, energy_cfg=cfg_low)
    report_high = compute_energy_report(metrics, baseline, shots=512, energy_cfg=cfg_high)

    assert report_high["quantum_energy_j"] == pytest.approx(2 * report_low["quantum_energy_j"])


# ---------------------------------------------------------------------------
# build_decision_matrix (latency_ratio_c as the cost-type latency criterion)
# ---------------------------------------------------------------------------

def test_build_decision_matrix_ranks_dominant_model_first():
    rows = [
        {
            "Model": "Blind",
            "qpu_yield_pct": 38.46,
            "throughput_pairs_per_s": 384.6,
            "qpu_cycles_saved_pct": 0.0,
            "energy_saved_pct": 0.0,
            "latency_ratio_c": 0.0,
        },
        {
            "Model": "Dominant",
            "qpu_yield_pct": 90.0,
            "throughput_pairs_per_s": 500.0,
            "qpu_cycles_saved_pct": 50.0,
            "energy_saved_pct": 40.0,
            "latency_ratio_c": 0.02,
        },
    ]
    weights = {
        "qpu_yield_pct": 0.25,
        "throughput_pairs_per_s": 0.20,
        "qpu_cycles_saved_pct": 0.20,
        "energy_saved_pct": 0.20,
        "latency_ratio_c": 0.15,
    }

    dm = build_decision_matrix(rows, weights)

    assert dm.iloc[0]["Model"] == "Dominant"
    assert dm.iloc[0]["Rank"] == 1
    assert dm.iloc[0]["Decision Score"] > dm.iloc[1]["Decision Score"]


def test_build_decision_matrix_latency_ratio_c_is_cost_type():
    # Two models identical except for latency_ratio_c: the one with the
    # SMALLER ratio (closer to zero, i.e. negligible next to T2) must win.
    rows = [
        {"Model": "LowLatency", "qpu_yield_pct": 50.0, "latency_ratio_c": 0.001},
        {"Model": "HighLatency", "qpu_yield_pct": 50.0, "latency_ratio_c": 0.5},
    ]
    weights = {"qpu_yield_pct": 0.5, "latency_ratio_c": 0.5}
    dm = build_decision_matrix(rows, weights)
    assert dm.iloc[0]["Model"] == "LowLatency"


def test_build_decision_matrix_missing_criterion_raises():
    rows = [{"Model": "A", "qpu_yield_pct": 50.0}]
    weights = {"qpu_yield_pct": 0.5, "throughput_pairs_per_s": 0.5}
    with pytest.raises(KeyError):
        build_decision_matrix(rows, weights)


def test_build_decision_matrix_empty_rows_returns_empty_frame():
    dm = build_decision_matrix([], {"qpu_yield_pct": 1.0})
    assert dm.empty



In [ ]:
%%writefile tests/test_metrics_prediction.py
import math

import numpy as np
import pytest

from qrepeater_twin.metrics import (
    compute_controller_decision_timing,
    compute_fidelity_statistics,
    compute_temporal_prediction_metrics,
    extract_fidelity_arrays_from_log,
    find_threshold_crossings,
    match_crossings,
)


# ---------------------------------------------------------------------------
# find_threshold_crossings
# ---------------------------------------------------------------------------

def test_find_threshold_crossings_falling_interpolates_correctly():
    # Crosses 0.65 between index 2 (0.8) and index 3 (0.6): linear
    # interpolation places it 75% of the way from 2 to 3.
    series = [1.0, 0.9, 0.8, 0.6, 0.4, 0.3]
    crossings = find_threshold_crossings(series, 0.65, direction="falling")
    assert len(crossings) == 1
    assert crossings[0] == pytest.approx(2.75)


def test_find_threshold_crossings_rising():
    series = [0.3, 0.4, 0.7, 0.9]
    crossings = find_threshold_crossings(series, 0.65, direction="rising")
    assert len(crossings) == 1
    assert 1.0 < crossings[0] < 2.0


def test_find_threshold_crossings_no_crossing_returns_empty():
    series = [0.9, 0.9, 0.9]
    crossings = find_threshold_crossings(series, 0.65, direction="falling")
    assert len(crossings) == 0


def test_find_threshold_crossings_multiple_events():
    # Degrades below 0.65 twice, recovers in between.
    series = [0.9, 0.5, 0.9, 0.4, 0.9]
    crossings = find_threshold_crossings(series, 0.65, direction="falling")
    assert len(crossings) == 2


def test_find_threshold_crossings_invalid_direction_raises():
    with pytest.raises(ValueError):
        find_threshold_crossings([0.9, 0.5], 0.65, direction="sideways")


# ---------------------------------------------------------------------------
# match_crossings
# ---------------------------------------------------------------------------

def test_match_crossings_pairs_nearest_neighbors():
    true_c = [10.0, 50.0]
    pred_c = [11.0, 49.5]
    matches, missed, false_alarms = match_crossings(true_c, pred_c)
    assert len(matches) == 2
    assert not missed and not false_alarms
    pairing = {ti: pi for ti, pi, _ in matches}
    assert pairing[0] == 0 and pairing[1] == 1


def test_match_crossings_respects_max_distance():
    true_c = [10.0, 100.0]
    pred_c = [11.0]  # only close to true[0]
    matches, missed, false_alarms = match_crossings(true_c, pred_c, max_distance=5.0)
    assert len(matches) == 1
    assert missed == [1]
    assert false_alarms == []


def test_match_crossings_empty_inputs():
    matches, missed, false_alarms = match_crossings([], [])
    assert matches == [] and missed == [] and false_alarms == []


def test_match_crossings_all_false_alarms_when_no_true_events():
    matches, missed, false_alarms = match_crossings([], [5.0, 12.0])
    assert matches == []
    assert missed == []
    assert false_alarms == [0, 1]


# ---------------------------------------------------------------------------
# compute_temporal_prediction_metrics
# ---------------------------------------------------------------------------

def _step_degradation(n, drop_at, shift=0.0, slope=0.04):
    t = np.arange(n)
    return np.clip(1.0 - slope * np.maximum(t - drop_at - shift, 0), 0.0, 1.0)


def test_compute_temporal_prediction_metrics_detects_lag():
    y_true = _step_degradation(60, drop_at=25, shift=0.0)
    y_pred = _step_degradation(60, drop_at=25, shift=2.0)  # predictor lags by 2 steps

    result = compute_temporal_prediction_metrics(y_true, y_pred, threshold=0.65, dt=1e-3)

    assert result["n_true_events"] == 1
    assert result["n_pred_events"] == 1
    assert result["n_matched"] == 1
    assert result["mean_timing_error_steps"] == pytest.approx(2.0, abs=1e-6)
    assert result["mean_timing_error_s"] == pytest.approx(2.0e-3, abs=1e-9)
    assert not result["matched_pairs_df"].empty


def test_compute_temporal_prediction_metrics_detects_anticipation():
    y_true = _step_degradation(60, drop_at=25, shift=0.0)
    y_pred = _step_degradation(60, drop_at=25, shift=-3.0)  # predictor anticipates by 3 steps

    result = compute_temporal_prediction_metrics(y_true, y_pred, threshold=0.65)
    assert result["mean_timing_error_steps"] == pytest.approx(-3.0, abs=1e-6)


def test_compute_temporal_prediction_metrics_false_alarm():
    y_true = np.ones(30) * 0.9  # never degrades
    y_pred = np.concatenate([np.ones(15) * 0.9, [0.5], np.ones(14) * 0.9])

    result = compute_temporal_prediction_metrics(y_true, y_pred, threshold=0.65)
    assert result["n_true_events"] == 0
    assert result["n_false_alarms"] == result["n_pred_events"]
    assert result["n_false_alarms"] >= 1


def test_compute_temporal_prediction_metrics_missed_event():
    y_true = _step_degradation(60, drop_at=25, shift=0.0)
    y_pred = np.ones(60) * 0.9  # predictor never flags degradation at all

    result = compute_temporal_prediction_metrics(y_true, y_pred, threshold=0.65)
    assert result["n_true_events"] == 1
    assert result["n_matched"] == 0
    assert result["n_missed_events"] == 1


def test_compute_temporal_prediction_metrics_shape_mismatch_raises():
    with pytest.raises(ValueError):
        compute_temporal_prediction_metrics([0.1, 0.2], [0.1, 0.2, 0.3])


def test_compute_temporal_prediction_metrics_perfect_prediction_zero_error():
    y_true = _step_degradation(60, drop_at=25, shift=0.0)
    result = compute_temporal_prediction_metrics(y_true, y_true, threshold=0.65)
    assert result["mean_timing_error_steps"] == pytest.approx(0.0, abs=1e-9)
    assert result["mean_abs_timing_error_steps"] == pytest.approx(0.0, abs=1e-9)


# ---------------------------------------------------------------------------
# extract_fidelity_arrays_from_log / compute_controller_decision_timing
# ---------------------------------------------------------------------------

def _fake_log(true_vals, pred_vals, threshold=0.65):
    log = []
    for i, (t, p) in enumerate(zip(true_vals, pred_vals)):
        action = "PURIFY" if p >= threshold else "HALT_PURIFICATION"
        log.append({"step": i, "action": action, "true_fidelity": t, "pred_fidelity": p})
    return log


def test_extract_fidelity_arrays_from_log_orders_by_step():
    log = _fake_log([0.9, 0.8, 0.5], [0.85, 0.82, 0.55])
    # Shuffle the log to confirm sorting by 'step' is enforced.
    shuffled = [log[2], log[0], log[1]]
    true_arr, pred_arr = extract_fidelity_arrays_from_log(shuffled)
    assert list(true_arr) == [0.9, 0.8, 0.5]
    assert list(pred_arr) == [0.85, 0.82, 0.55]


def test_extract_fidelity_arrays_from_log_empty_raises():
    with pytest.raises(ValueError):
        extract_fidelity_arrays_from_log([])


def test_extract_fidelity_arrays_from_log_missing_pred_fidelity_raises():
    # Mimics the blind baseline's log entries (PURIFY_BLIND has no pred_fidelity).
    blind_log = [{"step": 0, "action": "PURIFY_BLIND", "true_fidelity": 0.8}]
    with pytest.raises(ValueError):
        extract_fidelity_arrays_from_log(blind_log)


def test_compute_controller_decision_timing_matches_manual_computation():
    y_true = _step_degradation(60, drop_at=25, shift=0.0)
    y_pred = _step_degradation(60, drop_at=25, shift=2.0)
    log = _fake_log(list(y_true), list(y_pred))

    result_from_log = compute_controller_decision_timing(log, threshold=0.65)
    result_direct = compute_temporal_prediction_metrics(y_true, y_pred, threshold=0.65)

    assert result_from_log["mean_timing_error_steps"] == pytest.approx(result_direct["mean_timing_error_steps"])
    assert result_from_log["n_matched"] == result_direct["n_matched"]


# ---------------------------------------------------------------------------
# compute_fidelity_statistics
# ---------------------------------------------------------------------------

def test_compute_fidelity_statistics_basic_shape():
    y_true = _step_degradation(60, drop_at=25, shift=0.0)
    stats = compute_fidelity_statistics(y_true, threshold=0.65)

    assert 0.0 <= stats["mean_fidelity"] <= 1.0
    assert stats["min_fidelity"] <= stats["mean_fidelity"] <= stats["max_fidelity"]
    assert stats["n_degradation_events"] == 1
    assert 0.0 <= stats["pct_below_threshold"] <= 100.0


def test_compute_fidelity_statistics_empty_raises():
    with pytest.raises(ValueError):
        compute_fidelity_statistics([])


def test_compute_fidelity_statistics_constant_series_no_events():
    y_true = np.ones(20) * 0.95
    stats = compute_fidelity_statistics(y_true, threshold=0.65)
    assert stats["n_degradation_events"] == 0
    assert stats["n_recovery_events"] == 0
    assert stats["pct_below_threshold"] == pytest.approx(0.0)


In [ ]:
%%writefile tests/test_sensitivity.py
import itertools

import pytest

from qrepeater_twin.sensitivity import run_weight_sensitivity_analysis, summarize_robustness


BASE_WEIGHTS = {
    "qpu_yield_pct": 0.25,
    "throughput_pairs_per_s": 0.20,
    "qpu_cycles_saved_pct": 0.20,
    "energy_saved_pct": 0.20,
    "latency_ratio_c": 0.15,
}


def _dominant_rows():
    return [
        {"Model": "Blind", "qpu_yield_pct": 38.46, "throughput_pairs_per_s": 384.6,
         "qpu_cycles_saved_pct": 0.0, "energy_saved_pct": 0.0, "latency_ratio_c": 0.0},
        {"Model": "LSTM+MSE", "qpu_yield_pct": 60.0, "throughput_pairs_per_s": 420.0,
         "qpu_cycles_saved_pct": 20.0, "energy_saved_pct": 10.0, "latency_ratio_c": 0.01},
        {"Model": "RandomForest", "qpu_yield_pct": 55.0, "throughput_pairs_per_s": 400.0,
         "qpu_cycles_saved_pct": 15.0, "energy_saved_pct": 5.0, "latency_ratio_c": 0.005},
        {"Model": "EdgeLSTM+CS-MSE", "qpu_yield_pct": 90.0, "throughput_pairs_per_s": 500.0,
         "qpu_cycles_saved_pct": 50.0, "energy_saved_pct": 40.0, "latency_ratio_c": 0.02},
    ]


def test_vertex_enumeration_covers_2_pow_n_combinations():
    summary_df, trials_df = run_weight_sensitivity_analysis(_dominant_rows(), BASE_WEIGHTS, perturbation_pct=0.10)
    assert len(trials_df) == 2 ** len(BASE_WEIGHTS)


def test_dominant_model_wins_every_vertex():
    summary_df, trials_df = run_weight_sensitivity_analysis(_dominant_rows(), BASE_WEIGHTS, perturbation_pct=0.10)

    top = summary_df.iloc[0]
    assert top["Model"] == "EdgeLSTM+CS-MSE"
    assert top["Win Rate (%)"] == pytest.approx(100.0)
    assert top["Wins (of 2^n vertices)"] == 2 ** len(BASE_WEIGHTS)
    # Every other model must have zero wins.
    assert (summary_df.iloc[1:]["Win Rate (%)"] == 0.0).all()


def test_summarize_robustness_reports_full_robustness():
    summary_df, _ = run_weight_sensitivity_analysis(_dominant_rows(), BASE_WEIGHTS, perturbation_pct=0.10)
    verdict = summarize_robustness(summary_df)
    assert "EdgeLSTM+CS-MSE" in verdict
    assert "100%" in verdict or "PROVABLY INDEPENDENT" in verdict


def test_ambiguous_case_is_detected_as_weight_sensitive():
    # Two criteria, equal base weights, each model dominant on one
    # criterion: genuinely ambiguous -- winner must flip across some
    # vertices of the +/-10% hypercube.
    rows = [
        {"Model": "A", "crit1": 100.0, "crit2": 0.0},
        {"Model": "B", "crit1": 0.0, "crit2": 100.0},
    ]
    weights = {"crit1": 0.5, "crit2": 0.5}
    summary_df, trials_df = run_weight_sensitivity_analysis(rows, weights, perturbation_pct=0.10)

    assert summary_df.iloc[0]["Win Rate (%)"] < 100.0
    assert set(trials_df["Winner"]) == {"A", "B"}

    verdict = summarize_robustness(summary_df)
    assert "sensitive" in verdict.lower()


def test_vertex_weights_are_within_perturbation_bounds():
    _, trials_df = run_weight_sensitivity_analysis(_dominant_rows(), BASE_WEIGHTS, perturbation_pct=0.10)
    for criterion, base in BASE_WEIGHTS.items():
        col = trials_df[f"w[{criterion}]"]
        assert (col >= base * 0.9 - 1e-12).all()
        assert (col <= base * 1.1 + 1e-12).all()


def test_all_sign_combinations_are_present_exactly_once():
    _, trials_df = run_weight_sensitivity_analysis(_dominant_rows(), BASE_WEIGHTS, perturbation_pct=0.10)
    signs_seen = set()
    for _, row in trials_df.iterrows():
        signs = tuple(
            1 if row[f"w[{c}]"] > BASE_WEIGHTS[c] else -1
            for c in BASE_WEIGHTS
        )
        signs_seen.add(signs)
    assert signs_seen == set(itertools.product([-1, 1], repeat=len(BASE_WEIGHTS)))


def test_empty_rows_raises():
    with pytest.raises(ValueError):
        run_weight_sensitivity_analysis([], BASE_WEIGHTS)


def test_invalid_perturbation_pct_raises():
    with pytest.raises(ValueError):
        run_weight_sensitivity_analysis(_dominant_rows(), BASE_WEIGHTS, perturbation_pct=1.0)
    with pytest.raises(ValueError):
        run_weight_sensitivity_analysis(_dominant_rows(), BASE_WEIGHTS, perturbation_pct=-0.1)


def test_summarize_robustness_empty_summary():
    import pandas as pd
    empty_df = pd.DataFrame(columns=["Model", "Win Rate (%)"])
    result = summarize_robustness(empty_df)
    assert "No models" in result


In [ ]:
%%writefile tests/test_ablation.py
import pytest
import torch

from qrepeater_twin.ablation import _build_decomposition_df, run_ablation_study
from qrepeater_twin.config import AblationConfig, QuantumConfig, TrainConfig


# ---------------------------------------------------------------------------
# _build_decomposition_df: pure arithmetic, no training involved
# ---------------------------------------------------------------------------

def test_build_decomposition_df_matches_manual_factorial_formulas():
    cell_metric_means = {
        ("EdgeLSTM", "MSE"): {"qpu_yield_pct": 60.0, "mae": 0.05},
        ("EdgeLSTM", "CS-MSE"): {"qpu_yield_pct": 90.0, "mae": 0.04},
        ("StandardLSTM", "MSE"): {"qpu_yield_pct": 55.0, "mae": 0.06},
        ("StandardLSTM", "CS-MSE"): {"qpu_yield_pct": 65.0, "mae": 0.055},
    }
    df = _build_decomposition_df(cell_metric_means, ["qpu_yield_pct", "mae"])

    row = df[df["Metric"] == "qpu_yield_pct"].iloc[0]
    # Architecture effect = ((60+90)/2) - ((55+65)/2) = 75 - 60 = 15
    assert row["Architecture Effect"] == pytest.approx(15.0)
    # Loss effect = ((90+65)/2) - ((60+55)/2) = 77.5 - 57.5 = 20
    assert row["Loss Effect"] == pytest.approx(20.0)
    # Interaction = (90-60) - (65-55) = 30 - 10 = 20
    assert row["Interaction Effect"] == pytest.approx(20.0)


def test_build_decomposition_df_zero_interaction_when_purely_additive():
    # Architecture adds +10 regardless of loss; loss adds +5 regardless of
    # architecture -> perfectly additive, interaction must be exactly 0.
    cell_metric_means = {
        ("EdgeLSTM", "MSE"): {"metric": 100.0},
        ("EdgeLSTM", "CS-MSE"): {"metric": 105.0},
        ("StandardLSTM", "MSE"): {"metric": 90.0},
        ("StandardLSTM", "CS-MSE"): {"metric": 95.0},
    }
    df = _build_decomposition_df(cell_metric_means, ["metric"])
    row = df.iloc[0]
    assert row["Interaction Effect"] == pytest.approx(0.0, abs=1e-9)
    assert row["Architecture Effect"] == pytest.approx(10.0)
    assert row["Loss Effect"] == pytest.approx(5.0)


def test_build_decomposition_df_interpretation_mentions_additivity_when_small():
    cell_metric_means = {
        ("EdgeLSTM", "MSE"): {"metric": 100.0},
        ("EdgeLSTM", "CS-MSE"): {"metric": 105.0},
        ("StandardLSTM", "MSE"): {"metric": 90.0},
        ("StandardLSTM", "CS-MSE"): {"metric": 95.0},
    }
    df = _build_decomposition_df(cell_metric_means, ["metric"])
    assert "additive" in df.iloc[0]["Interpretation"].lower()


def test_build_decomposition_df_interpretation_flags_lower_is_better_correctly():
    # For 'mae' (lower is better): EdgeLSTM has LOWER mae than StandardLSTM
    # at both loss settings -> architecture effect is negative (Edge - Std < 0)
    # and must be reported as an IMPROVEMENT, not a regression.
    cell_metric_means = {
        ("EdgeLSTM", "MSE"): {"mae": 0.03},
        ("EdgeLSTM", "CS-MSE"): {"mae": 0.02},
        ("StandardLSTM", "MSE"): {"mae": 0.05},
        ("StandardLSTM", "CS-MSE"): {"mae": 0.045},
    }
    df = _build_decomposition_df(cell_metric_means, ["mae"])
    row = df.iloc[0]
    assert row["Architecture Effect"] < 0  # Edge's mae is lower (better)
    assert "improves" in row["Interpretation"].lower()


# ---------------------------------------------------------------------------
# run_ablation_study: light structural / smoke test (tiny epochs, 1 seed)
# ---------------------------------------------------------------------------

def test_run_ablation_study_returns_all_four_grid_cells():
    torch.manual_seed(0)
    device = torch.device("cpu")

    X_train = torch.rand(24, 10, 2)
    y_train = torch.rand(24, 1)
    X_test = torch.rand(12, 10, 2)
    y_test = torch.rand(12, 1)

    train_cfg = TrainConfig(hidden_size=4, epochs=3, threshold=0.65)
    quantum_cfg = QuantumConfig(shots=8)
    ablation_cfg = AblationConfig(
        standard_lstm_hidden_size=6, standard_lstm_num_layers=1, standard_lstm_dropout=0.0,
        epochs=3, lr=0.01, representative_lambda=5.0, seeds=[0],
    )

    results_df, decomposition_df, baseline_metrics, per_cell_seed_results = run_ablation_study(
        X_train, y_train, X_test, y_test, device,
        train_cfg=train_cfg, quantum_cfg=quantum_cfg, ablation_cfg=ablation_cfg,
    )

    assert len(results_df) == 4
    assert set(zip(results_df["Architecture"], results_df["Loss"])) == {
        ("EdgeLSTM", "MSE"), ("EdgeLSTM", "CS-MSE"),
        ("StandardLSTM", "MSE"), ("StandardLSTM", "CS-MSE"),
    }
    assert len(decomposition_df) == len(ablation_cfg.headline_metrics)
    assert set(per_cell_seed_results.keys()) == {
        ("EdgeLSTM", "MSE"), ("EdgeLSTM", "CS-MSE"),
        ("StandardLSTM", "MSE"), ("StandardLSTM", "CS-MSE"),
    }
    assert baseline_metrics["total_steps"] == 12


In [ ]:
%%writefile tests/test_plotting.py
import matplotlib
matplotlib.use("Agg")  # headless backend for test environments without a display

import numpy as np
import pandas as pd
import pytest
from matplotlib.figure import Figure

from qrepeater_twin import plotting


# ---------------------------------------------------------------------------
# _parse_mean_std_column
# ---------------------------------------------------------------------------

def test_parse_mean_std_column_basic():
    series = pd.Series(["12.34 +/- 1.02", "-5.0 +/- 0.3", "0.0000 +/- 0.0001"])
    means, stds = plotting._parse_mean_std_column(series)
    assert np.allclose(means, [12.34, -5.0, 0.0])
    assert np.allclose(stds, [1.02, 0.3, 0.0001])


def test_parse_mean_std_column_malformed_raises():
    series = pd.Series(["not a number"])
    with pytest.raises(ValueError):
        plotting._parse_mean_std_column(series)


# ---------------------------------------------------------------------------
# Every plotting function: smoke test that it returns a Figure and doesn't crash
# ---------------------------------------------------------------------------

def test_plot_pareto_frontier_returns_figure():
    df = pd.DataFrame({
        "Lambda": [1.0, 2.0, 5.0],
        "QPU Yield (%)": ["50.0 +/- 2.0", "60.0 +/- 1.5", "70.0 +/- 1.0"],
        "MAE": ["0.05 +/- 0.01", "0.04 +/- 0.008", "0.03 +/- 0.005"],
    })
    fig = plotting.plot_pareto_frontier(df)
    assert isinstance(fig, Figure)


def test_plot_model_comparison_bars_returns_figure():
    df = pd.DataFrame({"Model": ["A", "B", "C"], "QPU Yield (%)": ["50 +/- 2", "60 +/- 1", "55 +/- 3"]})
    fig = plotting.plot_model_comparison_bars(df)
    assert isinstance(fig, Figure)


def test_plot_confusion_matrix_returns_figure():
    fig = plotting.plot_confusion_matrix({"TP": 80, "FP": 20, "TN": 150, "FN": 30})
    assert isinstance(fig, Figure)


def test_plot_decision_matrix_returns_figure():
    dm = pd.DataFrame({"Model": ["A", "B"], "Decision Score": [0.8, 0.5], "Rank": [1, 2]})
    fig = plotting.plot_decision_matrix(dm)
    assert isinstance(fig, Figure)


def test_plot_sensitivity_summary_returns_figure():
    sm = pd.DataFrame({"Model": ["A", "B"], "Win Rate (%)": [100.0, 20.0]})
    fig = plotting.plot_sensitivity_summary(sm)
    assert isinstance(fig, Figure)


def test_plot_temporal_prediction_error_returns_figure_with_events():
    tm = {"timing_errors_steps": [1.0, -2.0, 0.5, 3.0], "n_missed_events": 1, "n_false_alarms": 0}
    fig = plotting.plot_temporal_prediction_error(tm)
    assert isinstance(fig, Figure)


def test_plot_temporal_prediction_error_handles_no_matched_events():
    # No matched events at all (e.g. predictor missed every degradation) --
    # must not crash on an empty histogram.
    tm = {"timing_errors_steps": [], "n_missed_events": 3, "n_false_alarms": 0}
    fig = plotting.plot_temporal_prediction_error(tm)
    assert isinstance(fig, Figure)


def test_plot_fidelity_timeseries_with_crossings_returns_figure():
    y_true = np.linspace(1, 0, 50)
    y_pred = y_true + 0.02
    fig = plotting.plot_fidelity_timeseries_with_crossings(
        y_true, y_pred, 0.65, true_crossings=[17.5], pred_crossings=[16.8],
    )
    assert isinstance(fig, Figure)


def test_plot_fidelity_timeseries_with_crossings_handles_no_crossings():
    y_true = np.ones(20) * 0.9
    y_pred = np.ones(20) * 0.9
    fig = plotting.plot_fidelity_timeseries_with_crossings(y_true, y_pred, 0.65)
    assert isinstance(fig, Figure)


def test_plot_ablation_interaction_returns_figure():
    decomp = pd.DataFrame({
        "Metric": ["qpu_yield_pct"],
        "EdgeLSTM+MSE": [60.0], "EdgeLSTM+CS-MSE": [90.0],
        "StandardLSTM+MSE": [55.0], "StandardLSTM+CS-MSE": [65.0],
        "Architecture Effect": [15.0], "Loss Effect": [20.0], "Interaction Effect": [20.0],
        "Interpretation": ["test"],
    })
    fig = plotting.plot_ablation_interaction(decomp, "qpu_yield_pct")
    assert isinstance(fig, Figure)


def test_plot_ablation_interaction_unknown_metric_raises():
    decomp = pd.DataFrame({
        "Metric": ["qpu_yield_pct"],
        "EdgeLSTM+MSE": [60.0], "EdgeLSTM+CS-MSE": [90.0],
        "StandardLSTM+MSE": [55.0], "StandardLSTM+CS-MSE": [65.0],
        "Architecture Effect": [15.0], "Loss Effect": [20.0], "Interaction Effect": [20.0],
        "Interpretation": ["test"],
    })
    with pytest.raises(ValueError):
        plotting.plot_ablation_interaction(decomp, "nonexistent_metric")


def test_save_path_actually_writes_a_file(tmp_path):
    df = pd.DataFrame({"Model": ["A", "B"], "QPU Yield (%)": ["50 +/- 2", "60 +/- 1"]})
    save_path = tmp_path / "chart.png"
    plotting.plot_model_comparison_bars(df, save_path=str(save_path))
    assert save_path.exists()
    assert save_path.stat().st_size > 0


In [ ]:
%%writefile tests/test_experiment_tracking.py
import dataclasses
import json

import pandas as pd
import pytest

from qrepeater_twin.experiment_tracking import (
    ExperimentRun,
    track_ablation_experiment,
    track_model_comparison_experiment,
    track_pareto_sweep_experiment,
)


@dataclasses.dataclass
class _DummyConfig:
    a: int = 1
    b: str = "x"


# ---------------------------------------------------------------------------
# ExperimentRun: low-level save_* / finalize() behavior
# ---------------------------------------------------------------------------

def test_experiment_run_creates_timestamped_directory(tmp_path):
    exp = ExperimentRun("my_experiment", base_dir=str(tmp_path))
    assert exp.dir.exists()
    assert exp.dir.parent == tmp_path
    assert exp.dir.name.startswith("my_experiment_")


def test_experiment_run_two_instances_get_distinct_directories(tmp_path):
    exp1 = ExperimentRun("same_name", base_dir=str(tmp_path))
    exp2 = ExperimentRun("same_name", base_dir=str(tmp_path))
    # Directories differ (by timestamp) even for identical names -- no run
    # ever silently overwrites a previous one.
    assert exp1.dir != exp2.dir


def test_save_config_serializes_dataclasses_and_plain_values(tmp_path):
    exp = ExperimentRun("cfg_test", base_dir=str(tmp_path))
    exp.save_config({"sim_config": _DummyConfig(), "device": "cpu", "note": "hello"})

    with open(exp.dir / "config.json") as f:
        loaded = json.load(f)

    assert loaded == {"sim_config": {"a": 1, "b": "x"}, "device": "cpu", "note": "hello"}


def test_save_table_writes_readable_csv(tmp_path):
    exp = ExperimentRun("table_test", base_dir=str(tmp_path))
    df = pd.DataFrame({"Lambda": [1, 2], "Yield": ["50 +/- 1", "60 +/- 2"]})
    exp.save_table(df, "results.csv")

    reloaded = pd.read_csv(exp.dir / "results.csv")
    assert list(reloaded["Lambda"]) == [1, 2]
    assert list(reloaded["Yield"]) == ["50 +/- 1", "60 +/- 2"]


def test_save_metrics_writes_valid_json(tmp_path):
    exp = ExperimentRun("metrics_test", base_dir=str(tmp_path))
    exp.save_metrics({"total_steps": 100, "useful_pairs": 42})

    with open(exp.dir / "metrics.json") as f:
        loaded = json.load(f)
    assert loaded == {"total_steps": 100, "useful_pairs": 42}


def test_finalize_manifest_lists_every_saved_file(tmp_path):
    exp = ExperimentRun("manifest_test", base_dir=str(tmp_path))
    exp.save_config({"a": 1})
    exp.save_table(pd.DataFrame({"x": [1]}), "t.csv")
    exp.save_metrics({"m": 1})
    manifest_path = exp.finalize()

    with open(manifest_path) as f:
        manifest = json.load(f)

    filenames = {entry["filename"] for entry in manifest["files"]}
    assert filenames == {"config.json", "t.csv", "metrics.json"}
    assert manifest["name"] == "manifest_test"


def test_save_metrics_handles_numpy_scalars_without_crashing(tmp_path):
    np = pytest.importorskip("numpy")
    exp = ExperimentRun("numpy_test", base_dir=str(tmp_path))
    exp.save_metrics({"mae": np.float64(0.05), "count": np.int64(42)})
    with open(exp.dir / "metrics.json") as f:
        loaded = json.load(f)
    assert loaded["mae"] == pytest.approx(0.05)
    assert loaded["count"] == 42


# ---------------------------------------------------------------------------
# track_pareto_sweep_experiment / track_model_comparison_experiment / track_ablation_experiment
# ---------------------------------------------------------------------------

def test_track_pareto_sweep_experiment_writes_expected_artifacts(tmp_path):
    results_df = pd.DataFrame({
        "Lambda": [1.0, 2.0],
        "QPU Yield (%)": ["50.0 +/- 2.0", "60.0 +/- 1.0"],
        "MAE": ["0.05 +/- 0.01", "0.03 +/- 0.005"],
    })
    baseline_metrics = {"total_steps": 100, "useful_pairs": 40, "attempted": 100,
                         "avg_classical_latency_s": 0.0}

    exp = track_pareto_sweep_experiment(
        "pareto_test", results_df, baseline_metrics,
        _DummyConfig(), _DummyConfig(), _DummyConfig(), _DummyConfig(), "cpu",
        base_dir=str(tmp_path),
    )

    files = {p.name for p in exp.dir.iterdir()}
    assert {"config.json", "pareto_frontier.csv", "pareto_frontier.png",
            "baseline_metrics.json", "manifest.json"} <= files


def test_track_model_comparison_experiment_writes_expected_artifacts(tmp_path):
    results_df = pd.DataFrame({"Model": ["EdgeLSTM+CS-MSE", "LSTM+MSE"],
                                "QPU Yield (%)": ["80 +/- 2", "60 +/- 3"]})
    decision_matrix_df = pd.DataFrame({"Model": ["EdgeLSTM+CS-MSE", "LSTM+MSE"],
                                        "Decision Score": [0.9, 0.5], "Rank": [1, 2]})
    sensitivity_summary_df = pd.DataFrame({"Model": ["EdgeLSTM+CS-MSE", "LSTM+MSE"],
                                            "Win Rate (%)": [100.0, 0.0]})
    sensitivity_trials_df = pd.DataFrame({"Winner": ["EdgeLSTM+CS-MSE"] * 4})
    baseline_metrics = {"total_steps": 100, "useful_pairs": 40}

    exp = track_model_comparison_experiment(
        "comparison_test", results_df, baseline_metrics, decision_matrix_df,
        (sensitivity_summary_df, sensitivity_trials_df, "verdict text"),
        _DummyConfig(), _DummyConfig(), _DummyConfig(), _DummyConfig(), _DummyConfig(), "cpu",
        base_dir=str(tmp_path),
    )

    files = {p.name for p in exp.dir.iterdir()}
    assert {"model_comparison.csv", "decision_matrix.csv", "decision_matrix.png",
            "sensitivity_summary.png", "sensitivity_trials.csv"} <= files


def test_track_ablation_experiment_writes_one_interaction_plot_per_metric(tmp_path):
    results_df = pd.DataFrame({
        "Model": ["EdgeLSTM+MSE", "EdgeLSTM+CS-MSE", "StandardLSTM+MSE", "StandardLSTM+CS-MSE"],
        "QPU Yield (%)": ["60 +/- 2", "90 +/- 1", "55 +/- 3", "65 +/- 2"],
    })
    decomposition_df = pd.DataFrame({
        "Metric": ["qpu_yield_pct", "mae"],
        "EdgeLSTM+MSE": [60.0, 0.05], "EdgeLSTM+CS-MSE": [90.0, 0.04],
        "StandardLSTM+MSE": [55.0, 0.06], "StandardLSTM+CS-MSE": [65.0, 0.055],
        "Architecture Effect": [15.0, -0.01], "Loss Effect": [20.0, -0.008],
        "Interaction Effect": [20.0, 0.002], "Interpretation": ["a", "b"],
    })
    baseline_metrics = {"total_steps": 100, "useful_pairs": 40}

    exp = track_ablation_experiment(
        "ablation_test", results_df, decomposition_df, baseline_metrics,
        _DummyConfig(), _DummyConfig(), _DummyConfig(), _DummyConfig(), "cpu",
        base_dir=str(tmp_path),
    )

    files = {p.name for p in exp.dir.iterdir()}
    assert "interaction_qpu_yield_pct.png" in files
    assert "interaction_mae.png" in files
    assert "ablation_decomposition.csv" in files


### 1.18 (Optional) run the test suite

Validates the entire decomposition (including the new temporal-timing, ablation, plotting, and experiment-tracking modules) before running the full pipeline.

In [ ]:
!python -m pytest tests/ -q


## 2. Import and device selection (CPU/GPU)

With the package materialized on disk, the rest of the notebook simply
imports it -- no business-logic class or function is redefined here.

In [ ]:
import torch
import pandas as pd

from qrepeater_twin.cli import get_device, main
from qrepeater_twin.config import (
    AblationConfig, BaselineConfig, ComparisonConfig, EnergyConfig,
    QuantumConfig, SimConfig, SweepConfig, TrainConfig,
)

DEVICE = get_device()
print(f"Selected PyTorch device: {DEVICE}")


## 3. Experiment configuration

Same hyperparameters as v3.1, plus `AblationConfig` for the new 2x2
factorial ablation study (Section 9 below).

In [ ]:
sim_cfg = SimConfig(n_steps=4000, dt=0.01, seed=42, window_size=20, test_size=0.2)

train_cfg = TrainConfig(
    hidden_size=16, epochs=150, lr=0.012, threshold=0.65,
    lambda_fn=4.0, discard_penalty_weight=10.0, max_discard_rate=0.60,
)

quantum_cfg = QuantumConfig(
    T1=50e-6, T2=30e-6, depol_prob=0.01, shots=512, seed=7, success_rate_cutoff=0.5,
)

sweep_cfg = SweepConfig(
    lambda_values=[1.0, 2.0, 5.0, 10.0, 20.0, 50.0],
    seeds=[42, 43, 44, 45, 46],  # multi-seed averaging: 5 independent rounds per lambda
)

baseline_cfg = BaselineConfig()
energy_cfg = EnergyConfig()

comparison_cfg = ComparisonConfig(
    representative_lambda=10.0,
    seeds=[42, 43, 44, 45, 46],
    include_xgboost=True,
    sensitivity_perturbation_pct=0.10,
)

ablation_cfg = AblationConfig(
    standard_lstm_hidden_size=64, standard_lstm_num_layers=2, standard_lstm_dropout=0.1,
    epochs=150, lr=0.012, representative_lambda=10.0,
    seeds=[42, 43, 44, 45, 46],
    headline_metrics=["qpu_yield_pct", "mae", "fp"],
)


## 4. Execution

Calls `main()` with `run_baseline_comparison=True` AND `run_ablation=True`:
runs the Pareto sweep, the cross-architecture baseline comparison, and the
2x2 factorial ablation study in one call.

In [ ]:
results = main(
    sim_cfg=sim_cfg, train_cfg=train_cfg, quantum_cfg=quantum_cfg,
    sweep_cfg=sweep_cfg, device=DEVICE, base_seed=42,
    run_baseline_comparison=True, baseline_cfg=baseline_cfg,
    energy_cfg=energy_cfg, comparison_cfg=comparison_cfg,
    run_ablation=True, ablation_cfg=ablation_cfg,
)
results_df, baseline_metrics, per_seed_results, comparison_results, ablation_results = results


## 5. Per-seed audit (optional)

`per_seed_results` preserves the raw metrics of each individual round
(`lambda -> [{seed, halted, attempted, useful_pairs, yield_qpu_pct, ...}, ...]`).

In [ ]:
lam_to_inspect = sweep_cfg.lambda_values[-1]  # e.g., the most conservative lambda in the sweep
pd.DataFrame(per_seed_results[lam_to_inspect])


### 5.1 Pareto Frontier plot

QPU Yield and MAE vs. `lambda_penalty`, with error bars from the
multi-seed mean +/- std.

In [ ]:
from qrepeater_twin import plotting

fig = plotting.plot_pareto_frontier(results_df, metric_cols=("QPU Yield (%)", "MAE"))


## 6. Baseline comparison: regression accuracy, confusion matrix, throughput, QPU economy, energy, C_latencia

`comparison_results` unpacks into the per-model results table, the blind
baseline metrics, the ranked decision matrix, per-seed raw results, and
the weight-sensitivity analysis.

In [ ]:
(comp_results_df, comp_baseline_metrics, decision_matrix_df,
 per_model_seed_results, sensitivity_results) = comparison_results
comp_results_df


### 6.1 Decision matrix (ranked)

In [ ]:
decision_matrix_df


### 6.2 Comparison and decision-matrix plots

In [ ]:
fig = plotting.plot_model_comparison_bars(comp_results_df, metric_col="QPU Yield (%)")


In [ ]:
fig = plotting.plot_decision_matrix(decision_matrix_df)


### 6.3 Per-seed audit for one model (optional)

Same idea as Section 5, but for a specific model's per-seed raw results.

In [ ]:
model_to_inspect = "EdgeLSTM+CS-MSE"  # try "LSTM+MSE", "RandomForest", "XGBoost", "Transformer"
pd.DataFrame(per_model_seed_results[model_to_inspect])


### 6.4 Confusion matrix for one model (representative single-seed view)

`comp_results_df` stores mean +/- std per model as formatted strings for
the statistically robust view; this cell instead reads ONE representative
seed's raw confusion counts from `per_model_seed_results` purely to
visualize a single confusion matrix (the aggregate table above remains
the source of truth for reporting).

In [ ]:
representative_seed_row = per_model_seed_results[model_to_inspect][0]
confusion_counts = {
    "TP": representative_seed_row["tp"], "FP": representative_seed_row["fp"],
    "TN": representative_seed_row["tn"], "FN": representative_seed_row["fn"],
}
fig = plotting.plot_confusion_matrix(confusion_counts, title=f"{model_to_inspect} (seed={representative_seed_row['seed']})")


## 7. Decision-weight sensitivity analysis (+/-10%)

`sensitivity_results` unpacks into the summary (win rate per model), all
vertex trials, and a plain-language robustness verdict.

In [ ]:
sensitivity_summary_df, sensitivity_trials_df, sensitivity_verdict = sensitivity_results
print(sensitivity_verdict)
sensitivity_summary_df


### 7.1 All vertex trials (optional)

One row per `2**n` combination of `+/-10%` weight perturbations.

In [ ]:
sensitivity_trials_df


### 7.2 Sensitivity plot

In [ ]:
fig = plotting.plot_sensitivity_summary(sensitivity_summary_df)


## 8. Temporal prediction analysis (NEW)

Task 1's core deliverable: does the EdgeLSTM+CS-MSE predictor detect
channel degradation at the right TIME, not just with a low average error?
This re-trains one EdgeLSTM+CS-MSE model (at `comparison_cfg.representative_lambda`,
seed `sweep_cfg.seeds[0]`) to obtain a full prediction sequence over the
test window, then runs it through `DigitalTwinOrchestrator.run_intelligent`
to get the admission-decision log analyzed below.

In [ ]:
from qrepeater_twin import (
    EdgeLSTM, train_edge_lstm, DigitalTwinOrchestrator, QuantumRepeaterNode,
    predict_sequence, compute_regression_metrics, compute_controller_decision_timing,
    compute_fidelity_statistics,
)
from qrepeater_twin.channel_simulator import WDMChannelSimulator

wdm_sim = WDMChannelSimulator(n_steps=sim_cfg.n_steps, dt=sim_cfg.dt, seed=sim_cfg.seed)
df = wdm_sim.generate_dataset()
X_train, y_train, X_test, y_test, _scaler = wdm_sim.preprocess(
    df, window_size=sim_cfg.window_size, test_size=sim_cfg.test_size,
)
X_train, y_train = X_train.to(DEVICE), y_train.to(DEVICE)
X_test, y_test = X_test.to(DEVICE), y_test.to(DEVICE)

seed = sweep_cfg.seeds[0]
torch.manual_seed(seed)
temporal_model = EdgeLSTM(input_size=2, hidden_size=train_cfg.hidden_size, num_layers=1).to(DEVICE)
temporal_model = train_edge_lstm(
    temporal_model, X_train, y_train, threshold=train_cfg.threshold,
    lambda_penalty=comparison_cfg.representative_lambda, lambda_fn=train_cfg.lambda_fn,
    discard_penalty_weight=train_cfg.discard_penalty_weight, max_discard_rate=train_cfg.max_discard_rate,
    epochs=train_cfg.epochs, lr=train_cfg.lr, device=DEVICE, seed=seed,
)

quantum_node = QuantumRepeaterNode(T1=quantum_cfg.T1, T2=quantum_cfg.T2, depol_prob=quantum_cfg.depol_prob,
                                    shots=quantum_cfg.shots, seed=quantum_cfg.seed)
temporal_orchestrator = DigitalTwinOrchestrator(model=temporal_model, quantum_node=quantum_node,
                                                 threshold=train_cfg.threshold, device=DEVICE)
temporal_metrics_run = temporal_orchestrator.run_intelligent(X_test, y_test)

regression_metrics = compute_regression_metrics(y_test, predict_sequence(temporal_model, X_test, device=DEVICE))
print("Pure regression quality (MAE/RMSE/R^2):", regression_metrics)

fidelity_stats = compute_fidelity_statistics(y_test.cpu().numpy(), threshold=train_cfg.threshold)
print("\nTrue fidelity trajectory statistics:", fidelity_stats)


### 8.1 Threshold-crossing timing: does the controller HALT at the right moment?

`compute_controller_decision_timing` reads the orchestrator's log
directly: a "falling" crossing of `pred_fidelity` below `threshold` IS the
controller's HALT decision boundary, so a positive `timing_error_steps`
here means the HALT decision arrived LATE (after the channel had already
degraded), and negative means it arrived EARLY (anticipating degradation
ahead of the true event).

In [ ]:
temporal_metrics = compute_controller_decision_timing(
    temporal_orchestrator.log, threshold=train_cfg.threshold, dt=comparison_cfg.cycle_time_s,
)
{k: v for k, v in temporal_metrics.items() if k != "matched_pairs_df"}


### 8.2 Matched degradation events (detail table)

In [ ]:
temporal_metrics["matched_pairs_df"]


### 8.3 Timing-error histogram

In [ ]:
fig = plotting.plot_temporal_prediction_error(temporal_metrics)


### 8.4 Fidelity trajectory: true vs. predicted, with degradation crossings

In [ ]:
from qrepeater_twin import extract_fidelity_arrays_from_log, find_threshold_crossings

true_seq, pred_seq = extract_fidelity_arrays_from_log(temporal_orchestrator.log)
true_crossings = find_threshold_crossings(true_seq, train_cfg.threshold, direction="falling")
pred_crossings = find_threshold_crossings(pred_seq, train_cfg.threshold, direction="falling")

fig = plotting.plot_fidelity_timeseries_with_crossings(
    true_seq, pred_seq, train_cfg.threshold,
    true_crossings=true_crossings, pred_crossings=pred_crossings,
)


## 9. Ablation study: {EdgeLSTM, StandardLSTM} x {MSE, CS-MSE} (NEW)

`ablation_results` unpacks into the per-cell results table, the factorial
decomposition (Architecture Effect / Loss Effect / Interaction Effect per
headline metric), the blind baseline metrics, and per-seed raw results.

In [ ]:
(ablation_results_df, decomposition_df, ablation_baseline_metrics,
 per_cell_seed_results) = ablation_results
ablation_results_df


### 9.1 Factorial decomposition

For each headline metric (`ablation_cfg.headline_metrics`): the
Architecture Effect (impact of EdgeLSTM vs. StandardLSTM), the Loss
Effect (impact of CS-MSE vs. plain MSE), and the Interaction Effect
(whether the combined gain is more than the sum of the two individual
effects).

In [ ]:
decomposition_df


In [ ]:
for _, row in decomposition_df.iterrows():
    print(f"[{row['Metric']}] {row['Interpretation']}\n")


### 9.2 Per-cell QPU Yield bar chart

In [ ]:
fig = plotting.plot_model_comparison_bars(ablation_results_df, metric_col="QPU Yield (%)", model_col="Model")


### 9.3 Architecture x Loss interaction plots

One plot per headline metric: parallel lines indicate a purely additive
effect (no synergy/antagonism); non-parallel lines make the interaction
effect visually explicit.

In [ ]:
for metric in decomposition_df["Metric"]:
    fig = plotting.plot_ablation_interaction(decomposition_df, metric)


### 9.4 Per-seed audit for one grid cell (optional)

In [ ]:
cell_to_inspect = ("EdgeLSTM", "CS-MSE")  # try ("StandardLSTM", "MSE"), etc.
pd.DataFrame(per_cell_seed_results[cell_to_inspect])


## 10. Automatic experiment tracking (NEW)

Records every run's configuration, result tables, and figures to a
timestamped directory under `experiments/` -- so every result above is
reproducible from disk alone, without re-running anything.

In [ ]:
from qrepeater_twin.experiment_tracking import (
    track_pareto_sweep_experiment, track_model_comparison_experiment, track_ablation_experiment,
)

pareto_exp = track_pareto_sweep_experiment(
    "pareto_sweep", results_df, baseline_metrics,
    sim_cfg, train_cfg, quantum_cfg, sweep_cfg, DEVICE,
)
print("Pareto sweep experiment saved to:", pareto_exp.dir)

comparison_exp = track_model_comparison_experiment(
    "model_comparison", comp_results_df, comp_baseline_metrics, decision_matrix_df,
    sensitivity_results, sim_cfg, train_cfg, quantum_cfg, baseline_cfg, comparison_cfg, DEVICE,
)
print("Model comparison experiment saved to:", comparison_exp.dir)

ablation_exp = track_ablation_experiment(
    "ablation_study", ablation_results_df, decomposition_df, ablation_baseline_metrics,
    sim_cfg, train_cfg, quantum_cfg, ablation_cfg, DEVICE,
)
print("Ablation study experiment saved to:", ablation_exp.dir)


### 10.1 Inspect one experiment's manifest (optional)

In [ ]:
import json

with open(pareto_exp.dir / "manifest.json") as f:
    manifest = json.load(f)
manifest
